# 🎙️ Chatterbox TTS — Kaggle 2× T4 GPU

**Resemble AI's Chatterbox Multilingual V3** (the checkpoint behind the
[official V3 Space](https://huggingface.co/spaces/ResembleAI/Chatterbox-Multilingual-TTS-V3)) — 23 languages,
zero-shot voice cloning — served as a **phone-sized web app** from a Kaggle **GPU T4 x2** runtime.

Everything stays in your Kaggle session; a Cloudflare tunnel carries it to your phone.

---

### 📲 How to run this (from your phone)

1. **Settings → Accelerator: `GPU T4 x2`, Internet: `On`**, and — if you want to skip the
   weight download next time — **Persistence: `Variables and Files`**. Then open the notebook.
2. **Tap ▶ on each numbered step**, top to bottom. Code stays collapsed into slim
   one-line bars after **Step 0** — *tap a bar* if you ever want to peek.
3. **Step 8 prints a big tappable `trycloudflare.com` URL** → open it, then *Add to
   Home Screen* and use it like an app.
4. First run downloads ~5 GB of weights (~4-6 min) into `/kaggle/working/.cache/huggingface`.
   Whether later sessions **skip** that download depends on how you save the notebook —
   see the persistence note below; without it, expect to re-download each session.

<details><summary><b>💾 Will the 5 GB of weights still be there next session? (read this once)</b></summary>

Kaggle does **not** keep `/kaggle/working` for interactive sessions by default. The cache survives only if:

* **Persistence: `Variables and Files`** (or `Files only`) is set in the notebook's session options —
  it must be enabled *before* the session stops, and Kaggle documents all persistence modes as
  **best-effort** (a crash, or overrunning the 20 GB working quota, can still lose it); **or**
* you **Save Version → Save & Run All**, which commits the output directory to that version.

With neither, step 4 re-downloads the checkpoint each session — annoying (~4-6 min) but harmless.
An alternative that never expires: download the weights once, publish `/kaggle/working/.cache/huggingface`
as a **Kaggle Dataset**, attach it to the notebook, and point `HF_HOME` at
`/kaggle/input/<your-dataset>` in step 2.

</details>

<details><summary><b>ℹ️ What this notebook does under the hood (tap to expand)</b></summary>

* **Custom mobile SPA instead of the Gradio Space** — the official Space caps text at 300 chars and renders a desktop
  grid; here you get sticky-thumb Generate, bottom-sheet language & voice pickers, clone-a-voice upload from your
  gallery/recorder, PWA install, and real SSE progress.
* **Up to two replicas per GPU (four total)** + a **work-stealing chunk queue** → up to four text parts render at once. Replicas are loaded sequentially and an OOM automatically leaves that GPU at its last safe count.
* **EPUB importer** — upload a book, inspect its chapter list, tick chapters, and create a separately named audio job for every title, with a live queue showing each chapter's progress. **Preview** drops a single chapter's text into the script box, and pressing **Generate** with an empty script box simply converts the ticked chapters.
* **Load the model from the web page** — a *Model* card at the top of the app downloads the checkpoint, picks V3/V2 and replicas-per-GPU, streams the loader log, and can unload to free VRAM. The notebook cell still autoloads by default (`AUTOLOAD = True` in step 4).
* **Reconnect-safe progress** — the job event stream is an append-only log replayed via `Last-Event-ID`, and the app falls back to polling if the stream stalls, so a dropped phone connection never loses a finished render.
* **Optional Google Drive mirroring** — every completed render is first saved under `/kaggle/working/chatterbox_outputs`, then mirrored to Drive when configured; a phone/tunnel disconnect cannot cancel it.
* **FP16 autocast with a per-GPU self-test** (auto-fallback to FP32), TF32 + cuDNN autotune.
* **Voice conditionals cached per GPU per clip** — reference embedding runs once, not per chunk.
* **Pipelined CPU encoding** (wav/mp3/opus/flac) and sequential, OOM-safe replica loading.
* Full tuning parity with the Space: exaggeration, CFG/pace, temperature, seed (deterministic per chunk), plus
  advanced min_p / top_p / repetition-penalty / gap / chunk-size controls.

</details>


### 0️⃣ Phone mode · ~1 s

> **Before you start:** open the session options panel (⋮ / right sidebar) and check
> **Accelerator = `GPU T4 x2`**, **Internet = `On`**, and **Persistence = `Variables and Files`**.
> That last one is what lets the ~5 GB of model weights survive into your next session —
> it is off by default, must be turned on *before* the session ends, and is best-effort even then.

Run this first: it injects a tiny stylesheet that **collapses every code cell into a slim bar** across the page —
tap a bar to peek or edit, it springs back after running. Cosmetic only; everything works the same.

<details><summary>ℹ️ How it works / how to revert</summary>

Pure CSS on the CodeMirror containers (`max-height` + fade + `:focus-within` expand), so it survives Kaggle's output
sanitiser. If the browser allows output JavaScript, a floating **☰ chip** also appears for *show/hide all code*.
To revert the classic look: ⋮ on this cell → **Clear output**.

</details>


In [ ]:
# ── STEP 0 · 📱 Phone mode: collapse all code cells into slim tap-to-peek bars ──
# Pure CSS, so it survives Kaggle's output sanitiser; a floating ☰ chip is added
# when output JavaScript is allowed. Revert any time: clear this cell's output.
from IPython.display import display, HTML

COMPACT_CSS = r"""
<style id="cbphone">
/* CodeMirror 6 (current JupyterLab + Kaggle) & CodeMirror 5 (classic):
   clip editors to a one-line peek bar with a fade */
html:not(.cbshowall) .cm-editor,
html:not(.cbshowall) .CodeMirror {
  max-height: 2.9em !important;
  overflow: hidden !important;
  border-radius: 10px !important;
  opacity: .85;
  -webkit-mask-image: linear-gradient(to bottom, #000 40%, transparent 95%);
          mask-image: linear-gradient(to bottom, #000 40%, transparent 95%);
}
/* tap the bar -> the editor gets focus -> expand fully until it loses focus */
html:not(.cbshowall) .cm-editor:focus-within,
html:not(.cbshowall) .CodeMirror:focus-within {
  max-height: none !important;
  opacity: 1;
  -webkit-mask-image: none; mask-image: none;
}
@media (max-width: 760px) {
  .cm-editor, .CodeMirror { font-size: 12.5px !important; }
  div.prompt, .jp-InputPrompt { font-size: 10px !important; }
}
#cbchip {
  position: fixed; right: 12px; bottom: 88px; z-index: 2147483000;
  background: #1f1b2b; color: #a78bfa; border: 1px solid #2b2540;
  border-radius: 999px; padding: 11px 15px;
  font: 600 13px/1 -apple-system, system-ui, sans-serif;
  box-shadow: 0 8px 24px rgba(0,0,0,.4); -webkit-user-select: none; user-select: none;
  cursor: pointer;
}
</style>
"""

CHIP_JS = r"""
(function () {
  try {
    if (document.getElementById('cbchip')) return;
    var chip = document.createElement('div');
    chip.id = 'cbchip';
    chip.textContent = '☰ code: bars';
    chip.onclick = function () {
      var shown = document.documentElement.classList.toggle('cbshowall');
      chip.textContent = shown ? '☰ code: shown' : '☰ code: bars';
    };
    document.body.appendChild(chip);
  } catch (e) {}
})();
"""

display(HTML(COMPACT_CSS))
try:
    from IPython.display import Javascript
    display(Javascript(CHIP_JS))
except Exception:
    pass   # sanitised on some frontends — the CSS above is still active

display(HTML(
    '<div style="margin:2px 0;padding:10px 12px;border-radius:12px;'
    'background:#17141f;border:1px solid #2b2540;color:#a49eb8;'
    'font:13px/1.45 -apple-system,system-ui,sans-serif">'
    '📱 <b style="color:#a78bfa">Phone mode on.</b> Code cells are now slim bars — '
    '<b style="color:#eeeaf6">tap one</b> to peek or edit, it collapses after running. '
    'Look for the <b style="color:#eeeaf6">☰ chip</b> to show everything at once, or '
    'clear this cell’s output to revert.</div>'))
print("phone mode: on")


### 1️⃣ Install everything · ~3 min (first run)

Chatterbox from GitHub `@master` — with the **V3** checkpoint support the PyPI wheel lacks — plus its pinned deps and
the web-server bits. Kaggle's own torch/CUDA stays untouched.

<details><summary>ℹ️ Note on the install strategy</summary>

*`--no-deps` + hand-picked pins:* master pins `torch==2.6.0`; Kaggle ships its own CUDA-matched torch/torchaudio and
we keep it — installing upstream's pin would replace the GPU-linked build. We also skip master's `gradio` pin
(custom UI below). `transformers==5.2.0` matches upstream master.

*The trade-off:* `transformers`/`diffusers` were pinned against torch 2.6.0 but will run against whatever torch
Kaggle's current image ships. The cell ends with a **version sanity check** that prints the actual
`torch.__version__`, CUDA/cuDNN, and the installed pins, and tells you what to downgrade in the unlikely case
step 4 errors out. Read that output on first run.

</details>


In [ ]:
# ── STEP 1 · Install everything (~3 min first run) ──────────────────────

import sys, subprocess, os, importlib.metadata as _md

def pip(*a):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

# --- Chatterbox itself -------------------------------------------------------
# The PyPI wheel (0.1.7) still ships the V2 multilingual checkpoint only.
# GitHub master adds `t3_model="v3"` support (the checkpoint the official
# "Chatterbox-Multilingual-TTS-V3" Space serves), so we install from source.
# --no-deps: DO NOT let it touch torch/torchaudio — Kaggle's CUDA builds stay put.
pip("--no-deps", "chatterbox-tts @ git+https://github.com/resemble-ai/chatterbox.git@master")

# --- Upstream-pinned runtime deps (minus torch/torchaudio/gradio) ------------
# transformers==5.2.0 matches upstream master; gradio is skipped on purpose
# (we serve our own mobile UI below).
pip("transformers==5.2.0", "librosa==0.11.0", "s3tokenizer", "diffusers==0.29.0",
    "conformer==0.3.2", "safetensors==0.5.3", "spacy-pkuseg", "pykakasi==2.3.0",
    "pyloudnorm", "omegaconf", "einops",
    "resemble-perth @ git+https://github.com/resemble-ai/Perth.git@master")

# --- Notebook-side deps ------------------------------------------------------
pip("fastapi>=0.115", "uvicorn[standard]>=0.30", "pydub", "python-multipart",
    "soundfile", "hf_transfer", "EbookLib", "beautifulsoup4", "google-api-python-client", "google-auth")

# System-level: ffmpeg (mp3/opus encode + voice-upload transcoding)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=False)

# Cloudflared for the public tunnel (self-contained binary, no login required)
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.check_call([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/usr/local/bin/cloudflared",
    ])
    os.chmod("/usr/local/bin/cloudflared", 0o755)

# --- Sanity check: we deliberately did NOT install upstream's torch==2.6.0 ---
# `--no-deps` keeps Kaggle's CUDA-linked torch, but transformers/diffusers were
# pinned against 2.6.0, so print what we actually ended up with and warn if the
# gap is big enough to matter. This is informational: Chatterbox itself only
# needs a recent torch, and the pins have historically been tolerant.
try:
    import torch as _torch
    _tv = _torch.__version__
    _base = tuple(int(x) for x in _tv.split("+")[0].split(".")[:2])
    print(f"\ntorch {_tv} (Kaggle's build, kept on purpose)"
          f" · CUDA {_torch.version.cuda} · cuDNN {_torch.backends.cudnn.version()}")
    for _pkg in ("transformers", "diffusers", "torchaudio", "safetensors", "numpy"):
        try: print(f"  {_pkg:<13} {_md.version(_pkg)}")
        except Exception: print(f"  {_pkg:<13} (not installed)")
    if _base < (2, 4):
        print(f"!! torch {_tv} is older than the 2.6.0 the deps were pinned against — "
              f"if step 4 raises an import/attribute error, run:\n"
              f"   !pip install -q 'transformers==4.46.3' 'diffusers==0.31.0'\n"
              f"   and re-run steps 1 and 4.")
    elif _base < (2, 6):
        print(f"   (pins target torch 2.6.0; {_tv} is close enough — proceed, and only "
              f"downgrade transformers/diffusers if step 4 actually errors)")
    else:
        print("   torch is at/above the pinned target — nothing to do.")
except Exception as _e:
    print(f"!! could not import torch to verify the environment: {_e!s}")

print("\ndeps ok")


### 2️⃣ Detect the GPUs · ~1 s

You should see **2× Tesla T4 (16 GB)** listed. By default the notebook requests **two independent replicas per GPU** (up to four workers); it loads them serially and automatically falls back to fewer replicas if VRAM is insufficient. Single-GPU or CPU runtimes still work, just slower; this step also enables TF32 + cuDNN autotune.


In [ ]:
# ── STEP 2 · Detect GPUs & set performance flags ─────────────────────────

import os, multiprocessing as mp

# --- ENV must be set before torch/huggingface_hub are imported ---------------
# Put the HF cache under /kaggle/working (20 GB quota) so the ~5 GB of Chatterbox
# weights CAN survive into the next session — but only if you enabled
# Persistence = "Variables and Files" in the session options, or you Save & Run
# All. Kaggle keeps nothing here by default and all persistence is best-effort,
# so treat a re-download as the normal case, not a failure.
# Already have the weights as an attached Kaggle Dataset? Point HF_HOME at it:
#   os.environ["HF_HOME"] = "/kaggle/input/<your-weights-dataset>"
os.environ.setdefault("HF_HOME", "/kaggle/working/.cache/huggingface")
# Multi-threaded (Rust) downloader — the T3 safetensors alone is ~2 GB.
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")   # we manage our own threads

import torch

# --- Free speed -----------------------------------------------------------
# TF32 matmuls on Ampere+ (ignored gracefully on the T4's older arch, but it
# also speeds up cuDNN fp32 convs; harmless if unsupported).
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True     # fixed-ish shapes per chunk → autotuned convs

N_GPU = torch.cuda.device_count() if torch.cuda.is_available() else 0
GPUS = []
for _i in range(N_GPU):
    _p = torch.cuda.get_device_properties(_i)
    GPUS.append({"id": _i, "name": _p.name,
                 "vram_gb": round(_p.total_memory / 2**30, 1),
                 "cc": f"{_p.major}.{_p.minor}"})

# Replica plan. A T4 has 16 GB: two FP16 Chatterbox replicas often fit, but the
# loader below is deliberately sequential and keeps the last safe count on OOM.
# Set this to 1 if your particular Kaggle image is memory constrained.
REPLICAS_PER_GPU = 2
BASE_DEVICES = [f"cuda:{i}" for i in range(N_GPU)] or ["cpu"]   # actual worker slots are made in Step 4

# CPU pools: Kaggle's GPU tier gives ~4 vCPUs / 29 GB RAM. That is plenty for
# pipelined ffmpeg encoding + voice-upload transcoding; oversubscribing it
# with dozens of threads would only add churn, so we cap hard.
N_CPU = max(2, min(8, mp.cpu_count() or 4))

if GPUS:
    for g in GPUS:
        print(f"  GPU {g['id']}: {g['name']} · {g['vram_gb']} GB · CC {g['cc']}")
    d = "s" if N_GPU > 1 else ""
    print(f"{N_GPU} GPU{d} → requesting up to {REPLICAS_PER_GPU * N_GPU} replicas ({REPLICAS_PER_GPU}/GPU), loaded one at a time")
else:
    print("!! No CUDA GPU found — running on CPU (consider switching to GPU T4 x2)")
print(f"CPU workers available: {N_CPU}")
print(f"HF cache: {os.environ['HF_HOME']}")
_cached = os.path.isdir(os.path.join(os.environ["HF_HOME"], "hub"))
print("  weights already cached from a previous session — step 4 will be fast"
      if _cached else
      "  cache is empty — step 4 downloads ~5 GB (set Persistence='Variables and Files' to keep it)")


### 3️⃣ Language catalogue · instant

The 23 languages of Chatterbox Multilingual (from `SUPPORTED_LANGUAGES`) with display names/flags for the picker sheet.


In [ ]:
# ── STEP 3 · Language catalogue (23 languages) ───────────────────────────

# The canonical code list comes straight from the installed Chatterbox, so the
# picker can never drift out of sync with what generate() actually accepts —
# a future version adding or removing a language is handled automatically.
# Only display metadata (native name, flag) and picker order live here.
try:
    from chatterbox.mtl_tts import SUPPORTED_LANGUAGES as _CB_LANGS
except Exception as _e:
    _CB_LANGS = None
    print(f"(chatterbox not importable yet — using the built-in 23-language "
          f"catalogue; re-run Step 1 if the install failed: {_e!s})")

# code -> (English name, native name, flag)
# Picker order (their dict insertion order below): English first, then the
# top-traffic languages, then the rest.
_DISPLAY = {
    "en": ("English",    "English",   "🇬🇧"),
    "es": ("Spanish",    "Español",   "🇪🇸"),
    "fr": ("French",     "Français",  "🇫🇷"),
    "de": ("German",     "Deutsch",   "🇩🇪"),
    "pt": ("Portuguese", "Português", "🇵🇹"),
    "hi": ("Hindi",      "हिन्दी",     "🇮🇳"),
    "zh": ("Chinese",    "中文",       "🇨🇳"),
    "ja": ("Japanese",   "日本語",     "🇯🇵"),
    "ko": ("Korean",     "한국어",     "🇰🇷"),
    "ar": ("Arabic",     "العربية",   "🇸🇦"),
    "it": ("Italian",    "Italiano",  "🇮🇹"),
    "ru": ("Russian",    "Русский",   "🇷🇺"),
    "tr": ("Turkish",    "Türkçe",    "🇹🇷"),
    "nl": ("Dutch",      "Nederlands","🇳🇱"),
    "pl": ("Polish",     "Polski",    "🇵🇱"),
    "sv": ("Swedish",    "Svenska",   "🇸🇪"),
    "da": ("Danish",     "Dansk",     "🇩🇰"),
    "fi": ("Finnish",    "Suomi",     "🇫🇮"),
    "no": ("Norwegian",  "Norsk",     "🇳🇴"),
    "el": ("Greek",      "Ελληνικά",  "🇬🇷"),
    "he": ("Hebrew",     "עברית",     "🇮🇱"),
    "ms": ("Malay",      "Bahasa Melayu", "🇲🇾"),
    "sw": ("Swahili",    "Kiswahili", "🇰🇪"),
}

if _CB_LANGS:
    # Bidirectional: pick up languages the library gained, hide ones it dropped.
    _added = sorted(set(_CB_LANGS) - set(_DISPLAY))
    _removed = sorted(set(_DISPLAY) - set(_CB_LANGS))
    for _c in _added:
        _DISPLAY[_c] = (_CB_LANGS[_c], _CB_LANGS[_c], "🌐")
    if _removed:
        print(f"!! hiding {len(_removed)} language(s) the installed library no "
              f"longer supports: {_removed}")
        for _c in _removed: _DISPLAY.pop(_c)
    _order = [c for c in _DISPLAY if c in _CB_LANGS] + _added
else:
    _added = []
    _order = list(_DISPLAY)

LANGUAGES = {_c: _DISPLAY[_c] for _c in _order}
DEFAULT_LANG = "en"

def lang_meta(code: str) -> dict:
    name, native, flag = LANGUAGES[code]
    return {"id": code, "name": name, "native": native, "flag": flag}

LANGUAGE_LIST = [lang_meta(c) for c in LANGUAGES]
print(f"{len(LANGUAGE_LIST)} languages loaded: {' '.join(LANGUAGES.keys())}"
      + (f" · +{len(_added)} from upstream" if _added else "")
      + (" (built-in fallback table)" if not _CB_LANGS else ""))


### 4️⃣ Model manager — load Chatterbox V3 · ~4–8 min first run
Defines the loader and leaves `AUTOLOAD = True`, so running this cell downloads the checkpoint once and loads the replicas as before. Set `AUTOLOAD = False` if you'd rather skip straight to the server and press **Load model** on the web page — every load/reload/unload control is exposed to the app in step 7.

Loading is serial (never four simultaneous copies to VRAM), a CUDA OOM is contained, and each replica gets an fp16 smoke test with automatic fp32 fallback.


In [ ]:
# ── STEP 4 · Model manager — load replicas here OR from the live web page ───
# This cell only *defines* the loader (download → replica load → fp16 self-test)
# and exposes it as a control surface the FastAPI app re-uses, so the server can
# start with no weights in memory and you can press "Load model" on your phone.
# AUTOLOAD=True keeps the classic notebook flow: run the cell, get a hot model.

import os, sys, threading, time, types, gc

# --- Perth watermark shim: it's only a provenance tagger; never block TTS on it.
try:
    import perth  # noqa: F401
    HAS_WATERMARK = True
except Exception as e:
    HAS_WATERMARK = False
    _p = types.ModuleType("perth")
    class _DummyWatermarker:
        def apply_watermark(self, wav, sample_rate): return wav
    _p.PerthImplicitWatermarker = _DummyWatermarker
    sys.modules["perth"] = _p
    print(f"(resemble-perth unavailable: {e!s} — outputs will not be watermarked)")

import torch
import numpy as np
from huggingface_hub import snapshot_download
from chatterbox.mtl_tts import ChatterboxMultilingualTTS, SUPPORTED_LANGUAGES as _CB_LANGS

# ---------------------------------------------------------------------------
# OPTIMISATION · kill T3's per-token tqdm bar.
# T3.inference() wraps its decode loop in `tqdm(range(max_new_tokens), ...)`,
# i.e. one bar update per generated token. With four worker threads each driving
# tqdm's refresh/ncols logic, that is GIL-serialised Python work plus four bars
# interleaving into one stdout — for a progress display nobody can see (our real
# progress goes over SSE, per chunk) and which spams the Kaggle cell output.
# Measured cost of the bar itself: ~10 ms per 4x1200 decode steps, i.e. small
# next to GPU decode — this is a cleanliness + contention fix, not a magic 2x.
# `tqdm` is imported at module scope in t3.py, so rebinding that one name
# neutralises the bars in BOTH inference() and inference_turbo().
#
# The shim does double duty as the cancel-checking point: upstream's decode
# (and the S3Gen vocoder) step through it once per token, so polling a
# thread-local cancel hook between items lets a Stop land mid-chunk instead of
# only between chunks. Thread-local so one job's Stop never aborts another job
# sharing the same replica.
# ---------------------------------------------------------------------------
_CANCEL_HOOK = threading.local()

class _ChunkCancelled(Exception):
    """Synthesis was cancelled while an upstream generation loop was running."""

def _set_cancel_hook(fn):
    _CANCEL_HOOK.fn = fn

def _clear_cancel_hook():
    _CANCEL_HOOK.fn = None

def _cancel_requested() -> bool:
    fn = getattr(_CANCEL_HOOK, "fn", None)
    return bool(fn and fn())

def _silence_t3_tqdm() -> bool:
    class _NoTqdm:
        """Iterable passthrough with the sliver of the tqdm API T3 touches."""
        _cb_cancel_hook = True          # marker so a re-run upgrades the shim
        def __init__(self, iterable=None, *a, **k): self._it = iterable if iterable is not None else []
        def __iter__(self):
            for _x in self._it:
                if _cancel_requested():
                    raise _ChunkCancelled()
                yield _x
        def __enter__(self): return self
        def __exit__(self, *a): return False
        def update(self, *a, **k): pass
        def close(self): pass
        def set_description(self, *a, **k): pass
        def set_postfix(self, *a, **k): pass
        @staticmethod
        def write(*a, **k): pass
    patched = []
    for _modname in ("chatterbox.models.t3.t3",
                     "chatterbox.models.s3gen.s3gen",
                     "chatterbox.models.s3gen.flow_matching"):
        _m = sys.modules.get(_modname)
        _cur = getattr(_m, "tqdm", None) if _m is not None else None
        # Skip when absent, and when this cell already swapped it in (re-run).
        if _cur is not None and not getattr(_cur, "_cb_cancel_hook", False):
            _m.tqdm = _NoTqdm
            patched.append(_modname.rsplit(".", 1)[-1])
    return bool(patched)

# t3.py is imported as a side effect of importing mtl_tts above, so patch now.
if _silence_t3_tqdm():
    print("per-token tqdm bars disabled (they cost GIL time in a 4-thread server)")

# Cross-check our picker (step 3) against the library we actually installed.
# Step 3 already derives the list from the same SUPPORTED_LANGUAGES source, so
# nothing is patched here — this is just a canary in case that cell was skipped
# or a stale notebook state is lingering.
_lang_diff = set(_CB_LANGS) ^ set(LANGUAGES)
if _lang_diff:
    print(f"!! language list drifted from upstream: {sorted(_lang_diff)} — "
          f"re-run Step 3 so the picker matches the installed model")

REPO_ID   = "ResembleAI/chatterbox"
T3_FILES  = {"v3": "t3_mtl23ls_v3.safetensors", "v2": "t3_mtl23ls_v2.safetensors"}
T3_MODELS = ["v3", "v2"]          # try newest first when t3="auto"
NEEDED    = ["ve.pt", "s3gen.pt", "grapheme_mtl_merged_expanded_v1.json",
             "conds.pt", "Cangjie5_TC.json"]

AUTOLOAD = True          # set False to load purely from the web page
MAX_REPLICAS_PER_GPU = 4

# ---------------------------------------------------------------------------
# Global registries. Re-running this cell must never leave half-dead replicas
# behind, but it also must not clobber a model that is currently rendering, so
# everything mutates under one lock.
# ---------------------------------------------------------------------------
MODELS: dict = globals().get("MODELS", {})              # slot -> ChatterboxMultilingualTTS
FP16: dict = globals().get("FP16", {})                  # slot -> bool
MODEL_DEVICE: dict = globals().get("MODEL_DEVICE", {})  # slot -> "cuda:0" / "cpu"
WORKERS: list = globals().get("WORKERS", [])            # ordered slot names
CKPT_DIR  = globals().get("CKPT_DIR", None)
ACTIVE_T3 = globals().get("ACTIVE_T3", None)
MODEL_SR  = globals().get("MODEL_SR", 24000)            # refreshed on every load
SAMPLE_RATE = MODEL_SR

_MODEL_LOCK = globals().get("_MODEL_LOCK") or threading.Lock()
MODEL_STATE = globals().get("MODEL_STATE") or {
    "status": "idle",          # idle | downloading | loading | testing | warming | ready | error
    "message": "no model loaded",
    "t3": None, "requested": 0, "loaded": 0,
    "log": [], "started": 0.0, "finished": 0.0,
}

def _mlog(msg: str, status: str | None = None, **fields):
    stamp = time.strftime("%H:%M:%S")
    MODEL_STATE["log"] = (MODEL_STATE.get("log") or [])[-199:] + [f"{stamp}  {msg}"]
    MODEL_STATE["message"] = msg
    if status: MODEL_STATE["status"] = status
    MODEL_STATE.update(fields)
    print(msg, flush=True)

def model_ready() -> bool:
    return bool(WORKERS)

def model_status() -> dict:
    return {
        "status": MODEL_STATE.get("status", "idle"),
        "message": MODEL_STATE.get("message", ""),
        "ready": model_ready(),
        "busy": MODEL_STATE.get("status") in ("downloading", "loading", "testing", "warming"),
        "t3": ACTIVE_T3, "checkpoint_dir": CKPT_DIR,
        "requested": MODEL_STATE.get("requested", 0),
        "loaded": len(WORKERS),
        "workers": [{"slot": s, "device": MODEL_DEVICE.get(s), "fp16": FP16.get(s, False)} for s in WORKERS],
        "sample_rate": MODEL_SR,
        "gpus": GPUS, "n_gpus_available": len(GPUS),
        "max_replicas_per_gpu": MAX_REPLICAS_PER_GPU,
        "default_replicas_per_gpu": REPLICAS_PER_GPU,
        "options": T3_MODELS,
        "vram": MODEL_STATE.get("vram", {}),
        "suggested_replicas_per_gpu": MODEL_STATE.get("suggested_replicas_per_gpu"),
        "attn": sorted({MODELS[s_].__dict__.get("_cb_attn", "unknown") for s_ in WORKERS}) or None,
        "log": (MODEL_STATE.get("log") or [])[-40:],
        "elapsed_s": round((MODEL_STATE.get("finished") or time.time()) - MODEL_STATE["started"], 1)
                      if MODEL_STATE.get("started") else 0.0,
    }

# ---------------------------------------------------------------------------
# Weights download ONCE into the persistent cache, then every replica loads
# from disk one at a time.
# ---------------------------------------------------------------------------
def _snapshot(t3_file: str) -> str:
    last = None
    for attempt in range(3):
        try:
            return snapshot_download(
                repo_id=REPO_ID, repo_type="model", revision="main",
                allow_patterns=NEEDED + [t3_file],
                token=os.getenv("HF_TOKEN"),
            )
        except Exception as e:
            last = e; _mlog(f"  download attempt {attempt+1} failed: {e!s}"); time.sleep(3)
    raise last

def download_checkpoint(t3: str = "auto") -> tuple[str, str]:
    """Fetch (or reuse) the weights. Returns (checkpoint_dir, t3_tag)."""
    wanted = T3_MODELS if t3 in ("auto", "", None) else [t3]
    errors = []
    for tag in wanted:
        if tag not in T3_FILES:
            errors.append(f"unknown checkpoint '{tag}'"); continue
        try:
            t0 = time.time()
            _mlog(f"downloading checkpoint '{tag}' (cached after the first run)…", "downloading")
            d = _snapshot(T3_FILES[tag])
            _mlog(f"checkpoint '{tag}' ready in {time.time()-t0:.1f}s -> {d}")
            return d, tag
        except Exception as e:
            errors.append(f"t3_model={tag}: {e!s}"); _mlog(f"  t3_model={tag} unavailable: {e!s}")
    raise RuntimeError("could not download a Chatterbox checkpoint — " + "; ".join(errors))

def _free_replicas():
    """Drop every loaded replica and return the VRAM to the allocator."""
    global MODELS, FP16, MODEL_DEVICE, WORKERS
    MODELS.clear(); FP16.clear(); MODEL_DEVICE.clear(); WORKERS.clear()
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def unload_models() -> dict:
    """Free all replicas (web UI 'Unload' / before a reload with new settings)."""
    global ACTIVE_T3
    with _MODEL_LOCK:
        _free_replicas()
        ACTIVE_T3 = None
        MODEL_STATE.update(status="idle", requested=0, loaded=0, finished=time.time())
        _mlog("model unloaded — VRAM released")
    return model_status()

# ---------------------------------------------------------------------------
# OPTIMISATION · fused attention. T3's backbone is a real HF LlamaModel/GPT2Model
# and the T4 (sm_75) supports PyTorch SDPA's memory-efficient backend. Depending
# on the transformers build the default may still be eager, so ask for "sdpa"
# explicitly — same math, fused kernels. Purely opportunistic: if the installed
# transformers/model doesn't support it we keep whatever it was using.
# ---------------------------------------------------------------------------
ATTN_IMPL = "sdpa"

def _set_attn_impl(model, impl: str = ATTN_IMPL) -> str:
    tfmr = getattr(getattr(model, "t3", None), "tfmr", None)
    if tfmr is None: return "n/a"
    def _current():
        return (getattr(getattr(tfmr, "config", None), "_attn_implementation", None) or "unknown")
    before = _current()
    if before == impl: return before
    try:
        tfmr.set_attn_implementation(impl)          # transformers >= 4.4x
    except AttributeError:
        # Older transformers has no setter; assigning the config field is the
        # documented equivalent there.
        try: tfmr.config._attn_implementation = impl
        except Exception: return before
    except Exception as e:
        # The model actively refused (kernel unavailable for this arch/dtype).
        # Do NOT force the config field — that would just lie about the backend
        # and can break the forward pass. Keep what worked.
        _mlog(f"  attention '{impl}' unavailable, staying on '{before}': {e!s}")
        return before
    return _current()

def _load_replica(slot: str, device: str):
    t0 = time.time()
    if device.startswith("cuda"): torch.cuda.empty_cache()
    m = ChatterboxMultilingualTTS.from_local(CKPT_DIR, device, t3_model=ACTIVE_T3)
    m._cb_lock = threading.Lock()
    m._cb_voice_path = None
    m._cb_default_conds = m.conds
    m._cb_conds_cache = {}            # voice_ref -> prepared Conditionals (per replica)
    m._cb_attn = _set_attn_impl(m)
    MODELS[slot], MODEL_DEVICE[slot] = m, device
    return time.time() - t0

# ---------------------------------------------------------------------------
# OPTIMISATION · warm several representative chunk lengths, not just one short
# sentence. cuDNN autotune (torch.backends.cudnn.benchmark) benchmarks kernels
# per input shape, so a single short warmup leaves the first real chunk of any
# other length paying that cost mid-job. Three lengths bracket what split_script
# actually emits (short / typical / near the ~260-char cap).
# ---------------------------------------------------------------------------
WARM_TEXTS = [
    "Short warm up.",
    "A medium length sentence to prime the decode loop for typical chunks. " * 2,
    ("A longer passage near the two-hundred-sixty character chunk cap, so cuDNN "
     "has already benchmarked kernels for this shape before real traffic. ") * 2,
]

def warm_replica(slot: str, texts=None) -> float:
    """Run a few shapes through a replica so real traffic never autotunes."""
    m, dev = MODELS[slot], MODEL_DEVICE[slot]
    use_fp16 = FP16.get(slot, False) and dev.startswith("cuda")
    if dev.startswith("cuda"): torch.cuda.set_device(int(dev.split(":")[1]))
    t0 = time.time()
    for t in (texts or WARM_TEXTS):
        try:
            with m._cb_lock, torch.inference_mode(), \
                 torch.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
                m.generate(t, language_id="en")
        except Exception as e:
            _mlog(f"  {slot}: warmup shape failed (harmless): {e!s}")
    return time.time() - t0

# ---------------------------------------------------------------------------
# Per-replica FP16 validation with automatic FP32 fallback. T4 tensor cores
# love fp16, but rather than trusting it blindly we render a sentence and
# verify. The winning dtype is applied per-chunk by the synthesis engine.
# ---------------------------------------------------------------------------
def _validate_replica(slot: str) -> bool:
    device = MODEL_DEVICE[slot]
    m = MODELS[slot]
    if device == "cpu":
        FP16[slot] = False
        return True
    torch.cuda.set_device(int(device.split(":")[1]))   # smoke test on ITS gpu
    for use_fp16 in (True, False):
        try:
            with m._cb_lock, torch.inference_mode(), \
                 torch.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
                torch.manual_seed(1234)
                wav = m.generate("Chatterbox speaks on every GPU you give it.",
                                 language_id="en")
            w = np.asarray(wav.squeeze().float().cpu(), dtype=np.float32)
            ok = w.size > 0 and np.isfinite(w).all() and float(np.std(w)) > 1e-5
            FP16[slot] = bool(use_fp16 and ok)
            if ok:
                return True
        except Exception as e:
            _mlog(f"  {device}: fp{16 if use_fp16 else 32} smoke test failed: {e!s}")
    FP16[slot] = False
    return False

def dev_fp16_default(slot: str) -> bool:
    """Precision guess used only when the fp16 smoke test is skipped."""
    return MODEL_DEVICE.get(slot, "cpu").startswith("cuda")

# ---------------------------------------------------------------------------
# Are we leaving VRAM on the table? Measure what a replica actually costs
# (weights + peak activations during the smoke test) and say plainly how many
# more would fit, instead of making you eyeball a number and guess.
# ---------------------------------------------------------------------------
VRAM_SAFETY_GB = 1.5      # headroom for fragmentation + a long chunk's KV cache

def _measure_vram(replicas_per_gpu: int) -> dict:
    if not torch.cuda.is_available(): return {}
    report, best = {}, None
    for base_device in {MODEL_DEVICE[s] for s in WORKERS if MODEL_DEVICE[s].startswith("cuda")}:
        idx = int(base_device.split(":")[1])
        n_here = sum(1 for s in WORKERS if MODEL_DEVICE[s] == base_device)
        if not n_here: continue
        total = torch.cuda.get_device_properties(idx).total_memory / 2**30
        used  = torch.cuda.memory_allocated(idx) / 2**30
        peak  = torch.cuda.max_memory_allocated(idx) / 2**30
        per   = max(used, peak) / n_here
        fits  = int((total - VRAM_SAFETY_GB) // per) if per > 0 else n_here
        report[base_device] = {"total_gb": round(total, 1), "used_gb": round(used, 1),
                               "peak_gb": round(peak, 1), "per_replica_gb": round(per, 2),
                               "replicas": n_here, "fits": fits}
        best = fits if best is None else min(best, fits)
    MODEL_STATE["vram"] = report
    for dev, r in sorted(report.items()):
        _mlog(f"  {dev}: {r['peak_gb']:.1f}/{r['total_gb']:.0f} GB peak with "
              f"{r['replicas']} replica(s) → ~{r['per_replica_gb']:.1f} GB each")
    if best is not None:
        capped = min(best, MAX_REPLICAS_PER_GPU)
        MODEL_STATE["suggested_replicas_per_gpu"] = capped
        if capped > replicas_per_gpu:
            _mlog(f"  💡 headroom for {capped} replicas/GPU (running {replicas_per_gpu}) — "
                  f"raise “Replicas per GPU” to {capped} and reload for more chunks in flight")
        elif capped < replicas_per_gpu:
            _mlog(f"  ⚠ only ~{capped} replica(s)/GPU really fit; drop to {max(1, capped)} if you hit OOM")
        else:
            _mlog(f"  VRAM is well used at {replicas_per_gpu} replica(s)/GPU")
    return report

def load_models(t3: str = "auto", replicas_per_gpu: int | None = None,
                validate: bool = True, warm: bool = True) -> dict:
    """Download (if needed) + load replicas serially. Safe to call repeatedly."""
    global CKPT_DIR, ACTIVE_T3, MODEL_SR, SAMPLE_RATE
    replicas = int(replicas_per_gpu or REPLICAS_PER_GPU)
    replicas = max(1, min(MAX_REPLICAS_PER_GPU, replicas))
    if not _MODEL_LOCK.acquire(blocking=False):
        raise RuntimeError("a model load is already running")
    try:
        MODEL_STATE.update(started=time.time(), finished=0.0, log=[],
                           requested=replicas * len(BASE_DEVICES), loaded=0)
        _mlog("preparing model…", "downloading")
        CKPT_DIR, ACTIVE_T3 = download_checkpoint(t3)
        MODEL_STATE["t3"] = ACTIVE_T3

        _free_replicas()                     # never stack replicas across reloads
        if torch.cuda.is_available():
            for _i in range(torch.cuda.device_count()):
                torch.cuda.reset_peak_memory_stats(_i)
        _mlog(f"loading up to {replicas * len(BASE_DEVICES)} replica(s) "
              f"({replicas}/GPU) one at a time…", "loading")
        for base_device in BASE_DEVICES:
            for replica_no in range(replicas):
                slot = f"{base_device}#{replica_no}"
                try:
                    dt = _load_replica(slot, base_device)
                    WORKERS.append(slot)
                    MODEL_STATE["loaded"] = len(WORKERS)
                    _mlog(f"  {slot}: ready in {dt:.1f}s")
                except torch.cuda.OutOfMemoryError:
                    _mlog(f"  {slot}: CUDA OOM — keeping {replica_no} safe replica(s) on {base_device}")
                    MODELS.pop(slot, None); MODEL_DEVICE.pop(slot, None)
                    torch.cuda.empty_cache(); break
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        _mlog(f"  {slot}: CUDA OOM — keeping {replica_no} safe replica(s) on {base_device}")
                        MODELS.pop(slot, None); MODEL_DEVICE.pop(slot, None)
                        torch.cuda.empty_cache(); break
                    MODELS.pop(slot, None); MODEL_DEVICE.pop(slot, None)
                    _mlog(f"  {slot}: load failed: {e!s}")
                    if not WORKERS: raise
                    break
        if not WORKERS:
            raise RuntimeError("no Chatterbox model replica could be loaded")

        MODEL_SR = next(iter(MODELS.values())).sr
        SAMPLE_RATE = MODEL_SR
        globals()["MODEL_SR"] = MODEL_SR
        globals()["SAMPLE_RATE"] = MODEL_SR
        _mlog(f"{len(WORKERS)} replica worker(s) ready · sample rate {MODEL_SR} Hz")

        if validate:
            MODEL_STATE["status"] = "testing"
            for slot in list(WORKERS):
                dev = MODEL_DEVICE[slot]
                ok = _validate_replica(slot)
                tag = "FP16" if FP16.get(slot) else "FP32"
                if dev.startswith("cuda"):
                    vram = torch.cuda.memory_allocated(int(dev.split(":")[1])) / 2**30
                    _mlog(f"  {slot}: {tag} {'OK' if ok else 'FAILED'} · {vram:.1f} GB VRAM used")
                else:
                    _mlog(f"  {slot}: {tag} {'OK' if ok else 'FAILED'}")
        else:
            for slot in WORKERS: FP16.setdefault(slot, dev_fp16_default(slot))

        attns = {MODELS[s_].__dict__.get("_cb_attn", "unknown") for s_ in WORKERS}
        _mlog(f"attention implementation: {', '.join(sorted(attns))}")

        if warm and any(MODEL_DEVICE[s_].startswith("cuda") for s_ in WORKERS):
            MODEL_STATE["status"] = "warming"
            for slot in list(WORKERS):
                _mlog(f"  {slot}: warmed {len(WARM_TEXTS)} chunk shapes in {warm_replica(slot):.1f}s")

        _measure_vram(replicas)

        MODEL_STATE["finished"] = time.time()
        _mlog(f"model '{ACTIVE_T3}' ready · {len(WORKERS)} worker(s) · "
              f"{ {s: ('fp16' if FP16.get(s) else 'fp32') for s in WORKERS} }", "ready")
        return model_status()
    except Exception as e:
        MODEL_STATE["finished"] = time.time()
        _mlog(f"load failed: {e!s}", "error")
        raise
    finally:
        _MODEL_LOCK.release()

# ---------------------------------------------------------------------------
# Opt-in per-stage profiler: where does a chunk's time actually go, T3's
# autoregressive decode or S3Gen's vocoder? Wraps each replica's inference
# entry points and accumulates stats. profile_report() prints the split;
# profile(False) restores the originals. Off by default — it prints nothing
# until you ask for the report, but it does add a timer per call.
# ---------------------------------------------------------------------------
PROFILE: dict[str, dict[str, list]] = {}
_PROFILE_LOCK = threading.Lock()

def profile(enable: bool = True) -> str:
    import functools
    def _wrap(fn, slot, label):
        if getattr(fn, "_cb_profiled", False): return fn
        @functools.wraps(fn)
        def inner(*a, **k):
            t0 = time.perf_counter()
            try: return fn(*a, **k)
            finally:
                dt = time.perf_counter() - t0
                with _PROFILE_LOCK:
                    PROFILE.setdefault(slot, {}).setdefault(label, []).append(dt)
        inner._cb_profiled = True
        inner._cb_original = fn
        return inner
    n = 0
    for slot in WORKERS:
        m = MODELS[slot]
        for obj, attr, label in ((getattr(m, "t3", None), "inference", "T3 decode"),
                                 (getattr(m, "s3gen", None), "inference", "S3Gen vocode")):
            if obj is None or not hasattr(obj, attr): continue
            cur = getattr(obj, attr)
            if enable:
                setattr(obj, attr, _wrap(cur, slot, label)); n += 1
            elif getattr(cur, "_cb_profiled", False):
                setattr(obj, attr, cur._cb_original); n += 1
    if enable: PROFILE.clear()
    msg = f"profiling {'enabled' if enable else 'disabled'} on {n} hook(s) across {len(WORKERS)} replica(s)"
    print(msg); return msg

def profile_report(reset: bool = False) -> dict:
    with _PROFILE_LOCK:
        snap = {s: {k: list(v) for k, v in d.items()} for s, d in PROFILE.items()}
        if reset: PROFILE.clear()
    if not snap:
        print("no profile data — call profile() first, then render something"); return {}
    totals: dict[str, float] = {}
    print(f"{'slot':<12}{'stage':<14}{'calls':>6}{'total s':>10}{'mean s':>9}")
    for slot in sorted(snap):
        for label, times in sorted(snap[slot].items()):
            tot = sum(times); totals[label] = totals.get(label, 0.0) + tot
            print(f"{slot:<12}{label:<14}{len(times):>6}{tot:>10.2f}{tot/len(times):>9.3f}")
    grand = sum(totals.values()) or 1.0
    print("-" * 51)
    for label, tot in sorted(totals.items(), key=lambda kv: -kv[1]):
        print(f"{'ALL':<12}{label:<14}{'':>6}{tot:>10.2f}{100*tot/grand:>8.1f}%")
    return snap

def start_load_async(t3: str = "auto", replicas_per_gpu: int | None = None,
                     validate: bool = True, warm: bool = True) -> dict:
    """Kick a load off in the background (used by the web page)."""
    if MODEL_STATE.get("status") in ("downloading", "loading", "testing", "warming"):
        return model_status()
    MODEL_STATE.update(status="downloading", message="queued…", started=time.time(),
                       finished=0.0, log=[])
    def _bg():
        try: load_models(t3=t3, replicas_per_gpu=replicas_per_gpu, validate=validate, warm=warm)
        except Exception: pass          # state/log already carry the error
    threading.Thread(target=_bg, name="cb-model-load", daemon=True).start()
    return model_status()

# ---------------------------------------------------------------------------
if AUTOLOAD:
    try:
        load_models(t3="auto", replicas_per_gpu=REPLICAS_PER_GPU)
    except Exception as e:
        print(f"\n!! automatic load failed: {e!s}\n"
              f"   Nothing is broken — run the remaining cells and press "
              f"“Load model” on the web page (or re-run this cell).")
else:
    print("AUTOLOAD is off — start the server and load the model from the web page.")


### 4½️⃣ (Optional) Performance tuning & profiling · skip on a first run
The speed work that matters is already automatic in step 4 — **per-token `tqdm` bars disabled** (GIL-serialised work + four bars fighting over one stdout; ~10 ms per 4×1200 decode steps, so a tidiness/contention win rather than a big speedup), **SDPA fused attention** requested per replica, a **multi-shape cuDNN warmup**, and a **VRAM headroom recommendation** telling you whether to raise *Replicas per GPU*.

This cell adds the measuring tools: `profile_run()` for the T3-decode vs S3Gen-vocode split, `bench_attention()` to A/B eager vs SDPA on your actual box, and an experimental, off-by-default `try_compile()`.


In [ ]:
# ── STEP 4½ · (Optional) performance tuning & profiling ──────────────────
# Nothing here is required — the defaults from step 4 are already applied:
#   · per-token tqdm bars disabled      (_silence_t3_tqdm, automatic)
#   · SDPA attention requested          (ATTN_IMPL, automatic per replica)
#   · multi-shape cuDNN warmup          (WARM_TEXTS, automatic after load)
#   · a VRAM headroom recommendation    (printed at the end of step 4)
# This cell is the knobs: measure where time goes, A/B the attention backend,
# and — if you want to go deep — an experimental torch.compile path.

import time, torch

# ---------------------------------------------------------------------------
# 1 · Where does a chunk's time actually go?  T3's autoregressive decode is
#     usually 80-90% of it; if so, vocoder-side tweaks are not worth your time.
# ---------------------------------------------------------------------------
def profile_run(text: str | None = None, language_id: str = "en"):
    """Profile one representative render, then print the per-stage split."""
    text = text or ("The quick brown fox jumps over the lazy dog. " * 6)
    profile(True)
    try:
        t0 = time.time()
        sr, wav, n = synth(text, language_id, None,
                           dict(exaggeration=0.5, cfg_weight=0.5, temperature=0.8,
                                repetition_penalty=1.2, min_p=0.05, top_p=1.0,
                                seed=0, gap_ms=120, chunk_chars=260))
        wall = time.time() - t0
        print(f"\n{n} chunks -> {len(wav)/sr:.2f}s audio in {wall:.2f}s wall "
              f"(RTF {wall/max(1e-6, len(wav)/sr):.2f}x) on {len(WORKERS)} worker(s)\n")
        profile_report()
        print("\nNote: stage totals sum across concurrent workers, so they can exceed\n"
              "wall time — read the percentages, not the absolute seconds.")
    finally:
        profile(False)

# ---------------------------------------------------------------------------
# 2 · A/B the attention backend. Step 4 already asks for SDPA, but on some
#     transformers builds that is a no-op (already default) and on others the
#     model refuses and stays eager. This times both so you know which you got.
# ---------------------------------------------------------------------------
def bench_attention(text: str | None = None, repeats: int = 2):
    text = text or ("A medium length sentence to prime the decode loop for typical chunks. " * 2)
    slot = WORKERS[0]
    m, dev = MODELS[slot], MODEL_DEVICE[slot]
    use_fp16 = FP16.get(slot, False) and dev.startswith("cuda")
    results = {}
    for impl in ("eager", "sdpa"):
        got = _set_attn_impl(m, impl)
        if got != impl:
            print(f"{impl:>6}: unavailable (still '{got}') — skipping"); continue
        warm_replica(slot, [text])                       # exclude autotune from the timing
        ts = []
        for _ in range(repeats):
            torch.cuda.synchronize() if dev.startswith("cuda") else None
            t0 = time.time()
            with m._cb_lock, torch.inference_mode(), \
                 torch.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
                torch.manual_seed(1234)                  # same tokens both ways
                m.generate(text, language_id="en")
            torch.cuda.synchronize() if dev.startswith("cuda") else None
            ts.append(time.time() - t0)
        results[impl] = min(ts)
        print(f"{impl:>6}: {min(ts):.2f}s (best of {repeats})")
    _set_attn_impl(m, ATTN_IMPL)                         # restore the default
    if len(results) == 2:
        e, s = results["eager"], results["sdpa"]
        ratio = e / s
        if ratio > 1.03:   verdict = f"sdpa is {ratio:.2f}x faster — keep it"
        elif ratio < 0.97: verdict = f"sdpa is {1/ratio:.2f}x SLOWER — set ATTN_IMPL='eager' and reload"
        else:              verdict = "no measurable difference (likely the same kernels either way)"
        print(f"\n-> {verdict}. Step 4 uses '{ATTN_IMPL}'; change ATTN_IMPL and reload to switch.")
    return results

# ---------------------------------------------------------------------------
# 3 · EXPERIMENTAL · torch.compile the decode step.
#
#     Honest assessment before you spend time here. Upstream's T3.inference()
#     does NOT use HF `generate()` — the HF path is commented out and replaced
#     by a hand-written loop that feeds a growing DynamicCache. That matters:
#       · the steady-state step really is a fixed shape (batch=2 for CFG,
#         seq_len=1), which is the good case for mode="reduce-overhead";
#       · but the cache object grows every step, so without swapping in a
#         StaticCache you get guard failures and recompiles rather than a
#         stable CUDA graph;
#       · CUDA graphs + our four concurrent worker threads on shared GPUs is
#         exactly where reduce-overhead tends to misbehave;
#       · first call pays 1-3 min of compile time, PER REPLICA.
#     So: real theoretical upside, genuinely fiddly, and it can be slower.
#     Left OFF by default. Try it on ONE replica and time it honestly.
# ---------------------------------------------------------------------------
def try_compile(slot: str | None = None, mode: str = "default"):
    """Compile one replica's backbone. Returns True if it survived a real render."""
    slot = slot or WORKERS[0]
    m, dev = MODELS[slot], MODEL_DEVICE[slot]
    if not dev.startswith("cuda"):
        print("compile only makes sense on CUDA — skipping"); return False
    if getattr(m, "_cb_compiled", False):
        print(f"{slot}: already compiled"); return True
    use_fp16 = FP16.get(slot, False)
    probe = "A medium length sentence to prime the decode loop for typical chunks. " * 2

    torch.cuda.synchronize(); t0 = time.time()
    with m._cb_lock, torch.inference_mode(), \
         torch.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
        torch.manual_seed(99); m.generate(probe, language_id="en")
    torch.cuda.synchronize(); before = time.time() - t0
    print(f"{slot}: baseline {before:.2f}s")

    original = m.t3.tfmr
    try:
        m.t3.tfmr = torch.compile(original, mode=mode, dynamic=True)
        torch.cuda.synchronize(); t0 = time.time()
        with m._cb_lock, torch.inference_mode(), \
             torch.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
            torch.manual_seed(99); m.generate(probe, language_id="en")
        torch.cuda.synchronize(); first = time.time() - t0
        print(f"{slot}: first compiled call {first:.2f}s (includes compilation)")

        torch.cuda.synchronize(); t0 = time.time()
        with m._cb_lock, torch.inference_mode(), \
             torch.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
            torch.manual_seed(99); wav = m.generate(probe, language_id="en")
        torch.cuda.synchronize(); after = time.time() - t0
        import numpy as _np
        w = _np.asarray(wav.squeeze().float().cpu(), dtype=_np.float32)
        if not (w.size and _np.isfinite(w).all() and float(w.std()) > 1e-5):
            raise RuntimeError("compiled model produced bad audio")
        print(f"{slot}: warm compiled {after:.2f}s -> {before/after:.2f}x vs baseline")
        if after >= before:
            print(f"{slot}: no win — reverting"); m.t3.tfmr = original; return False
        m._cb_compiled = True
        print(f"{slot}: keeping the compiled backbone "
              f"(pays back after ~{max(1, int(first/max(before-after, 1e-6)))} renders)")
        return True
    except Exception as e:
        m.t3.tfmr = original
        print(f"{slot}: compile failed, reverted cleanly: {e!s}")
        return False

print("tuning helpers ready:")
print("  profile_run()      · per-stage time split (T3 decode vs S3Gen vocode)")
print("  bench_attention()  · time eager vs sdpa on this box")
print("  try_compile()      · EXPERIMENTAL torch.compile on one replica")
print(f"\ncurrent: attn={sorted({MODELS[s].__dict__.get('_cb_attn','?') for s in WORKERS})} "
      f"· workers={len(WORKERS)} · fp16={ {s: FP16.get(s) for s in WORKERS} }")
if MODEL_STATE.get("suggested_replicas_per_gpu"):
    print(f"VRAM suggestion from step 4: {MODEL_STATE['suggested_replicas_per_gpu']} replica(s)/GPU")


### 5️⃣ Parallel synthesis engine · ~5 s (self-test)
Sentence-aware chunks enter a shared queue. Each independent model worker picks the next part, so up to four chunks can synthesize concurrently while final audio stays in chapter order. The self-test only runs if a model is already resident — otherwise the engine just defines itself and waits for you to load weights from the app.


In [ ]:
# ── STEP 5 · Parallel synthesis engine (work-stealing across GPUs) ──────

import re, io, time, threading, queue as pyq
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import soundfile as sf

# The engine reads the sample rate live: the model can be (re)loaded from the
# web page after this cell has already run, so nothing may be resident yet.
SAMPLE_RATE = globals().get("MODEL_SR", 24000)

def _sr() -> int:
    return int(globals().get("MODEL_SR", SAMPLE_RATE) or 24000)

# Sentence boundaries incl. CJK/Arabic punctuation; blank lines always split.
_SENT_END = re.compile(r"(?<=[\.\!\?。！？!?；;…\u060C\u061F])\s+|\n+")

def split_script(text: str, max_chars: int = 260) -> list[str]:
    text = (text or "").strip()
    if not text: return []
    out = []
    for s in (x.strip() for x in _SENT_END.split(text)):
        if not s: continue
        if len(s) <= max_chars:
            out.append(s); continue
        # long sentence: split on clause punctuation, then on whitespace
        buf = ""
        for piece in re.split(r"(?<=[,;:，；：、])\s*|\s+", s):
            if not piece: continue
            if len(buf) + len(piece) + 1 > max_chars and buf:
                out.append(buf.strip()); buf = piece
            else:
                buf = f"{buf} {piece}".strip()
        if buf: out.append(buf.strip())
    # absolute hard cap — even an unbreakable token (URL, hash, code, a
    # space-less CJK run) is split at the requested max_per_chunk, so per-chunk
    # latency stays bounded and T3 never sees a chunk it wasn't tuned for.
    hard = []
    for c in out:
        while len(c) > max_chars:
            hard.append(c[: max_chars]); c = c[max_chars:]
        hard.append(c)
    return [c for c in hard if c]

def _edge_fade(w: np.ndarray, sr: int, ms: int = 5) -> np.ndarray:
    """5ms in/out fade kills boundary clicks when chunks abut."""
    n = int(sr * ms / 1000)
    if n <= 0 or len(w) <= 2 * n: return w
    w = w.astype(np.float32, copy=True)
    ramp = np.linspace(0.0, 1.0, n, dtype=np.float32)
    w[:n] *= ramp
    w[-n:] *= ramp[::-1]
    return w

# ---------------------------------------------------------------------------
# Voice conditionals cache — per replica, per voice clip.
# prepare_conditionals() builds s3-token prompt + xvector + speaker emb; the
# per-call exaggeration delta is patched inside generate() by upstream, so we
# only pay the expensive embedding pass once per clip. The cache is keyed by
# voice clip (NOT by whatever this slot was last used with): concurrent jobs
# with different cloned voices — an EPUB batch plus a manual generate, say —
# swap pre-built Conditionals in and out with a pointer move instead of
# re-embedding the reference clip on every interleave. Same _cb_lock still
# guards the swap, so no audio is corrupted; we are just fixing the thrash.
# ---------------------------------------------------------------------------
def ensure_conditionals(model, voice_ref: str | None, exaggeration: float):
    cache = getattr(model, "_cb_conds_cache", None)
    if cache is None:                  # model loaded before this cell ran
        cache = model._cb_conds_cache = {}
    if voice_ref:                      # cloned voice
        conds = cache.get(voice_ref)
        if conds is None:              # first use of this clip: embed it once
            model.prepare_conditionals(voice_ref, exaggeration=exaggeration)
            conds = model.conds
            cache[voice_ref] = conds
        elif model.conds is not conds: # another voice is installed: cheap swap
            model.conds = conds
        model._cb_voice_path = voice_ref
    else:                              # built-in conds.pt voice
        if model.conds is not model._cb_default_conds:
            model.conds = model._cb_default_conds
            model._cb_voice_path = None

# ---------------------------------------------------------------------------
# Work-stealing synthesis: one worker thread per GPU pulls from a shared queue.
# ---------------------------------------------------------------------------
def synth(text: str, language_id: str, voice_ref: str | None, params: dict,
          devices=None, on_progress=None, is_cancelled=None):
    devices = list(devices or WORKERS)
    if not devices:
        raise RuntimeError("no model is loaded — press “Load model” in the app "
                           "(or run the model manager cell)")
    is_cancelled = is_cancelled or (lambda: False)
    sr = _sr()
    chunks = split_script(text, params.get("chunk_chars", 260))
    total = len(chunks)
    if total == 0:
        return sr, np.zeros(1, dtype=np.float32), 0

    work: pyq.Queue = pyq.Queue()
    for i, c in enumerate(chunks):
        work.put((i, c, 0))                       # (index, text, attempts)

    results: dict[int, np.ndarray] = {}
    done = [0]
    errors: list[str] = []
    lock = threading.Lock()

    def worker(slot: str):
        dev = MODEL_DEVICE[slot]
        if dev.startswith("cuda"):
            # pin this thread's CUDA context so no op can fall back to cuda:0
            torch.cuda.set_device(int(dev.split(":")[1]))
        model = MODELS[slot]
        use_fp16 = FP16.get(slot, False) and dev.startswith("cuda")
        while True:
            if is_cancelled():
                return
            try:
                i, chunk, attempts = work.get_nowait()
            except pyq.Empty:
                return
            try:
                seed = int(params.get("seed", 0))
                if seed:                               # deterministic per chunk
                    torch.manual_seed(seed + i)
                    if torch.cuda.is_available():
                        torch.cuda.manual_seed_all(seed + i)
                # Acquire the replica in small slices so a Stop can also
                # preempt a job waiting behind another job's chunk, not just
                # the chunk currently rendering.
                while not model._cb_lock.acquire(timeout=0.2):
                    if is_cancelled():
                        return
                try:
                    # Register this job's cancel flag as the tqdm polling hook
                    # (see the model-manager cell): upstream's decode/vocoder
                    # loops yield once per token, so a Stop lands between
                    # tokens — the already-dispatched chunk is genuinely
                    # abandoned, not silently finished.
                    _set_cancel_hook(is_cancelled)
                    # Voice embeddings stay FP32 even when decode is FP16:
                    # prepare runs outside autocast, matching upstream's own
                    # generate() where it primes conds before the cast.
                    ensure_conditionals(model, voice_ref, params["exaggeration"])
                    with torch.inference_mode(), \
                         torch.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
                        wav = model.generate(
                            chunk,
                            language_id=language_id,
                            audio_prompt_path=None,    # conds already primed
                            exaggeration=params["exaggeration"],
                            cfg_weight=params["cfg_weight"],
                            temperature=params["temperature"],
                            repetition_penalty=params["repetition_penalty"],
                            min_p=params["min_p"],
                            top_p=params["top_p"],
                        )
                finally:
                    _clear_cancel_hook()
                    model._cb_lock.release()
                w = np.asarray(wav.squeeze().detach().float().cpu(), dtype=np.float32)
                if w.size == 0 or not np.isfinite(w).all():
                    raise RuntimeError("model returned bad audio")
                with lock:
                    results[i] = w
                    done[0] += 1
                    n_done = done[0]
                if on_progress:
                    on_progress(n_done, total)
            except Exception as e:
                if is_cancelled():
                    return
                if attempts < 1:                       # retry once, likely on the other GPU
                    work.put((i, chunk, attempts + 1))
                else:
                    with lock:
                        errors.append(f"chunk {i}: {e!s}")
                    return

    threads = [threading.Thread(target=worker, args=(d,), daemon=True,
                                name=f"cb-{d}") for d in devices]
    for t in threads: t.start()
    while any(t.is_alive() for t in threads):
        for t in threads: t.join(timeout=0.2)
        if is_cancelled():                             # workers drop out at next chunk
            for t in threads: t.join()
            raise RuntimeError("cancelled")
    if errors:
        raise RuntimeError(errors[0])

    gap = np.zeros(int(sr * params.get("gap_ms", 120) / 1000), dtype=np.float32)
    parts: list[np.ndarray] = []
    for i in range(total):
        parts.append(_edge_fade(results.get(i, np.zeros(1, dtype=np.float32)), sr))
        if i != total - 1:
            parts.append(gap)
    return sr, np.concatenate(parts), total

# ---------------------------------------------------------------------------
# Encoding on the CPU pool, concurrent with whatever the GPUs are doing next.
# ---------------------------------------------------------------------------
_old = globals().get("_enc_pool")
if _old is not None:                     # notebook re-run: retire the old pool
    try: _old.shutdown(wait=False, cancel_futures=True)
    except Exception: pass
_enc_pool = ThreadPoolExecutor(max_workers=max(2, N_CPU // 2), thread_name_prefix="enc")

def encode(sr: int, wav: np.ndarray, fmt: str) -> tuple[bytes, str]:
    fmt = (fmt or "wav").lower()
    buf = io.BytesIO()
    if fmt == "mp3":
        from pydub import AudioSegment
        pcm = (np.clip(wav, -1, 1) * 32767).astype(np.int16).tobytes()
        AudioSegment(pcm, frame_rate=sr, sample_width=2, channels=1) \
            .export(buf, format="mp3", bitrate="160k", parameters=["-threads", str(N_CPU)])
        return buf.getvalue(), "audio/mpeg"
    if fmt == "opus":
        sf.write(buf, wav, sr, format="OGG", subtype="OPUS"); return buf.getvalue(), "audio/ogg"
    if fmt == "flac":
        sf.write(buf, wav, sr, format="FLAC"); return buf.getvalue(), "audio/flac"
    sf.write(buf, wav, sr, format="WAV", subtype="PCM_16"); return buf.getvalue(), "audio/wav"

def encode_async(sr: int, wav: np.ndarray, fmt: str):
    return _enc_pool.submit(encode, sr, wav, fmt)

# ---- sanity check: only when a model is already resident -------------------
if globals().get("WORKERS"):
    t0 = time.time()
    sr, w, n = synth("Chatterbox speaks fast on two GPUs. Parallel chunks keep both busy. "
                     "This is a self test, so short. And one more sentence for the queue.",
                     "en", None,
                     dict(exaggeration=0.5, cfg_weight=0.5, temperature=0.8,
                          repetition_penalty=1.2, min_p=0.05, top_p=1.0,
                          seed=0, gap_ms=120, chunk_chars=260))
    print(f"synth ok: {n} chunks -> {len(w)/sr:.2f}s audio in {time.time()-t0:.2f}s wall "
          f"on {len(WORKERS)} model worker(s)")
else:
    print("engine ready — no model resident yet; load it from the app (or the model manager cell)")


### 5½️⃣ EPUB + Google Drive resilience
Optional but recommended. Configure Drive once using a Kaggle secret containing a **service-account JSON** and share the target Drive folder with that service account email. This is safer and more reliable in Kaggle than browser OAuth. Completed audio is always persisted locally first.


In [ ]:
# ── STEP 5½ · EPUB helpers + durable output/Google Drive mirror ───────────
# Kaggle Add-ons → Secrets: GOOGLE_DRIVE_SERVICE_ACCOUNT_JSON and
# GOOGLE_DRIVE_FOLDER_ID. Share that Drive folder with the service account.
import os, re, json, threading
from pathlib import Path
from ebooklib import epub, ITEM_DOCUMENT
from bs4 import BeautifulSoup

OUTPUT_DIR = Path("/kaggle/working/chatterbox_outputs"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EPUB_DIR = Path("/kaggle/working/chatterbox_epubs"); EPUB_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_FOLDER_ID = os.getenv("GOOGLE_DRIVE_FOLDER_ID", "").strip()
DRIVE_SERVICE = None
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    _secret = _secrets.get_secret("GOOGLE_DRIVE_SERVICE_ACCOUNT_JSON")
    if not DRIVE_FOLDER_ID:
        try: DRIVE_FOLDER_ID = _secrets.get_secret("GOOGLE_DRIVE_FOLDER_ID").strip()
        except Exception: pass
    if _secret and DRIVE_FOLDER_ID:
        from google.oauth2 import service_account
        from googleapiclient.discovery import build
        _info = json.loads(_secret)
        _creds = service_account.Credentials.from_service_account_info(_info, scopes=["https://www.googleapis.com/auth/drive.file"])
        DRIVE_SERVICE = build("drive", "v3", credentials=_creds, cache_discovery=False)
        print("Google Drive mirror enabled")
    else: print("Google Drive mirror off (secrets/folder ID not configured)")
except Exception as e: print(f"Google Drive mirror off: {e!s}")

def safe_filename(title, fallback="chapter"):
    title = re.sub(r'[\\/:*?"<>|\x00-\x1f]+', " ", title or "")
    title = re.sub(r"\s+", " ", title).strip(". ")
    return (title or fallback)[:140]

def mirror_to_drive(path: Path):
    if not DRIVE_SERVICE: return None
    from googleapiclient.http import MediaFileUpload
    media = MediaFileUpload(str(path), resumable=True)
    return DRIVE_SERVICE.files().create(body={"name": path.name, "parents": [DRIVE_FOLDER_ID]}, media_body=media, fields="id").execute()["id"]

def _resolve(book, idref: str):
    """Map a spine idref to a manifest item.
    get_item_with_id() is the normal path; some writers (including
    ebooklib.write_epub itself: it renumbers manifest ids to chapter_0/1/…
    but leaves the ORIGINAL ids in the spine) break that link, so fall back
    to matching the href stem."""
    item = book.get_item_with_id(idref)
    if item is not None:
        return item
    for it in book.get_items_of_type(ITEM_DOCUMENT):
        if Path(it.get_name()).stem == idref:
            return it
    return None

def epub_chapters(path: Path):
    book = epub.read_epub(str(path))
    # Walk the SPINE, never the manifest: the spine is the book's reading order
    # (and may repeat items), while get_items_of_type(ITEM_DOCUMENT) returns
    # manifest order — cover/TOC/front matter first, indexes, anything the
    # publisher listed late. This is the documented ebooklib gotcha.
    ordered, seen = [], set()
    for ref in book.spine or []:
        idref = ref[0] if isinstance(ref, (tuple, list)) else ref
        if idref in seen: continue
        seen.add(idref)
        item = _resolve(book, idref)
        if item is None or getattr(item, "get_type", lambda: ITEM_DOCUMENT)() != ITEM_DOCUMENT:
            continue
        ordered.append(item)
    # Some legacy/malformed books have no spine at all — fall back to manifest.
    if not ordered:
        ordered = list(book.get_items_of_type(ITEM_DOCUMENT))
    chapters=[]
    for item in ordered:
        # Structural documents, not chapters. Blocking them keeps a "Convert
        # all" from rendering the table of contents as the first "chapter".
        base = Path(item.get_name()).name.lower()
        if base in ("nav.xhtml", "nav.htm", "toc.xhtml", "toc.htm", "toc.ncx"):
            continue
        soup=BeautifulSoup(item.get_content(), "html.parser")
        for x in soup(["script", "style"]): x.decompose()
        text=soup.get_text(" ", strip=True)
        if len(text) < 40: continue
        heading=soup.find(["h1","h2","h3","title"])
        title=heading.get_text(" ", strip=True) if heading else Path(item.get_name()).stem
        chapters.append({"id": item.get_id(), "title": title or f"Chapter {len(chapters)+1}", "text": text})
    return chapters
print(f"Durable local output: {OUTPUT_DIR}")


### 6️⃣ Build the mobile app UI · instant
The app supports regular scripts, voice clones, and the EPUB panel added below.


In [ ]:
# ── STEP 6 · Build the mobile app UI (inlined single-file frontend) ──────

INDEX_HTML = r"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover, maximum-scale=5">
<meta name="theme-color" content="#0d0b14">
<meta name="apple-mobile-web-app-capable" content="yes">
<meta name="apple-mobile-web-app-status-bar-style" content="black-translucent">
<meta name="apple-mobile-web-app-title" content="Chatterbox">
<meta name="mobile-web-app-capable" content="yes">
<link rel="manifest" href="/manifest.webmanifest">
<link rel="icon" href="/icon.svg" type="image/svg+xml">
<link rel="apple-touch-icon" href="/icon-180.png"><!-- iOS ignores SVG touch icons; PNG only -->
<title>Chatterbox TTS · GPU</title>
<style>
:root{
  --bg:#0d0b14; --surface:#17141f; --surface-2:#1f1b2b; --line:#2b2540;
  --text:#eeeaf6; --muted:#a49eb8; --accent:#a78bfa; --accent-2:#8b5cf6;
  --good:#4ade80; --warn:#f59e0b; --bad:#ef4444;
  --r:14px; --r-sm:10px; --tap:48px;
  --sat:env(safe-area-inset-top); --sab:env(safe-area-inset-bottom);
  --sal:env(safe-area-inset-left); --sar:env(safe-area-inset-right);
}
@media(prefers-color-scheme: light){
  :root{ --bg:#f6f5fb; --surface:#ffffff; --surface-2:#efecf8; --line:#e2ddf2;
         --text:#171226; --muted:#655e7d; --accent:#7c5bf0; --accent-2:#6941e0; }
}
*{ box-sizing:border-box; -webkit-tap-highlight-color:transparent; }
html,body{ margin:0; padding:0; background:var(--bg); color:var(--text);
  font: 16px/1.45 -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue",
        Arial, "Noto Sans", sans-serif; -webkit-font-smoothing:antialiased;
  overscroll-behavior-y:none; }
body{ min-height:100dvh; padding-left:var(--sal); padding-right:var(--sar);
  padding-bottom: calc(var(--sab) + 92px); }
button, input, textarea, select{ font:inherit; color:inherit; }
button{ background:none; border:0; cursor:pointer; }

/* header */
header{ position:sticky; top:0; z-index:5; background:color-mix(in oklab, var(--bg) 88%, transparent);
  backdrop-filter:saturate(140%) blur(10px); -webkit-backdrop-filter:saturate(140%) blur(10px);
  padding: calc(var(--sat) + 10px) 16px 10px; border-bottom:1px solid var(--line);
  display:flex; align-items:center; gap:10px; }
.logo{ width:32px; height:32px; border-radius:9px; background:
  linear-gradient(135deg, var(--accent), var(--accent-2)); display:grid; place-items:center;
  color:#fff; font-weight:800; font-size:14px; }
.title{ font-weight:700; font-size:16px; letter-spacing:.2px; }
.sub{ font-size:11px; color:var(--muted); margin-top:1px; }
.pill{ margin-left:auto; display:inline-flex; align-items:center; gap:6px;
  padding:6px 10px; background:var(--surface-2); border:1px solid var(--line);
  border-radius:999px; font-size:12px; color:var(--muted); }
.dot{ width:8px; height:8px; border-radius:50%; background:var(--good); box-shadow:0 0 0 3px color-mix(in oklab, var(--good) 30%, transparent);}

/* main */
main{ max-width: 680px; margin: 0 auto; padding: 12px 14px; display:flex; flex-direction:column; gap:12px; }
.card{ background:var(--surface); border:1px solid var(--line); border-radius:var(--r); padding:14px; }
.label{ font-size:12px; color:var(--muted); text-transform:uppercase; letter-spacing:.6px; margin-bottom:8px; display:flex; justify-content:space-between; align-items:center; }

/* picker cards (language / voice) */
.pickgrid{ display:grid; grid-template-columns: 1fr 1fr; gap:10px; }
.pick{ display:flex; align-items:center; gap:12px; padding:12px; min-height:var(--tap); text-align:left; }
.pick .avatar{ width:44px; height:44px; border-radius:50%; flex:0 0 auto;
  background:linear-gradient(135deg, var(--accent), var(--accent-2));
  color:#fff; display:grid; place-items:center; font-weight:700; font-size:16px; }
.pick .who{ flex:1; min-width:0; }
.pick .who b{ display:block; font-size:15px; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }
.pick .who span{ font-size:11.5px; color:var(--muted); display:block; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }
.pick .chev{ color:var(--muted); font-size:20px; }
.pick .k{ font-size:10px; text-transform:uppercase; letter-spacing:.6px; color:var(--muted); margin-bottom:2px; }
@media (max-width:420px){ .pickgrid{ grid-template-columns:1fr; } }

/* script */
textarea{ width:100%; min-height:40vh; max-height:62vh; background:var(--surface);
  border:1px solid var(--line); border-radius:var(--r); padding:14px; color:var(--text);
  font-size:16px; line-height:1.5; resize:vertical; }
textarea:focus{ outline:2px solid var(--accent); outline-offset:1px; }
.meter{ display:flex; justify-content:space-between; font-size:12px; color:var(--muted); padding:6px 4px 0; }

/* controls */
.row{ display:grid; grid-template-columns: 1fr auto; gap:10px; align-items:center; margin-bottom:12px; }
.row:last-child{ margin-bottom:0; }
.row .name{ font-size:14px; }
.row .hint{ display:block; font-size:11px; color:var(--muted); margin-top:1px; }
.val{ font-variant-numeric: tabular-nums; color:var(--muted); font-size:13px; min-width:56px; text-align:right; }
input[type=range]{ -webkit-appearance:none; appearance:none; width:100%; height:36px; background:transparent; }
input[type=range]::-webkit-slider-runnable-track{ height:6px; border-radius:3px; background:var(--surface-2); }
input[type=range]::-moz-range-track{ height:6px; border-radius:3px; background:var(--surface-2); }
input[type=range]::-webkit-slider-thumb{ -webkit-appearance:none; appearance:none; width:26px; height:26px; border-radius:50%;
  background:var(--accent); margin-top:-10px; border:3px solid var(--bg); box-shadow:0 2px 6px rgba(0,0,0,.3);}
input[type=range]::-moz-range-thumb{ width:22px; height:22px; border-radius:50%; background:var(--accent); border:3px solid var(--bg); }
input[type=number]{ width:100%; min-height:44px; padding:0 12px; background:var(--surface-2); border:1px solid var(--line); border-radius:10px; }

details.adv summary{ list-style:none; display:flex; align-items:center; gap:8px; min-height:var(--tap);
  color:var(--muted); font-size:14px; font-weight:600; cursor:pointer; }
details.adv summary::-webkit-details-marker{ display:none; }
details.adv summary .chev{ transition:transform .15s ease; font-size:18px; }
details.adv[open] summary .chev{ transform:rotate(90deg); }
details.adv .body{ padding-top:4px; }

.segmented{ display:grid; grid-auto-flow:column; grid-auto-columns:1fr; gap:6px; background:var(--surface-2); padding:4px;
  border-radius:12px; border:1px solid var(--line); }
.segmented button{ min-height:40px; border-radius:9px; font-weight:600; font-size:13px; color:var(--muted); }
.segmented button.on{ background:var(--surface); color:var(--text); box-shadow:0 1px 2px rgba(0,0,0,.15); }

/* sticky action */
.fab-wrap{ position:fixed; left:0; right:0; bottom:0; padding: 10px 14px calc(var(--sab) + 10px);
  background: linear-gradient(to top, var(--bg) 55%, transparent);
  z-index:4; }
.fab-wrap .inner{ max-width:680px; margin:0 auto; display:flex; gap:10px; align-items:center; }
.fab{ flex:1; min-height:56px; background:var(--accent); color:#fff; border-radius:14px;
  font-weight:700; font-size:16px; letter-spacing:.2px; display:inline-flex; align-items:center; justify-content:center; gap:10px;
  box-shadow: 0 6px 20px rgba(124,91,240,.35); transition: transform .06s ease; }
.fab:active{ transform: scale(.98); }
.fab[disabled]{ background:var(--surface-2); color:var(--muted); box-shadow:none; }
.fab .spinner{ width:18px; height:18px; border-radius:50%; border:2px solid rgba(255,255,255,.4); border-top-color:#fff; animation:spin .8s linear infinite; }
@keyframes spin{ to{ transform:rotate(360deg); } }
.icon-btn{ width:56px; height:56px; border-radius:14px; background:var(--surface); border:1px solid var(--line); display:grid; place-items:center; color:var(--text); font-size:18px; }

/* progress bar */
.progress{ position:fixed; top:0; left:0; right:0; height:3px; background:transparent; z-index:10; pointer-events:none; }
.progress .bar{ height:100%; width:0%; background:linear-gradient(90deg, var(--accent), var(--accent-2)); transition: width .2s ease; }

/* audio result */
.result{ display:none; }
.result.on{ display:block; }
.result audio{ width:100%; margin-top:8px; }
.result .stats{ display:flex; gap:12px; font-size:12px; color:var(--muted); flex-wrap:wrap; margin-top:8px; }
.result .stats b{ color:var(--text); font-weight:600; }
.result .dlrow{ display:flex; gap:8px; margin-top:10px; }
.btn{ flex:1; min-height:44px; background:var(--surface-2); border:1px solid var(--line);
  border-radius:10px; font-weight:600; font-size:14px; color:var(--text); display:inline-flex; align-items:center; justify-content:center; gap:8px; text-decoration:none; }
.btn.primary{ background:var(--accent); color:#fff; border-color:transparent; }

/* history */
.hist-item{ display:flex; align-items:center; gap:10px; padding:10px; border-radius:10px; }
.hist-item + .hist-item{ border-top:1px solid var(--line); border-radius:0; }
.hist-item .who{ flex:1; min-width:0; }
.hist-item .who b{ display:block; font-size:14px; }
.hist-item .who span{ font-size:12px; color:var(--muted); white-space:nowrap; overflow:hidden; text-overflow:ellipsis; display:block; }
.hist-item .play{ width:40px; height:40px; border-radius:50%; background:var(--accent); color:#fff; display:grid; place-items:center; }

/* bottom sheet */
.sheet-back{ position:fixed; inset:0; background:rgba(0,0,0,.55); opacity:0; pointer-events:none; transition:opacity .18s ease; z-index:20; }
.sheet-back.on{ opacity:1; pointer-events:auto; }
.sheet{ position:fixed; left:0; right:0; bottom:0; z-index:21; background:var(--surface); border-top-left-radius:20px; border-top-right-radius:20px;
  transform: translateY(100%); transition: transform .22s cubic-bezier(.2,.8,.2,1);
  max-height: 88dvh; display:flex; flex-direction:column; padding-bottom: var(--sab); }
.sheet.on{ transform: translateY(0); }
.sheet .grabber{ width:44px; height:5px; background:var(--line); border-radius:3px; margin: 8px auto 4px; }
.sheet .head{ display:flex; align-items:center; padding: 6px 14px 10px; gap:10px; border-bottom:1px solid var(--line); }
.sheet .head b{ font-size:16px; }
.sheet .head button{ margin-left:auto; color:var(--muted); font-size:15px; min-height:44px; padding: 0 10px; }
.sheet .search{ padding: 10px 14px; border-bottom:1px solid var(--line); }
.sheet .search input{ width:100%; min-height:44px; padding: 0 12px; background:var(--surface-2); border:1px solid var(--line); border-radius:10px; }
.sheet .list{ overflow-y:auto; padding: 6px 8px 8px; }
.sheet .group{ font-size:11px; color:var(--muted); text-transform:uppercase; letter-spacing:.7px; padding: 12px 8px 6px; }
.sheet .item{ display:flex; align-items:center; gap:12px; padding:12px 10px; border-radius:12px; min-height:var(--tap); width:100%; text-align:left; }
.sheet .item:active{ background: var(--surface-2); }
.sheet .item.on{ background: color-mix(in oklab, var(--accent) 15%, transparent); }
.sheet .item .avatar{ width:36px; height:36px; border-radius:50%; background:linear-gradient(135deg, var(--accent), var(--accent-2)); color:#fff; display:grid; place-items:center; font-weight:700; font-size:14px; }
.sheet .item .meta b{ display:block; font-size:15px; }
.sheet .item .meta span{ font-size:12px; color:var(--muted); }
.sheet .item .check{ margin-left:auto; color:var(--accent); opacity:0; }
.sheet .item.on .check{ opacity:1; }
.sheet .item .del{ margin-left:auto; color:var(--muted); font-size:18px; padding:6px 10px; }
.sheet .upload{ display:flex; align-items:center; justify-content:center; gap:10px; margin:10px 8px;
  min-height:var(--tap); border:1.5px dashed var(--line); border-radius:12px; color:var(--accent);
  font-weight:600; font-size:14px; width:calc(100% - 16px); }
.sheet .upload .spinner{ width:16px; height:16px; border-radius:50%; border:2px solid color-mix(in oklab, var(--accent) 40%, transparent); border-top-color:var(--accent); animation:spin .8s linear infinite; }
.sheet .note{ font-size:11.5px; color:var(--muted); padding:4px 14px 0; }

/* toast */
.toast{ position:fixed; left:14px; right:14px; bottom: calc(var(--sab) + 110px); z-index:30;
  background:var(--bad); color:#fff; padding:12px 14px; border-radius:12px; box-shadow: 0 8px 30px rgba(0,0,0,.3);
  transform: translateY(20px); opacity:0; transition: all .2s ease; pointer-events:none; text-align:center; }
.toast.on{ transform:none; opacity:1; }

.hide{ display:none !important; }

/* model card */
.badge{ font-size:11px; padding:3px 9px; border-radius:999px; background:var(--surface-2);
  border:1px solid var(--line); color:var(--muted); text-transform:none; letter-spacing:0; }
.badge.ok{ background:color-mix(in oklab, var(--good) 22%, transparent); color:var(--text); }
.badge.work{ background:color-mix(in oklab, var(--warn) 25%, transparent); color:var(--text); }
.badge.err{ background:color-mix(in oklab, var(--bad) 25%, transparent); color:var(--text); }
select.mini{ min-height:40px; padding:0 10px; background:var(--surface-2); border:1px solid var(--line);
  border-radius:10px; color:var(--text); }
.hint.tip{ padding:8px 10px; border-radius:9px;
  background:color-mix(in oklab, var(--accent) 14%, transparent);
  border:1px solid color-mix(in oklab, var(--accent) 35%, transparent); color:var(--text); }
.linkbtn{ color:var(--accent); font-weight:700; text-decoration:underline; font-size:12px; padding:2px 0; }
pre.log{ margin:10px 0 0; padding:10px; background:var(--surface-2); border:1px solid var(--line);
  border-radius:10px; font-size:11px; line-height:1.4; max-height:150px; overflow:auto;
  white-space:pre-wrap; word-break:break-word; color:var(--muted); }

/* epub queue */
.qrow{ display:flex; align-items:center; gap:8px; font-size:12.5px; padding:6px 0; border-bottom:1px solid var(--line); }
.qrow:last-child{ border-bottom:0; }
.qrow .qname{ flex:1; min-width:0; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }
.qrow .qstate{ color:var(--muted); font-variant-numeric:tabular-nums; }
.qrow.done .qstate{ color:var(--good); }
.qrow.error .qstate{ color:var(--bad); }
</style>
</head>
<body>
  <div class="progress"><div class="bar" id="pbar"></div></div>

  <header>
    <div class="logo">C</div>
    <div>
      <div class="title">Chatterbox TTS</div>
      <div class="sub" id="sub">connecting…</div>
    </div>
    <div class="pill"><span class="dot" id="statusDot"></span><span id="statusText">…</span></div>
  </header>

  <main>
    <div class="card" id="modelCard">
      <div class="label"><span>Model</span><span id="modelBadge" class="badge">checking…</span></div>
      <div class="hint" id="modelMsg" style="margin-bottom:6px">…</div>
      <div class="hint hide" id="modelTip" style="margin-bottom:10px"></div>
      <div id="modelForm">
        <div class="row">
          <div class="name">Checkpoint<span class="hint">V3 is the newest multilingual model</span></div>
          <select id="modelT3" class="mini">
            <option value="auto">Auto (V3 → V2)</option>
            <option value="v3">V3</option>
            <option value="v2">V2</option>
          </select>
        </div>
        <div class="row">
          <div class="name">Replicas per GPU<span class="hint">more = more parallel chunks, more VRAM</span></div>
          <select id="modelReps" class="mini">
            <option value="1">1</option><option value="2" selected>2</option>
            <option value="3">3</option><option value="4">4</option>
          </select>
        </div>
        <div class="dlrow">
          <button class="btn primary" id="modelLoad">Load model</button>
          <button class="btn" id="modelUnload">Unload</button>
        </div>
      </div>
      <pre id="modelLog" class="log hide"></pre>
    </div>

    <div class="pickgrid">
      <button class="card pick" id="langBtn" aria-label="Choose language">
        <div>
          <div class="k">Language</div>
          <div class="who"><b id="lName">🇬🇧 English</b><span id="lMeta">English</span></div>
        </div>
        <div class="chev">›</div>
      </button>
      <button class="card pick" id="voiceBtn" aria-label="Choose voice">
        <div class="avatar" id="vAvatar">C</div>
        <div style="min-width:0; flex:1;">
          <div class="k">Voice</div>
          <div class="who"><b id="vName">Default voice</b><span id="vMeta">built-in conds.pt</span></div>
        </div>
        <div class="chev">›</div>
      </button>
    </div>

    <div>
      <textarea id="script" spellcheck="true" autocapitalize="sentences"
        enterkeyhint="enter" inputmode="text"
        placeholder="Paste or type your script here — any of 23 languages…">Chatterbox speaks twenty-three languages, clones any voice from a few seconds of reference audio, and on this Kaggle box it renders across two T4 GPUs in parallel. Try a long paragraph — you will watch both GPUs share the work.</textarea>
      <div class="meter"><span id="charCount">0 chars</span><span id="estAudio">≈ 0s audio</span></div>
    </div>

    <div class="card">
      <div class="label">EPUB → chapter audio</div>
      <div class="hint" style="margin:6px 0 10px">Upload a book, tick the chapters to render. Each finished file is named after its chapter and saved locally (and mirrored to Drive if enabled).</div>
      <input type="file" id="epubFile" accept=".epub,application/epub+zip" style="display:none">
      <button class="btn" id="epubPick">Upload EPUB</button>
      <div id="epubInfo" class="hint" style="margin-top:9px"></div>
      <div id="epubTools" class="dlrow hide" style="margin-top:8px">
        <button class="btn" id="epubAll">Select all</button>
        <button class="btn" id="epubNone">Clear</button>
      </div>
      <div id="epubChapters" style="max-height:230px;overflow:auto;margin-top:8px"></div>
      <button class="btn primary" id="epubGo" style="display:none;margin-top:10px">Convert selected chapters</button>
      <div id="epubQueue" style="margin-top:10px"></div>
    </div>

    <div class="card">
      <div class="row">
        <div class="name">Exaggeration<span class="hint">0 = flat · 0.5 = neutral · 1+ = theatrical</span></div>
        <div class="val" id="exVal">0.50</div>
      </div>
      <input type="range" id="exaggeration" min="0" max="2" step="0.05" value="0.5" style="margin-bottom:14px">
      <div class="row">
        <div class="name">CFG / pace<span class="hint">lower = faster speech · higher = slower &amp; more guided</span></div>
        <div class="val" id="cfgVal">0.50</div>
      </div>
      <input type="range" id="cfg" min="0" max="1" step="0.05" value="0.5" style="margin-bottom:14px">
      <div class="row">
        <div class="name">Temperature<span class="hint">lower = steadier · higher = more varied</span></div>
        <div class="val" id="tempVal">0.80</div>
      </div>
      <input type="range" id="temperature" min="0.05" max="1.5" step="0.05" value="0.8">
    </div>

    <div class="card">
      <div class="row">
        <div class="name">Seed<span class="hint">0 = random · any other number reproduces the render exactly</span></div>
      </div>
      <input type="number" id="seed" min="0" max="99999999" step="1" value="0" inputmode="numeric">
    </div>

    <details class="card adv" id="advCard">
      <summary><span class="chev">›</span> Advanced <span style="margin-left:auto; font-size:11px; font-weight:400;">defaults match the official V3 space</span></summary>
      <div class="body">
        <div class="row">
          <div class="name">Repetition penalty<span class="hint">raise if it stutters/loops</span></div>
          <div class="val" id="repVal">1.20</div>
        </div>
        <input type="range" id="rep" min="1" max="5" step="0.1" value="1.2" style="margin-bottom:14px">
        <div class="row">
          <div class="name">min_p</div>
          <div class="val" id="minpVal">0.05</div>
        </div>
        <input type="range" id="minp" min="0" max="0.25" step="0.01" value="0.05" style="margin-bottom:14px">
        <div class="row">
          <div class="name">top_p</div>
          <div class="val" id="toppVal">1.00</div>
        </div>
        <input type="range" id="topp" min="0.5" max="1" step="0.01" value="1" style="margin-bottom:14px">
        <div class="row">
          <div class="name">Gap between sentences</div>
          <div class="val" id="gapVal">120 ms</div>
        </div>
        <input type="range" id="gap" min="0" max="500" step="10" value="120" style="margin-bottom:14px">
        <div class="row">
          <div class="name">Chunk size<span class="hint">smaller = better GPU balance · larger = fewer boundary seams</span></div>
          <div class="val" id="chunkVal">260 chars</div>
        </div>
        <input type="range" id="chunk" min="140" max="400" step="20" value="260">
      </div>
    </details>

    <div class="card">
      <div class="label">Format</div>
      <div class="segmented" id="fmt" role="radiogroup">
        <button data-v="wav"  class="on">WAV</button>
        <button data-v="mp3">MP3</button>
        <button data-v="opus">Opus</button>
        <button data-v="flac">FLAC</button>
      </div>
    </div>

    <div class="card result" id="result">
      <div class="label"><span>Latest render</span><span id="resStamp"></span></div>
      <audio id="player" controls preload="metadata" playsinline></audio>
      <div class="stats">
        <span><b id="resDur">0.00s</b> audio</span>
        <span>rendered in <b id="resTime">0.00s</b></span>
        <span>RTF <b id="resRTF">0.00×</b></span>
        <span id="resChunks"></span>
      </div>
      <div class="dlrow">
        <a class="btn primary" id="downloadBtn" download="chatterbox.wav">↓ Download</a>
        <button class="btn" id="shareBtn">Share</button>
      </div>
    </div>

    <div class="card hide" id="histCard">
      <div class="label">History</div>
      <div id="histList"></div>
    </div>
  </main>

  <div class="fab-wrap">
    <div class="inner">
      <button class="icon-btn" id="stopBtn" title="Stop" aria-label="Stop" style="display:none;">■</button>
      <button class="fab" id="goBtn"><span id="goLabel">Generate</span></button>
    </div>
  </div>

  <div class="sheet-back" id="sheetBack"></div>

  <div class="sheet" id="langSheet" role="dialog" aria-label="Choose a language">
    <div class="grabber"></div>
    <div class="head"><b>Language</b><button class="sheetClose">Done</button></div>
    <div class="search"><input id="langSearch" placeholder="Search languages…" enterkeyhint="search"></div>
    <div class="list" id="langList"></div>
  </div>

  <div class="sheet" id="voiceSheet" role="dialog" aria-label="Choose a voice">
    <div class="grabber"></div>
    <div class="head"><b>Voice</b><button class="sheetClose">Done</button></div>
    <div class="list" id="voiceList"></div>
    <button class="upload" id="uploadBtn"><span id="uploadLabel">＋ Clone a voice from an audio clip…</span></button>
    <div class="note">5–15 s of clear speech works best. The clip is transcoded to 24 kHz WAV and used only inside this Kaggle session.</div>
    <input type="file" id="fileInput" accept="audio/*,.wav,.mp3,.m4a,.ogg,.oga,.webm,.flac,.aac" style="display:none">
  </div>

  <div class="toast" id="toast">…</div>

<script>
const $ = s => document.querySelector(s);
const state = { languages: [], lang: 'en', voices: [], voice: 'default',
                fmt: 'wav', busy: false, job: null, ctrl: null, history: [],
                model: {ready:false, status:'idle'}, epubJobs: [] };

function toast(msg, kind='bad'){
  const t = $('#toast'); t.textContent = msg;
  t.style.background = kind==='good' ? 'var(--good)' : 'var(--bad)';
  t.classList.add('on'); clearTimeout(toast._t);
  toast._t = setTimeout(()=>t.classList.remove('on'), 3400);
}
function esc(s){ return (s||'').replace(/[&<>"']/g, c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c])); }
function initials(v){ const n=(v.name||'').replace(/[^a-z0-9]/gi,''); return (n[0]||'C').toUpperCase(); }

/* ---------- language ---------- */
function setLang(code){
  const l = state.languages.find(x=>x.id===code) || state.languages[0]; if(!l) return;
  state.lang = l.id;
  $('#lName').textContent = `${l.flag} ${l.name}`;
  $('#lMeta').textContent = l.native === l.name ? l.id : l.native;
  try{ localStorage.setItem('cb.lang', l.id); }catch{}
  renderLangSheet(($('#langSearch').value||''));
}
function renderLangSheet(q){
  q = (q||'').trim().toLowerCase();
  const list = $('#langList'); list.innerHTML='';
  let any = false;
  for(const l of state.languages){
    if(q && !(l.id.includes(q) || l.name.toLowerCase().includes(q) || (l.native||'').toLowerCase().includes(q))) continue;
    any = true;
    const b = document.createElement('button'); b.className='item'+(l.id===state.lang?' on':'');
    b.innerHTML = `<span class="avatar">${l.flag}</span>
      <span class="meta"><b>${esc(l.name)}</b><span>${esc(l.native)} · ${l.id}</span></span>
      <span class="check">✓</span>`;
    b.addEventListener('click', ()=>{ setLang(l.id); closeSheets(); });
    list.appendChild(b);
  }
  if(!any) list.innerHTML = '<div style="padding:20px;text-align:center;color:var(--muted)">No matches.</div>';
}

/* ---------- voice ---------- */
function setVoice(id){
  const v = state.voices.find(x=>x.id===id) || state.voices[0]; if(!v) return;
  state.voice = v.id;
  $('#vName').textContent = v.name;
  $('#vMeta').textContent = v.kind === 'builtin' ? 'built-in conds.pt'
                          : `cloned · ${v.seconds ? v.seconds.toFixed(1)+'s ref' : 'your clip'}`;
  $('#vAvatar').textContent = v.kind === 'builtin' ? 'C' : initials(v);
  try{ localStorage.setItem('cb.voice', v.id); }catch{}
  renderVoiceSheet();
}
function renderVoiceSheet(){
  const list = $('#voiceList'); list.innerHTML='';
  const mk = (v)=>{
    const b = document.createElement('button'); b.className='item'+(v.id===state.voice?' on':'');
    b.innerHTML = `<span class="avatar">${v.kind==='builtin'?'C':initials(v)}</span>
      <span class="meta"><b>${esc(v.name)}</b><span>${v.kind==='builtin'?'built-in Chatterbox voice':'cloned voice'+(v.seconds?` · ${v.seconds.toFixed(1)}s`:'')}</span></span>
      ${v.kind==='builtin'
        ? '<span class="check">✓</span>'
        : '<span class="check" style="margin-right:2px">✓</span><span class="del" data-del="1" aria-label="Delete">🗑</span>'}`;
    b.addEventListener('click', e=>{
      if(e.target.dataset.del){ delVoice(v.id); return; }
      setVoice(v.id); closeSheets();
    });
    list.appendChild(b);
  };
  const def = state.voices.find(v=>v.kind==='builtin');
  if(def){ const h=document.createElement('div'); h.className='group'; h.textContent='Built-in'; list.appendChild(h); mk(def); }
  const clones = state.voices.filter(v=>v.kind!=='builtin');
  if(clones.length){ const h=document.createElement('div'); h.className='group'; h.textContent='Your cloned voices'; list.appendChild(h); clones.forEach(mk); }
}
async function delVoice(id){
  try{
    const r = await fetch('/api/voices/'+id, {method:'DELETE'});
    if(!r.ok) throw new Error();
    state.voices = state.voices.filter(v=>v.id!==id);
    if(state.voice===id) setVoice('default'); else renderVoiceSheet();
    toast('Voice deleted','good');
  }catch{ toast('Delete failed'); }
}
async function uploadVoice(file){
  if(!file) return;
  if(file.size > 30*1024*1024){ toast('Clip too large (30 MB max)'); return; }
  const lbl = $('#uploadLabel');
  lbl.innerHTML = '<span class="spinner"></span> Transcoding &amp; uploading…';
  $('#uploadBtn').style.pointerEvents = 'none';
  try{
    const fd = new FormData(); fd.append('file', file, file.name||'clip.webm');
    const r = await fetch('/api/voices', {method:'POST', body:fd});
    const j = await r.json();
    if(!r.ok) throw new Error(j.detail||'upload failed');
    state.voices.push(j.voice);
    setVoice(j.voice.id);
    toast(`Voice “${j.voice.name}” ready`, 'good');
  }catch(e){ toast(e.message||'Upload failed'); }
  finally{
    lbl.textContent = '＋ Clone a voice from an audio clip…';
    $('#uploadBtn').style.pointerEvents = '';
    $('#fileInput').value = '';
  }
}

/* ---------- sheets ---------- */
let openSheetEl = null;
function openSheet(el){ closeSheets(); openSheetEl = el; el.classList.add('on'); $('#sheetBack').classList.add('on'); }
function closeSheets(){ for(const s of document.querySelectorAll('.sheet')) s.classList.remove('on');
  $('#sheetBack').classList.remove('on'); openSheetEl = null; }

/* ---------- format ---------- */
function fmt(v){ state.fmt = v; for(const b of $('#fmt').children) b.classList.toggle('on', b.dataset.v===v);
  $('#downloadBtn').setAttribute('download', 'chatterbox.'+v);
  try{ localStorage.setItem('cb.fmt', v); }catch{} }

/* ---------- meters ---------- */
function updateMeters(){
  const c = $('#script').value.length;
  $('#charCount').textContent = `${c.toLocaleString()} chars`;
  const s = c / 15;   // chatterbox ≈ 15 spoken chars/sec in latin scripts
  $('#estAudio').textContent = `≈ ${s>=60 ? (s/60).toFixed(1)+'m' : s.toFixed(1)+'s'} audio`;
}

/* ---------- history ---------- */
function pushHistory(item){
  state.history.unshift(item); state.history = state.history.slice(0, 6);
  const list = $('#histList'); list.innerHTML='';
  for(const h of state.history){
    const el = document.createElement('div'); el.className='hist-item';
    el.innerHTML = `<button class="play" aria-label="Play">▶</button>
      <div class="who"><b>${esc(h.label)}</b><span>${esc(h.preview)}</span></div>
      <a class="btn" style="flex:0 0 auto; min-width:64px" href="${h.url}" download="chatterbox.${h.fmt}">↓</a>`;
    el.querySelector('.play').addEventListener('click', ()=>{ const p=$('#player'); p.src=h.url; p.play(); });
    list.appendChild(el);
  }
  $('#histCard').classList.toggle('hide', !state.history.length);
}

/* ---------- generate ---------- */
function setBusy(b){ state.busy = b;
  $('#goBtn').disabled = b || !state.model.ready;
  $('#goLabel').innerHTML = b ? '<span class="spinner"></span> Rendering…' : 'Generate';
  $('#stopBtn').style.display = b ? 'grid' : 'none';
  $('#pbar').style.width = b ? '5%' : '0%';
}

async function loadStatus(){
  try{
    const r = await fetch('/api/status'); const j = await r.json();
    state.languages = j.languages; state.voices = j.voices;
    $('#statusDot').style.background = j.model_ready ? 'var(--good)' : 'var(--warn)';
    $('#statusText').textContent = j.model_ready ? (j.n_gpus ? `${j.n_gpus} GPU${j.n_gpus===1?'':'s'}` : 'CPU')
                                                 : 'no model';
    const gpu = (j.gpus&&j.gpus.length)? j.gpus.map(g=>g.name.replace('Tesla ','')).join(' + ') : 'CPU';
    $('#sub').textContent = `${j.model} · ${gpu}${j.fp16 ? ' · fp16' : ''}`;
    const saved = localStorage.getItem('cb.lang');
    setLang(saved && j.languages.some(l=>l.id===saved) ? saved : 'en');
    const savedV = localStorage.getItem('cb.voice');
    setVoice(savedV && j.voices.some(v=>v.id===savedV) ? savedV : 'default');
    const savedFmt = localStorage.getItem('cb.fmt'); if(savedFmt) fmt(savedFmt);
    renderLangSheet(''); renderVoiceSheet();
    if(j.model_state) renderModel(j.model_state);
  }catch(e){ $('#sub').textContent = 'backend not reachable';
    $('#statusDot').style.background = 'var(--bad)'; $('#statusDot').style.boxShadow = 'none';
    $('#statusText').textContent = 'offline';
    toast('Backend not reachable'); }
}

/* ---------- model loading (from this page) ---------- */
let modelPoll = null;
function renderModel(m){
  state.model = m;
  const badge = $('#modelBadge'), msg = $('#modelMsg'), log = $('#modelLog');
  const busy = !!m.busy;
  badge.textContent = busy ? m.status : (m.ready ? 'ready' : (m.status==='error' ? 'error' : 'not loaded'));
  badge.className = 'badge ' + (busy ? 'work' : m.ready ? 'ok' : (m.status==='error' ? 'err' : ''));
  const workers = (m.workers||[]).length;
  msg.textContent = m.ready
    ? `${m.t3 ? m.t3.toUpperCase()+' · ' : ''}${workers} worker${workers===1?'':'s'} · `
      + (m.workers||[]).map(w=>`${w.device}${w.fp16?' fp16':''}`).join(', ')
      + (m.attn && m.attn.length ? ` · ${m.attn.join('/')}` : '')
      + (m.elapsed_s ? ` · loaded in ${m.elapsed_s}s` : '')
    : (m.message || 'no model loaded');
  // VRAM headroom: tell the user concretely when more replicas would fit.
  const tip = $('#modelTip'), sug = m.suggested_replicas_per_gpu, cur = +$('#modelReps').value;
  const vram = Object.entries(m.vram||{});
  if(m.ready && vram.length){
    const usage = vram.map(([d,v])=>`${d} ${v.peak_gb}/${v.total_gb} GB`).join(' · ');
    if(sug && sug > cur){
      tip.className='hint tip'; tip.classList.remove('hide');
      tip.innerHTML = `${esc(usage)} — headroom for <b>${sug} replicas/GPU</b>. `
        + `<button class="linkbtn" id="modelApplyTip">Use ${sug} and reload</button>`;
      $('#modelApplyTip').addEventListener('click', ()=>{
        $('#modelReps').value=String(sug); $('#modelReps').dataset.touched='1'; loadModel(true);
      });
    } else {
      tip.className='hint'; tip.classList.remove('hide');
      tip.textContent = `${usage} — VRAM well used at ${cur} replica${cur===1?'':'s'}/GPU.`;
    }
  } else tip.classList.add('hide');
  if(m.log && m.log.length){ log.classList.remove('hide'); log.textContent = m.log.slice(-14).join('\n'); log.scrollTop = log.scrollHeight; }
  else log.classList.add('hide');
  $('#modelLoad').disabled = busy;
  $('#modelUnload').disabled = busy || !m.ready;
  $('#modelLoad').textContent = busy ? 'Loading…' : (m.ready ? 'Reload model' : 'Load model');
  if(m.default_replicas_per_gpu && !$('#modelReps').dataset.touched)
    $('#modelReps').value = String(Math.min(4, m.default_replicas_per_gpu));
  $('#goBtn').disabled = state.busy || !m.ready;
  if(busy && !modelPoll) modelPoll = setInterval(pollModel, 1500);
  if(!busy && modelPoll){ clearInterval(modelPoll); modelPoll = null; loadStatusSoft(); }
}
async function pollModel(){
  try{ renderModel(await (await fetch('/api/model')).json()); }catch{}
}
async function loadStatusSoft(){
  try{ const j = await (await fetch('/api/status')).json();
    const gpu = (j.gpus&&j.gpus.length)? j.gpus.map(g=>g.name.replace('Tesla ','')).join(' + ') : 'CPU';
    $('#sub').textContent = `${j.model} · ${gpu}${j.fp16 ? ' · fp16' : ''}`;
    $('#statusDot').style.background = j.model_ready ? 'var(--good)' : 'var(--warn)';
    $('#statusText').textContent = j.model_ready ? (j.n_gpus ? `${j.n_gpus} GPU${j.n_gpus===1?'':'s'}` : 'CPU') : 'no model';
  }catch{}
}
async function loadModel(reload){
  const body = { t3: $('#modelT3').value, replicas_per_gpu: +$('#modelReps').value, reload: !!reload };
  $('#modelLoad').disabled = true;
  try{
    const r = await fetch('/api/model/load', {method:'POST', headers:{'content-type':'application/json'}, body: JSON.stringify(body)});
    const j = await r.json();
    if(!r.ok) throw new Error(j.detail||'could not start the load');
    renderModel(j); toast('Loading model — first run downloads ~2 GB','good');
  }catch(e){ toast(e.message||'Load failed'); $('#modelLoad').disabled=false; }
}
async function unloadModel(){
  $('#modelUnload').disabled = true;
  try{
    const r = await fetch('/api/model/unload', {method:'POST'});
    const j = await r.json();
    if(!r.ok) throw new Error(j.detail||'could not unload');
    renderModel(j); toast('Model unloaded','good'); loadStatusSoft();
  }catch(e){ toast(e.message||'Unload failed'); pollModel(); }
}

/* ---------- job tracking: SSE first, polling as the safety net ----------
   Phones drop connections. EventSource reconnects on its own and the server
   replays missed events via Last-Event-ID, but if the stream is wedged (dead
   tunnel, proxy buffering, a `done` consumed by a socket that never came back)
   we stop trusting it and poll the job state instead. Either path resolves
   with the job meta once the audio is ready on the server. */
const STALL_MS = 12000;      // no SSE traffic for this long -> start polling
function trackJob(jobId, ctrl, onProgress){
  return new Promise((resolve, reject)=>{
    let settled=false, es=null, pollTimer=null, stallTimer=null;
    const cleanup = ()=>{ try{es&&es.close();}catch{} clearInterval(pollTimer); clearTimeout(stallTimer); };
    const finish = (fn,v)=>{ if(settled) return; settled=true; cleanup(); fn(v); };
    const bump = ()=>{ clearTimeout(stallTimer); stallTimer=setTimeout(startPolling, STALL_MS); };

    ctrl.signal.addEventListener('abort', ()=> finish(reject, new Error('aborted')));

    // --- polling fallback ---------------------------------------------------
    async function pollOnce(){
      if(settled) return;
      try{
        const r = await fetch('/api/jobs/'+jobId, {cache:'no-store'});
        if(r.status===404) return finish(reject, new Error('job expired on the server'));
        if(!r.ok) return;
        const j = await r.json();
        if(j.total) onProgress({done:j.done, total:j.total});
        if(j.state==='done' && j.audio_ready) return finish(resolve, j.meta||{});
        if(j.state==='error')     return finish(reject, new Error(j.error||'render failed'));
        if(j.state==='cancelled') return finish(reject, new Error('aborted'));
      }catch{ /* offline: keep trying, the render continues on Kaggle */ }
    }
    function startPolling(){
      if(settled || pollTimer) return;
      try{ es && es.close(); }catch{}
      es = null;
      pollTimer = setInterval(pollOnce, 2500); pollOnce();
    }

    // --- primary path: SSE --------------------------------------------------
    try{
      es = new EventSource('/api/jobs/'+jobId+'/events');
      es.addEventListener('progress', e=>{ bump();
        // Last-Event-ID replay is EventSource's job; the cursor is never read here.
        try{ onProgress(JSON.parse(e.data)); }catch{}
      });
      es.addEventListener('done', e=>{ let m={}; try{ m=JSON.parse(e.data); }catch{} finish(resolve, m); });
      es.addEventListener('error', e=>{
        // Distinguish an application error (has a payload) from a transport
        // drop (no payload) — the latter is exactly what EventSource retries.
        let msg=null; try{ msg = JSON.parse(e.data).message; }catch{}
        if(msg) return finish(reject, new Error(msg));
        bump();                       // transport hiccup: let it reconnect, watchdog covers us
      });
      es.addEventListener('cancelled', ()=> finish(reject, new Error('aborted')));
      es.addEventListener('eof', ()=>{ startPolling(); });   // server closed: confirm via REST
      es.addEventListener('open', bump);
      bump();
    }catch{ startPolling(); }
  });
}

async function generate(){
  const text = $('#script').value.trim();
  const picked = selectedChapters();
  if(!text){
    // The most common case: a book is loaded and chapters are ticked, but the
    // script box was never filled. Generate should just do the obvious thing.
    if(epubBook && picked.length) return convertEpub();
    toast(epubBook ? 'Tick a chapter, or type a script'
                   : 'Script is empty — type something or load an EPUB');
    return;
  }
  if(!state.model.ready){ toast('Load the model first'); return; }
  if(state.busy) return;
  setBusy(true);
  const payload = {
    text,
    language_id: state.lang,
    voice: state.voice,
    exaggeration: parseFloat($('#exaggeration').value),
    cfg_weight: parseFloat($('#cfg').value),
    temperature: parseFloat($('#temperature').value),
    seed: parseInt($('#seed').value||'0',10) || 0,
    repetition_penalty: parseFloat($('#rep').value),
    min_p: parseFloat($('#minp').value),
    top_p: parseFloat($('#topp').value),
    gap_ms: parseInt($('#gap').value,10),
    chunk_chars: parseInt($('#chunk').value,10),
    format: state.fmt,
  };

  const ctrl = new AbortController(); state.ctrl = ctrl;
  const t0 = performance.now();
  try{
    const jr = await fetch('/api/jobs', {method:'POST', headers:{'content-type':'application/json'},
                                          body: JSON.stringify(payload), signal: ctrl.signal});
    if(!jr.ok){ const j = await jr.json().catch(()=>({})); throw new Error(j.detail||'server rejected'); }
    const {job_id} = await jr.json(); state.job = job_id;

    const meta = await trackJob(job_id, ctrl, (d)=>{
      $('#pbar').style.width = (5 + 88*(d.done/Math.max(1,d.total))) + '%';
      $('#goLabel').innerHTML = d.total
        ? `<span class="spinner"></span> ${d.done}/${d.total} chunks`
        : '<span class="spinner"></span> Rendering…';
    });

    $('#pbar').style.width = '96%';
    const ar = await fetch('/api/jobs/'+job_id+'/audio', {signal: ctrl.signal});
    if(!ar.ok) throw new Error('audio fetch failed');
    const blob = await ar.blob();
    const url  = URL.createObjectURL(blob);
    const m = (meta && Object.keys(meta).length) ? meta : JSON.parse(ar.headers.get('X-Meta') || '{}');

    const dur = m.audio_s || 0, took = (performance.now()-t0)/1000;
    $('#player').src = url;
    $('#resDur').textContent  = dur.toFixed(2)+'s';
    $('#resTime').textContent = (m.wall_s || took).toFixed(2)+'s';
    $('#resRTF').textContent  = (dur>0 ? ((m.wall_s||took)/dur).toFixed(2) : '-')+'×';
    $('#resChunks').textContent = m.chunks ? `${m.chunks} chunk${m.chunks>1?'s':''} · ${m.gpus||''} GPU(s)` : '';
    $('#resStamp').textContent = new Date().toLocaleTimeString();
    const a = $('#downloadBtn'); a.href = url; a.setAttribute('download','chatterbox.'+state.fmt);
    $('#result').classList.add('on');
    const lname = (state.languages.find(l=>l.id===state.lang)||{}).name || state.lang;
    const vname = (state.voices.find(v=>v.id===state.voice)||{}).name || 'voice';
    pushHistory({ url, fmt: state.fmt, label: `${lname} · ${vname}`, preview: text.slice(0,80) });
    $('#pbar').style.width = '100%';
    try{ await $('#player').play(); }catch{}
  }catch(e){
    if(e.name!=='AbortError' && e.message!=='aborted') toast(e.message||'Generation failed');
  }finally{
    setTimeout(()=>{ if(!state.busy) $('#pbar').style.width='0%'; }, 500);
    state.job=null; state.ctrl=null; setBusy(false);
  }
}

async function stopJob(){
  if(state.ctrl) state.ctrl.abort();
  if(state.job){ try{ await fetch('/api/jobs/'+state.job, {method:'DELETE'}); }catch{} }
}

async function share(){
  const p = $('#player'); if(!p.src) return;
  try{
    const blob = await (await fetch(p.src)).blob();
    const file = new File([blob], 'chatterbox.'+state.fmt, {type: blob.type});
    if(navigator.canShare && navigator.canShare({files:[file]})){
      await navigator.share({ files:[file], title:'Chatterbox render' });
    }else{
      $('#downloadBtn').click();
    }
  }catch(e){ /* user cancelled */ }
}

/* ---------- EPUB ---------- */
let epubBook = null;
function chapterPayload(){ return {
  language_id: state.lang, voice: state.voice, format: state.fmt,
  exaggeration:+$('#exaggeration').value, cfg_weight:+$('#cfg').value,
  temperature:+$('#temperature').value, seed:parseInt($('#seed').value||'0',10)||0,
  repetition_penalty:+$('#rep').value, min_p:+$('#minp').value, top_p:+$('#topp').value,
  gap_ms:+$('#gap').value, chunk_chars:+$('#chunk').value
}; }
function selectedChapters(){
  return [...document.querySelectorAll('.epubCheck:checked')].map(x=>x.value);
}
function renderChapters(){
  $('#epubChapters').innerHTML = epubBook.chapters.map((c,i)=>
    `<label class="chapRow" style="display:flex;gap:8px;align-items:flex-start;padding:7px 0;border-bottom:1px solid var(--line);font-size:13px">
       <input type="checkbox" class="epubCheck" value="${esc(c.id)}" ${i===0?'checked':''} style="margin-top:4px">
       <span style="flex:1;min-width:0"><b>${esc(c.title)}</b><br><small>${c.chars.toLocaleString()} chars</small></span>
       <button type="button" class="btn chapLoad" data-id="${esc(c.id)}" style="flex:0 0 auto;min-height:32px;min-width:74px;font-size:12px">Preview</button>
     </label>`).join('');
  for(const b of document.querySelectorAll('.chapLoad'))
    b.addEventListener('click', e=>{ e.preventDefault(); loadChapterText(b.dataset.id); });
  for(const c of document.querySelectorAll('.epubCheck'))
    c.addEventListener('change', updateEpubButtons);
  updateEpubButtons();
}
function updateEpubButtons(){
  const n = selectedChapters().length;
  $('#epubGo').textContent = n ? `Convert ${n} selected chapter${n===1?'':'s'}` : 'Select chapters to convert';
  $('#epubGo').disabled = !n;
}
async function loadChapterText(id){
  try{
    const r = await fetch(`/api/epub/${epubBook.epub_id}/chapters/${encodeURIComponent(id)}`);
    const j = await r.json();
    if(!r.ok) throw new Error(j.detail||'could not read the chapter');
    $('#script').value = j.text; updateMeters();
    toast(`Loaded “${j.title}” into the script box`, 'good');
    try{ $('#script').scrollIntoView({behavior:'smooth', block:'center'}); }catch{}
  }catch(e){ toast(e.message||'Could not load chapter'); }
}
async function uploadEpub(file){
  if(!file) return; $('#epubInfo').textContent='Reading EPUB…';
  const fd=new FormData(); fd.append('file',file);
  try{
    const r=await fetch('/api/epub',{method:'POST',body:fd}); const j=await r.json();
    if(!r.ok) throw new Error(j.detail||'EPUB upload failed'); epubBook=j;
    if(!j.chapters.length){ $('#epubInfo').textContent='No readable chapters found in that EPUB.'; return; }
    $('#epubInfo').textContent=`${j.chapters.length} readable chapters — tick what to convert, or Preview one into the script box`;
    renderChapters();
    $('#epubTools').classList.remove('hide');
    $('#epubGo').style.display='inline-flex';
  }catch(e){ $('#epubInfo').textContent=e.message; toast(e.message); }
  finally{ $('#epubFile').value=''; }
}
async function convertEpub(){
  if(!epubBook) return;
  if(!state.model.ready){ toast('Load the model first'); return; }
  const chapter_ids = selectedChapters();
  if(!chapter_ids.length) return toast('Select at least one chapter');
  $('#epubGo').disabled=true; $('#epubGo').textContent='Queueing chapters…';
  try {
    const r=await fetch('/api/epub/'+epubBook.epub_id+'/jobs',{method:'POST',
      headers:{'content-type':'application/json'},
      body:JSON.stringify({...chapterPayload(),chapter_ids})});
    const j=await r.json();
    if(!r.ok) throw new Error(j.detail||'Could not queue chapters');
    const failed = j.failed||[];
    state.epubJobs = j.jobs.map(x=>({...x, state:'queued'}))
      .concat(failed.map(f=>({title:f.title, job_id:null, state:'error', error:f.error})));
    const note = failed.length ? `, ${failed.length} failed to queue (${failed[0].title})` : '';
    $('#epubInfo').textContent=`Queued ${j.jobs.length} chapter(s)${note}. They keep rendering on Kaggle even if this page disconnects; files go to ${j.saved_to}${j.drive_enabled?' and Google Drive':''}.`;
    toast(failed.length ? `${j.jobs.length} queued · ${failed.length} failed` : `Queued ${j.jobs.length} chapter(s)`, 'good');
    trackEpubJobs();
  }
  catch(e){toast(e.message);}
  finally{ updateEpubButtons(); }
}
let epubPoll=null;
function renderEpubQueue(){
  const box=$('#epubQueue');
  if(!state.epubJobs.length){ box.innerHTML=''; return; }
  box.innerHTML = state.epubJobs.map(j=>{
    const pct = j.total ? `${j.done}/${j.total}` : '';
    const label = j.state==='done' ? '✓ saved' : j.state==='error' ? '✕ failed'
                : j.state==='cancelled' ? 'cancelled' : (j.state==='running' ? `rendering ${pct}` : 'queued');
    const dl = j.state==='done'
      ? `<a class="btn" style="flex:0 0 auto;min-height:30px;min-width:44px;font-size:12px" href="/api/jobs/${j.job_id}/audio">↓</a>` : '';
    return `<div class="qrow ${esc(j.state)}"><span class="qname">${esc(j.title)}</span><span class="qstate">${esc(label)}</span>${dl}</div>`;
  }).join('');
}
async function trackEpubJobs(){
  renderEpubQueue();
  if(epubPoll) clearInterval(epubPoll);
  const tick = async ()=>{
    try{
      const all = await (await fetch('/api/jobs')).json();
      const by = Object.fromEntries(all.jobs.map(j=>[j.job_id, j]));
      state.epubJobs = state.epubJobs.map(j=> by[j.job_id] ? {...j, ...by[j.job_id]} : j);
      renderEpubQueue();
      if(state.epubJobs.every(j=>['done','error','cancelled'].includes(j.state))){
        clearInterval(epubPoll); epubPoll=null;
      }
    }catch{}
  };
  epubPoll = setInterval(tick, 2000); tick();
}

/* ---------- wire up ---------- */
$('#langBtn').addEventListener('click', ()=>{ renderLangSheet(''); openSheet($('#langSheet'));
  setTimeout(()=>$('#langSearch').focus({preventScroll:true}), 200); });
$('#voiceBtn').addEventListener('click', ()=>{ renderVoiceSheet(); openSheet($('#voiceSheet')); });
$('#sheetBack').addEventListener('click', closeSheets);
for(const b of document.querySelectorAll('.sheetClose')) b.addEventListener('click', closeSheets);
$('#langSearch').addEventListener('input', e=>renderLangSheet(e.target.value));
$('#uploadBtn').addEventListener('click', ()=>$('#fileInput').click());
$('#epubPick').addEventListener('click', ()=>$('#epubFile').click());
$('#epubFile').addEventListener('change', e=>uploadEpub(e.target.files[0]));
$('#epubGo').addEventListener('click', convertEpub);
$('#epubAll').addEventListener('click', ()=>{ for(const c of document.querySelectorAll('.epubCheck')) c.checked=true; updateEpubButtons(); });
$('#epubNone').addEventListener('click', ()=>{ for(const c of document.querySelectorAll('.epubCheck')) c.checked=false; updateEpubButtons(); });
$('#modelLoad').addEventListener('click', ()=>loadModel(state.model.ready));
$('#modelUnload').addEventListener('click', unloadModel);
$('#modelReps').addEventListener('change', e=>{ e.target.dataset.touched='1'; });
$('#fileInput').addEventListener('change', e=>uploadVoice(e.target.files[0]));
$('#fmt').addEventListener('click', e=>{ if(e.target.dataset.v) fmt(e.target.dataset.v); });
$('#exaggeration').addEventListener('input', e=>{ $('#exVal').textContent = parseFloat(e.target.value).toFixed(2); });
$('#cfg').addEventListener('input', e=>{ $('#cfgVal').textContent = parseFloat(e.target.value).toFixed(2); });
$('#temperature').addEventListener('input', e=>{ $('#tempVal').textContent = parseFloat(e.target.value).toFixed(2); });
$('#rep').addEventListener('input', e=>{ $('#repVal').textContent = parseFloat(e.target.value).toFixed(2); });
$('#minp').addEventListener('input', e=>{ $('#minpVal').textContent = parseFloat(e.target.value).toFixed(2); });
$('#topp').addEventListener('input', e=>{ $('#toppVal').textContent = parseFloat(e.target.value).toFixed(2); });
$('#gap').addEventListener('input', e=>{ $('#gapVal').textContent = e.target.value+' ms'; });
$('#chunk').addEventListener('input', e=>{ $('#chunkVal').textContent = e.target.value+' chars'; });
$('#script').addEventListener('input', updateMeters);
$('#goBtn').addEventListener('click', generate);
$('#stopBtn').addEventListener('click', stopJob);
$('#shareBtn').addEventListener('click', share);
// swipe-down on any sheet grabber/head
for(const s of document.querySelectorAll('.sheet')){
  let sy=0, dy=0, drag=false;
  s.addEventListener('touchstart', e=>{ if(e.target.classList.contains('grabber')||e.target.closest('.head')){ drag=true; sy=e.touches[0].clientY; } },{passive:true});
  s.addEventListener('touchmove',  e=>{ if(!drag) return; dy=e.touches[0].clientY-sy; if(dy>0){ s.style.transform=`translateY(${dy}px)`; } },{passive:true});
  s.addEventListener('touchend',   ()=>{ if(!drag) return; drag=false; s.style.transform=''; if(dy>90) closeSheets(); dy=0; });
}

updateMeters(); loadStatus();
if('serviceWorker' in navigator){ navigator.serviceWorker.register('/sw.js').catch(()=>{}); }
</script>
</body></html>"""

MANIFEST_JSON = '''{
  "name": "Chatterbox TTS · GPU",
  "short_name": "Chatterbox",
  "start_url": "/",
  "display": "standalone",
  "background_color": "#0d0b14",
  "theme_color": "#0d0b14",
  "orientation": "portrait",
  "icons": [
    { "src": "/icon-180.png", "sizes": "180x180", "type": "image/png", "purpose": "any" },
    { "src": "/icon-512.png", "sizes": "512x512", "type": "image/png", "purpose": "any maskable" },
    { "src": "/icon.svg", "sizes": "any", "type": "image/svg+xml", "purpose": "any" }
  ]
}'''

ICON_SVG = '''<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 512 512">
  <defs><linearGradient id="g" x1="0" y1="0" x2="1" y2="1">
    <stop offset="0" stop-color="#a78bfa"/><stop offset="1" stop-color="#8b5cf6"/></linearGradient></defs>
  <rect width="512" height="512" rx="112" fill="url(#g)"/>
  <text x="50%" y="58%" text-anchor="middle" font-family="-apple-system,Segoe UI,Roboto,sans-serif"
        font-weight="800" font-size="280" fill="#fff">C</text>
</svg>'''

# PNG fallback for iOS "Add to Home Screen": iOS ignores SVG touch icons and
# falls back to a screenshot otherwise. 180x180 is Apple's canonical size; the
# 512 also feeds the PWA manifest for Android/desktop installs. (SVG looks
# sharp in the browser, but every install path needs the raster.)
ICON_PNG_180 = 'iVBORw0KGgoAAAANSUhEUgAAALQAAAC0CAYAAAA9zQYyAAAVeUlEQVR42u2d248cx3XGv6ruue2F5JJLanmnKIqUKEoyRcmSJdmJAFlCYkR2YiPJS14C5CUI4JfkPwiQhzzlIQiMAIEBw0keYgSJc7Uk2JAMRZYj0qJupCiRypJLaskll8u9zkx3nTxU90zPTPdcey5VcxpYLDl9mdqe35z+zlenqsQ//GWJEN2o5hfi9iVt1GJ/3PnU5rWj+6npheLPpYG0C31vV9ttoxaXauPvopYNGFa7kjc3lQ+GYWaYRwBmUAi0qTCncAMY5rins5kwA4AEOodxZCIz0oem3Wt3vb+fMKNHaLq4zzSg+9nuPXNZM8e8RO22a8RgphGUGam0q/17JhlmhtkWmAHAZZgN18yjCvOANHOyy8Ews5thOMzUADS7GQyzwTBXbTuGeXxgtsCaS4K5khSOkzVHHVx7ZDVzL9CgjwngkGHWQLObwQmggW5GHMzxSSEngAyzYTKjLaAZZk4ATYO5qqEZZobZAphjIzS7GexmmApzQ4QedzcDowRzP90MDLFdfYS5xodmN4MTQJPcjKRzXdbMDLMtMAOAyzBzAmiyZo4tH2WYGWYbYCY061hhN4NhNgxmJAI9LDcDdg6bEsGzULQBBYng+PBc0q+BYnId6v4+D+J+DhrmeKDZzeipXUJogsMPhjzAV4DyAaUCWEUE9JhLVT5UpX9JCUin+jv8ZhAAovFNAOMOdxnm3ttVgVgBXhnwPL3PcYH8JFCYFpjeITCxDchPCkxMA5mcQDYHOBkNani+7wHlElDcIGytAeurhI17wOodwsYqobih30OA4GT0ewihozfReMPcsh6aE8DkLZQRpIByUUffTA7YcZ/A7H6BPQcEZuYEtu0SKEwKiNbj65tuSgGbq4SVJcLtBcLNqwo3rxJW7xC8sgbbzQBShJF7PDRz/Qsuw9xZu0SgE/yyjqS5CWDfAwIHTkjse0Bi5j4BJyYzIapeV3QgU0OJIiUwuV1gcrvAvgcAwEG5CNxaULh+SeHqRYWlBUKxCGSyOvKHuttqmOsl3w//okQMc+t2hbLCK2pZsGOPwNHHJO5/VGLX3lpESdWGciGQyhZ+KeKuSwTcmidc+cDH5fMKd28ShNRPDQiAfPthjgWarbmYmyR1RPbKwO79Aie/4uDoY1LDEoU4RXjbDeEhFFFJU9oC5j9W+PhtHwufKlAgh2yHuSEp5GFTjVGZCChuADtmBR77NQcPPiErkiIKca8auatN1LomYQTP5oFjpyWOnZaY/1jh/Js+bnymWn7ZjNLMaAE0uxmNUdkraVgf+5qD0y84yE1UQRZySBA341tELD3S/z/0sNb2P/zzIkpbgHDq/UFzE8BEoFkzN8Jc3AB27RV49hUHe4/KkQY5NnKHYAdWohD2WHPNznUZ5kYHo7gOHH9S4iu/5SJXCEAWBoCcoP9rYCZ7YW5q240jzGE0+/JvOPjSC05NVLZiMxXmNs/VxUnsZkAI3XGhFPC177g4fkZqDQqLYI7IkdjPxQKYG1yOVN0MGFJoFERmUsALv+fi6KPSrqhc93eLLu/nqLkZcTA3B3pcajNI116EMCsV1FYMELLkbslabZ9GcO4mAo6yZm5enDRmMAsBFDeBZ7/p4IHHdWTuJ8zRnj4RFCRFfeRW54YyqNsOHNsSwLhD3HFMAAEN7tY68MhzEqeedfoqM6iuKi/cihvA+j1CcYNQLukSUyAoNMoCubxAYRrIFQSk09jVHWqIduCOHG4tzI1Aj4vPLHT38Nz9As98w610QvQVZADrK4QblwlfXFZYukFYWyaUtnS1nPIjx0v9hXMDsAtTAttmBXbOCdx3WGJ2v8DENlHbQ9iJNLEkAYzb746TmxFuinRtw/O/7cJxAyBSBjoa8Rc+JVx4x8f1SwrrqwCULtQXDkEKXfaJbGM4JQJKm8DWOmFpgfDZrwDh+JicAnYflDh0UuLwSYmpHaL691KLJ43FMFcj9JgVGpXWgae/4WDnnEhdaoTRXkjg1lXCu6/6uHpRwff0lyiXD4+j6qiWOn84+m/hAK6jI3V4fHET+PxDhSvvK0xuBw4+JPHwM7pHM6w/aZDmAg0+tBUwN8w+OkbzM4ug/HP3QYFTzzmto1mXMAPA2dd8nHvdh1fWxUJutmoPdjSglQCFWhiFA2SDZ2upCFx4R+HSWYUDxyUe/3UHB44HXfV+5O+jOrANtuaabe5YWHNRueEDZ150tNRQ6UmNEObiJvDTf/Rw5bxCbgLIBl3nFRmRxuhsqtXbuQn9HvMfKVy9oHD4EYmnXnYxe0BU5I+NCWDidLrjMtKkvAXsOyZw8CGZanQOYd5aB/7z78pYvEIoTAeJnoq2rT9TDYTvkS3otlw+r3D1Ygmnnnfw5EuurhIk+2EGAHecJoEhAKeedyp1G0ixw6JcBP77+2Usfk7IT1UtuH7DXPP0CcAOI/bZn/i4ekHhyZdcCCliBxraBDOBOlzWzeABrV5JjzY5eEJWHtVpuhlv/MjD9U+rkXnQMFNdm0BAYQpYvkF47QdlOG4wBQLZ4WbEwdzRKlhGz2gk9TjAY09ISJleBV14nQ/f8vHJLxUK0cg8AvMzK1V1R4jsseaSYK5JCvuRNHTqZqAfMAfZfn4KOHxSxvhZPehmCdy7Tfjlf/kV/dpwo7u5FSnez7jZlox0M5rBHNlkWtDUv9RppKEudFZbk8BIPd3A3BGB6RmReq/gu6/62FrTj3OiNmVGik8zmwuN2o7MbS/rZsn0XKRQ8WZ7in51rsbSAuGz91TVmuunZm51z8YV5rpN9k0zjxDM2QIq4wLT7OL+6H98eMXazou+aWaGuSXMyUBbMqWtEDoZnN4lsH1WVF5LIzpvrhLmP1LI5KquAk9p21pa9RPmeKBtmp9Z6OL9XftErcbtiWj96+pFwtpK1QpjmAfrZiRtrrUwRz6YXXPVarSeJYcIgVat3YwxgLmbdnXDQDswg6IR2saF4EnXFc/MpSechdCdNLeuERwXUP2an9mQyNzN0406Po3avra0yc2of5l8nRBObRepJIQhu3dv6uJ86VBi5Rq7Gf2z5pqdK21xM+JeVgrI5gQmt4l0DI4I0OUSxfc2spsxUM3csAqWrQv0iMCyy08Abg5Ih2i9Ld9S6UdmdjN6hpnQwoc2FeZKhCbd5S0EUulQCbe1ZWKYh+xmJLVL2gpzOIFMNi96zmWiCSEAbK3VFTcxzCMBczzQFi0ETxSZ6DsVovV1iluto75tbsYwCo26aZfbEcwGLQQfyuXK5OQpSWilAK9MjWP12M0YamQOX+jDKljDhzl6UNrzbRAFNc/EMI8azNUIbSPMYaTux2xI46iZRxXmZtMY2BSZK6M2/HRZFoif/44TwOHDHJsU2rYQfDhwNC3lIRxUZ1timEcK5niXw0A3I/F4oUdjp0m0EEAmLxrbYeFC8P1yM1K5Jwmba7qb0axtIgp0GtJZVSd2UcE0CETgBLDTyNzHp5m0FeZwMcrNdapE1rS2yW2iuuQww9y5zKD+tUsaUWjUJTRS6l4930s3Mdy+W3SUmDLM6Vpzzc6V6DXKjIibUX9oOFSqtEnYXKXKa2lsM/eJ2klbbEoAR9Waa7Nd0hY3Iw4aKXU39dpKl5XlMQkhAMzsEZiYBnw/JtlkN2MgbkZSu6StMIfOhl8GVpZSCs1BEliYFpiZE/C9mEUtxw1mGh2YmwM94j2AbUNDwO2FFAV0cP0DD0ooP9kNHJtCI7RwMwbcLmkczNQZzNIBbl9X6U2fGxB8+BEHuXy144YTwMG6GUmHSGthDpJAJwMsLxI27qWTGIZLPuycE5g7KlEu1kZphpmG2i5pupvR6lwp9aQwi/MqlcQweo2Tzzg180wzzDR0l0VakwA22ZQCrl1USGsTUr/vkVMSc/dLlLdaux1szQ3GZZHWwhx8cIoIbha4dolQ2qrC2HOQDvT5ky87UPWjB9iaG1q7pAmFRt1G5vBGOxlt3S1cUqno6DBKkwIOPSzx0NNOZUrdUYS5rWR4gMOmupF37bZLNn0SGZQANraNGsyJT971a5yKNBwPIuDZVxzs2idQ2my96OUgYQ7bUtpAsDbcaLgZndYOddIuaYOb0QpmUnqw7LWLCne+oJrFKXt1PAAgPynw4h9kkM3pjpykCWgGBrPQyXB5Sw9weOR5B9kJUbUYLXAzEuuhbYc5Cl9pE/jg5356bkdo4ylg9wGBl/8wA+loqKUcDsxC6inQNteA2f0Cr/xxBi/8vhu/vKxlMDfadoZac61gDtfyy+aBT8/6WF7Uo7bTKlYK9fSB4xK/+UcZ5Ap6AU7ZzqIBKcAcLsWsSIOcLQh89dsuvvOnWRw4IVHcsMuaa7ZLWmHNtTk/s3Q0aGdf9VJLDuuh3v+gxLe+m8XeoxKbq/qRL2VCPXaPMAupf3wP2FzXK1498XUHv/tnGTzxdQduJmYxe7LDzUi6Z67RMHe4dJpSQK4AXDqncPwphYMnZKqL14dQ79gt8M0/yeDc6z7e+5mH9RUgk6/OERJd/bVtmEWgFoIJbpQCyhs619s+K/ClMxInn3MqKxWQCiK36OAjS2F+5mHCDDSZxsAoa67DdgkJvPUvHn7nu1m9jl+K83eEUkY6wJmXHBx/UuL9N3xcOqeweiewEV39E/dFEhFPO3oPlQ94fpB0CmBiGjhwQuLYaQdHTgnkJyPreosEv92wQqNOYdZAW5gAtnqUuxm9gtXb/+bhq992U10mOep+kAKmdwo8+y0Xp18kzH+sMP+Rws15wvoKaW1LkYUxI2WvUbvRyehZVKd3Cuw5JLH3AYm9RwWmd4oaJ6cCcj+suRFMAOOeZu5YwRyeS0B+Ujsec0ckHjwjoVT8fBtpRGu9TLHAiaccnHjKgVcC7t4i3FsirN4hbKwSipsE5WkoHVcgm9d24NSMwNQOYNsugcnttd+68NpCtJZN4fdFWAxzMtAWJICt2kWkk6g3/qmM7bsz2HMoXT1dE62jo8OFft/Z/QKz+zt/LFS0t6heu6fN4ASwbdvOdpjD/0sJeGXgJ9/3cG+JKkldP7YwilbkSJAYktIJXtJPeAxFVsYVMiXNbxnMiLPtRrnQKO3VpsIovbZM+I+/LWNtub9QxwIu9Rcr6SdVgC2HuedVsEbCzUBv7SJfd7gsLxJ+/L0yVoJIrQYA9VC2VmMADXEzks6VA4G5nwlgCpNNqmD55LuLhH/96zIW/09ByuoKseOwDXPYVFowa6AtdjM6aZdSuvNj/R7hx39Txif/6+sEUQxGggwsOlviZiSdK8cO5maT0yjtUSsfeO0HHt78kafHDMraxMw0iJs9aUa9NqMTmONdjj5DM3SYW0QZCvzobB5472c+/vmvSrh6UVUSM1PADl0UCCA3iVh7z/QEMO4lOQxoRhXmGhgIKEwCd24Q/v17Zfz0771KwjjKYIc2nxB6LuviJvCr132Uwuo/shdmIK5jxZBCo0G0SylUaj0+fMvHlQ98PPS0g1PPO9i2K6bbWQwJ4rDHUFY7htZXCBd+ofDhz33c+YKQK9gZmTteBWskrLkhtouCRWPzk0C5BLz7qo8Lv/Bx9HGJh77sYO6IrOldbOjJ65ecCFbLrXjUQmv/G5cVLr6jcPm8wtodPUA4P1ltlw3WXLP9bmrQGOJmdN6uarQWQsuQcgn44E2FC28r7DkscP+jDg49LLFzr2joOo/CV9lEza+GUSSUAEhYDhotC/XKwO0FhSvvK3z+gcLSAsErAdkcUJiKaGnL3Iykp4ZrtZuBdGCulyFC6Fn8SQFfXCYsXPKQnwBmgtmU9h2V2LVfYGqH0GWinURq0Rx+rwzcu01YuqZw7RPCjcsKKzcJpSLgOEAmC2QyevRKtHNoHGDWQI95AtgJzDX7gwMyOSArNDyL84Trn/k4J30UJoDJGYEduwW2z+qfqRmB/CSQmxDIZAE3q+eZlqHfTYDy9MKe5SKwtUHYWgNWlwl3bxJWbhGWFwlry4TiZtB1n9G11YXJaqJKZL+bkbTfZZg7hzlOUhB0ZMxm9b89D1j+grB0jWokh5vRiabjAtIVtXUaQVRVHsH3AK+k5U34BRICkK6OxLmJYPAKRRYDxXhYc832u+xmdA9z/QdTKdQP9G4Ir4i8TqTHAHplgIhqXIroMKuweClXqIy6qr4PVZPV5u0aL5gbgDa90GiYMDez0xr86gBW2eyDo1qXZRgLwZsGcw3Q7GakC3NLaKj9D9b2YVNpwQzUBwmGeTAw9xGacYYZBLicADLMJmvm+v0uwzxgmMdgStuBwlz3dJYMM8NsC8wVDd0/16DLcjReCH6w98RUmRFzmMsJoOGauZ9PMwM0c/NqO4aZ3QyDYY4HeowKjRhmu2BuWj7KMHMCaBrM1QjNMDPMhrkZSftdLjQaIZgtmJ95mDADdbYdFxphuNZcL/fYwkKjloc1Kx9lN4MTQBM1c/0mGWaG2RaYgWhxEieAdsFsuZuRtMtlmNnNMDUBjGuX21mUYZgZ5tGFuepy9MHN6C7dHhM3w5CF4EfZzUhql8sJICeAJmvmpj40w8wwmwwz0HJZN9bMDLM5MCcDzTBzAmggzPFAM8wMswFuRtIBbuMNYJj7H2XsnZ95mDA3JIWjOD8zFxqZNT9zVwz02q4GycFuBieAhmrmRtuOYWaYLYEZAFzWzAbAzG5Gm+2i1qO+OQFkmEfRzUjKTSTDzDDbAnM80Cm5GTAFZvQRZovcjGEUGnXsGo3WKlismTkB7PSeNULnMswMs6kJYNx+l90M1sy2wFyN0JwAMswGJoBx+yXDzDDbAnNDhLYJ5m7a1d0HM96FRoP7klFb15YdJ4CGROYOvZ+uLsGFRn1oF7qHuRKh2c1gN8PEBDDufJfdDIbZFpirGpoTQE4ALYBZz5zEMDPMBroZSfdMMswMsy0wNwfaVM3cL5gtcjNMKTTq5p71aRUsTgA5Aez0nlEq7XIZZobZ1AQwbl/Kq2AxzOxmDA/maoTmBJBhNjABjDtXMswMsy0wt3Y5eCH4vpw77m4G0goA6a+CZZlm7hAaTgCH52Y0LR9lmNnNMFlm1EgOhplhtgXmGsnBCeAYJYCWwkxtr4LFMLObYQDM1QjN8zMzzBbADMT40COlmXuBBi3cjGG1iwuN+gZzo4bmBJATQEMjczVCM8wMsyUwJ0dohpndDANhjgea3QxOAA2FuVFyMMwMs8Ewg6JA87CpTtN4djNGDOaq5GDNzAlgV5/laMGsgWaYGWbDZUaNhmaYGWZbYI53OTgBZJgNhZmALpZ1Y5jZzRhRmOMjNBcaDQ3mUVsI3gSY6z9LOTDN3As07VhGw2iX5QvBmxCZkzU0J4CsmQ2VGdH/uAwzw2w0zHX7/h9pTOeN2TMaWwAAAABJRU5ErkJggg=='
ICON_PNG_512 = 'iVBORw0KGgoAAAANSUhEUgAAAgAAAAIACAYAAAD0eNT6AAA04klEQVR42u3daXAc553f8V8/PYOTBHiBIAnel0SKp255LVmSLdleax2Vd9fr3ewm9qtsVY7KmyS1b/J2U7WVpCpVcZVzbmVjO47tddkrex1LlmRZt3VRB2/xgEBSJMADNzAz3U9ejEEQwgCYo2emu5/vvwrFEjlqNp5/g5/f/Lun2/vuX+Wsyi1b0x/X4X9cYrN12m4t+2sbvAZL98zGqmeyddi0rdP/3sTjy8alZ7ZOm7Z12kSSetaAfbVR/iVJ6pmt07fVkJ5FVwb8wR/8wR/8wR/83cJfttwAAP7gD/7gD/7gD/6pwb+8CQD4gz/4gz/4gz/4pwr/pQMA+IM/+IM/+IM/+KcO/8UDAPiDP/iDP/iDP/inEv+FAwD4gz/4gz/4gz/4pxb/0gEA/MEf/MEf/MEf/FON//wAAP7gD/7gD/7gD/6px39uAAB/8Ad/8Ad/8Ad/J/CfDQDgD/7gD/7gD/7g7wz+xQAA/uAP/uAP/uAP/k7hPzsBAH/wB3/wB3/wB39n8F80AIA/+IM/+IM/+IN/OvFfMACAP/iDP/iDP/iDf3rxLxkAwB/8wR/8wR/8wT/d+M8LAOAP/uAP/uAP/uCffvznBADwB3/wB3/wB3/wdwP/mwEA/MEf/MEf/MEf/N3BX5IM+IM/+IM/+IM/+LuFv9US9wEAf/AHf/AHf/AH//Thr6oCAPiDP/iDP/iDP/gnGv/KAwD4gz/4gz/4gz/4Jx5/2UoCAPiDP/iDP/iDP/inAv/yJwDgD/7gD/7gD/7gnxr8ywsA4A/+4A/+4A/+4J8q/JcOAOAP/uAP/uAP/uCfOvwXDwDgD/7gD/7gD/7gn0r8Fw4A4A/+4A/+4A/+4J9a/EsHAPAHf/AHf/AHf/BPNf7zAwD4gz/4gz/4gz/4px7/uQEA/MEf/MEf/MEf/J3AfzYAgD/4gz/4gz/4g78z+BcDAPiDP/iDP/iDP/g7hf/sBAD8wR/8wR/8wR/8ncG/LgEA/MEf/MEf/MEf/OONf+QBAPzBH/zBH/zBH/zjj3+kAQD8wR/8wR/8wR/8k4F/ZAEA/MEf/MEf/MEf/JODfyQBAPzBH/zBH/zBH/yThX/NAQD8wR/8wR/8wR/8k4d/TQEA/MEf/MEf/MEf/JOJf9UBAPzBH/zBH/zBH/yTi39VAQD8wR/8wR/8wR/8k41/xQEA/MEf/MEf/MEf/JOPv2wFAQD8wR/8wR/8wR/804F/2RMA8Ad/8Ad/8Ad/8E8P/mUFAPAHf/AHf/AHf/BPF/5LBgDwB3/wB3/wB3/wTx/+iwYA8Ad/8Ad/8Ad/8E8n/gsGAPAHf/AHf/AHf/BPL/4lAwD4gz/4gz/4gz/4pxv/eQEA/MEf/MEf/MEf/NOP/5wAAP7gD/7gD/7gD/5u4H8zAIA/+IM/+IM/+IO/O/hLkgF/8Ad/8Ad/8Ad/t/C3quFxwOAP/uAP/uAP/uCfTPxVlwAA/uAP/uAP/uAP/rHGP/oAAP7gD/7gD/7gD/6xxz/aAAD+4A/+4A/+4A/+icA/ugAA/uAP/uAP/uAP/onBP5oAAP7gD/7gD/7gD/6Jwr/2AAD+4A/+4A/+4A/+icO/tgAA/uAP/uAP/uAP/onEv/oAAP7gD/7gD/7gD/6Jxb+6AAD+4A/+4A/+4A/+ica/8gAA/uAP/uAP/uAP/onHX7aSAAD+4A/+4A/+4A/+qcC//AkA+IM/+IM/+IM/+KcGf0nKgD/4gz/43yxPauuQ2pd5xa/lUscyT60dUrbNU0urlG2VWlo9ZdukbIvkZyTPSMb3ZIxkjOT5kvGk0Eo2LH6FoRQGVvmcVMhJhbyUn7bKTUlT49L0pNX0hDQ1bjUxIk2MWE2MFv8c/MEf/KPvWQb8wR/83cK/tUPqWuVp+czXyplfpfblRcSjKuPpE3NGT22dn0gcS1Q+J41dtxq9VvwauSYND1rdGLSaGLHgD/7gX+WLM+AP/uCfTvyNL63o8bRynadVvcVfV/Z66ljuKUmVbZFW9hb3/ZOVm5JuXLG6ftnq6sVQQxetrn9sFeTBH/zBf6kXZ8Af/ME/Hfh3dnnq2eRp7SZPazcbrV7vyfhKdbW0SWs3e1q7eXbUEAbS9ctWV/qtLp8PdLm/OEEAf/AH/6UCAPiDP/gnAv+OLk99Ozxt2GnUu9lTZ3ey3tnXq4wvrd7gafUGT3vuL4aCiRGrC6dDXTgV6sJpq6lx25SegT/4x6lnGfAHf/BPBv5+Vlq31WjDTk99O41W9AB+udXR5WnXnb523elLVhq6aHXhVKiBU6Gu9IcKA/AHf7fwnxsAwB/8wT92+GdbpL7dRlv3Gm3cbZRpAfOay5PW9Hla0+fr4MO+8tPSpTOhzh8Nde6DYO6nDsAf/FOKvyR53/53OQv+4A/+8cE/k5U27zHaeodR3y4jPyOqQRUG0sCpUGeOBOo/GiqfA3/wTyf+xQkA+IM/+McC/56NxTH1tv1G2VYwbkYZX9p8u9Hm242CvNR/ItTZI4H6T4RzPlkA/uCfhp5lIt9/8Ad/8C97X1vb9dtz00bdnNOPVflZads+o237jPI56ey7gY69FmhowFbXdvAH/5j1LBPp/oM/+IN/WfvavcbT3gd87ThklMmCbdwr2yLtvtvX7rt9DQ1YHXst0IdHgpJTAfAH/6T0zPv2X+ai+XvAH/zBf8l93bDDaO8DRht3mXJugkfFuHJT0qk3i1OB4UEL/uCfuJ5VFADAH/zBv7p93bjb6NAjvtb0oX7qykoXz4R6/8VA/cdD8Af/xPQsU/PfA/7gD/4L7mvfTqNDj/rq2Qj8qS2vONnZsMPo2iWrt58r6Nz7oWwI/uAf755lavp7wB/8wb/kvvZu8XT34xn1bAJ+l2rVek+f/ZOsbgxaHXku0IfvBArDxv+bAP7gX1Z2XeoUAPiDP/iXv91lKzzd/bivrfuMKGr0mtWRXwU6+Uaw+N0GwR/8m9CzRQMA+IM/+Je33UyLdOBBX3f8js+Ne6h5NT5s9d2/zNX93wTwB/9KKgP+4A/+tW138x6j+5/wE/eYXapxVfWxAf7gX8f9zYA/+IN/ddXe6em+L/naegfjfsptSMA/gT2zJQIA+IM/+C9dOw8a3fPFjFrbcYoCf/BPHv7zJgDgD/7gv3i1dUqffjKjjbt5108BCfgnF/85AQD8wR/8F68N240e/Iqvds71U+AP/gnH/2YAAH/wB/+Fy/jS4Ud97f8dn9v3UuAP/qnAX5Iy4A/+4L9wdXZ5euRrGW7hS4E/+KcKf6uFPgYI/uAP/lq72dMjf5RR+zLwp8Af/NOFv0oGAPAHf/DX7ruM7v9SRsbHJwr8wT99+M8PAOAP/o7jb4x07xd93X4v8lPgD/7pxX9uAAB/8Hcc/0xWeviPMtq4i4/4UeAP/unGfzYAgD/4O45/a7v0uT/N8theCvzB3wn8iwEA/MHfcfw7uz09/mcZdfeAPwX+4O8G/rMTAPAHf0fx717j6fF/nFFnF/hT4A/+7uBfnwAA/uCfEPy7Vnv6wtcz3NmPSmpOAH/wrxp/STKNXDDwB/9Y4f8N8KfAH/zdxD/aAAD+4J8U/FcV3/l3gD8F/uDvKP7RBQDwB/+E4L9shafPfyOjDs75U+AP/g7jH00AAH/wTwj+re3SY3/GBX8U+IM/+NceAMAf/BOCv5+RPvsnWXWvAX8K/MEf/GsLAOAP/gnB3/Okh/4go7WbwZ8Cf/AH/9oCAPiDf0Lwl6R7vuBryx5u70uBP/iDf20BAPzBP0H47zxktPd+HuxDgT/4g39tAQD8wT9B+K9e7+mB38uIosAf/MF//kYyUX1j4A/+ccK/rUN65GsZ+fhft5qelIaHrEavWo2PWE2MFH+dnpByU1a5SSk3LYWBVRhIYSDJKz5y2fjFX/2Mp5Y2KdvmqaVVammXOpZ5al/uqaNL6uzytHyVp84VnkzCz+KAP/jHCX+p3AAA/uCfIPw9I33mDzNatoKL/qKqsetWgxeshgashi5YXb9sNTVmK//5t1IQSkFh9jcmRpc+CIyRlq301LXa08p1nlat87RqvdGKHk8mbWd4wB/8G4B/eQEA/ME/QfhL0oEHfa3fzkV/tdTUhHThZKiLZ0Jd+tBq7IZdABLbkOM2DKWRq1YjV60GTt4SDHxpTZ9RzyZPvZuN1m7x1Nmd4OAH/uDfIPyXDgDgD/4Jw39Nn6eDD3PRXzU1MWJ19v1Q54+GunzeyoZLQWKb/nMWBtKV/lBX+qX3XwokSd09nvp2GvXtNFq/3ailDfzBH/xLbSQD/uCfFvwzWemh388k/lxxIysMpPNHQ516K9SF02Fp9GOK/0KbGB60Gh4MdPTlQMaX1m8z2rLXaPNeE9/TQuAP/g3Gf+EAAP7gnzD8JeneL2TUtZrz/uXU1Lh0/PVAx14LNTla2cETZ/w/+ZthIF04XQw3L/9E6tnkaedhX9sPGLUvi8mxAv7g3wT8SwcA8Af/BOLft8No99289V+qJket3n0h1PHXg1suxEsn/qVq8COrwY8KevUpqW+n0W33+Nqy1zTvQkLwB/8m4T8/AIA/+CcQfz8jPfB7nPdfrHJT0pHnAx19NVCQrxaSZOM/52WhNHAy1MDJUO3LPO2+uxgGGjpBAn/wbyL+cwMA+IN/AvGXlQ4/4mvZSkb/JZfHSiffCPXm0wVNjdcCSXrwnzcVGbM68nygI78KtGWP0f4Hfa3bVudpEviDf5Pxnw0A4A/+CcV/Za+nvZ/i3X+pun7F6tc/LGhowNZ0HKQZ/09u4/zR4qcg1mz0dODBjLYdMPK8xv/sgj/4N2INMuAP/knFX570qS9z1f+8pQmld38d6O1fBsW774F/xTX0kdWz38mr+2lPhx/1teOQL8/EYw3AH/yj2tcM+IN/IvGXtH1/8QYw1GxNjFg9/72CPj5X+8+uq/jfus3hweJ6vvVMoEOP+tp1l1/9RAD8wT9G+C8aAMAf/OOMv5+R7vwco/9b68KpUL/6fhXn+sF/yW2OXLV64fsFvfdCoHu+mNHmuDxeGvzBv4aNZMAf/JOGvyTtud/nXv+31NFXAr32s2DxG/mAf837ev2y1S/+Oq/ebUb3/a6vtZubGATAH/xr3IgBf/BPGv6t7dKBh3j3LxXP97/8k4JefQr8GwnJx2dD/fibeT3/fwuaGLVNaDz4g3/tG8mAP/gnCX9JOvAZPzn3d69jhYH0/PcKOvdBGMnagn/lsJ16M9C59wId/mxG+x/0G3NDIfAH/4g2YsAf/JOEf1undNs9vPsP8tIz/xv84wBJPie9/vcF/fA/5nTpbFjfxoM/+Ee4rwb8wT8p+EvS3gd8ZbK883/m2wUNnAT/OEFyY9DqqW/l9cIPCpqeBH/wjzf+kmTAH/yTgn9Lm7TnPrff/Yeh9Nz/KejCKfCPJSShdOL1QN//q5zOHAnrsr/gD/7RbMrKNHsnwB/8y93XPff5yra6/e7/pR8VdP4o+Mcdkskxq19+O69ffjuvqfHo/hECf/CPCn9piRsBgT/4xwV/PyvtecDtd/9HfhXo1FvgnyRIzhwJdenDvD71DzIR9Az8wT86/CVVOAEAf/BvAv5S8a5/bR3u4n/ug1BvPh1Esrbg31hIZqYB4A/+ccJftpIAAP7g3yT8Jbev/B8esvr1DwrVLzr4JwsS8Af/BuBf/gQA/MG/ifiv2ehpTZ+bd/0L8tKz3ykonwN/8Ad/8I8O//ICAPiDfxPxl6Q997r77v/VnxZ0/TIP9gF/8Af/aPFfOgCAP/g3Gf+WdmnrPjef9ztwMtSJ34Q1ry34gz/4g3+p7RrwB/+44i9J2/YZ+Rk5V7kp6cUfBTWvLfiDP/iD/0LbNeAP/nHFX5K2H3Bz/P+bnxc0MWJrWlvwB3/wB//FtmvAH/zjin9nt6feze5d/Dc0YHXyjbCmtQV/8Ad/8F9quwb8wT+O+Bff/RvJMf+tlV7+u4KsrX5twR/8wR/8y9muAX/wjyP+NwOAY3Xm3VBDA7bqtQV/8Ad/8C93uwb8wT+O+Het8rSy1623/2EgvfVMUPXagj/4gz/4V7JdA/7gHzf8JWnT7e69+z/xRqjRa9X9ywj+4A/+4F/pdk29Ggz+4F/LBjfuduzdfyi990JQ1dqCP/iDP/hXs11Tj0UDf/CvZYPZVql3i1sTgDNHQo3dsBWvLfiDP/iDf7XbNVEvGviDf60b7NtpZFz6+L+t4N0/+IM/+IN/RAeViXLRwB/8o9hg32633v1fOhvq+pXKDh7wB3/wB/9at2uiWjTwB/+oIFm31a3z/8deCytaW/AHf/AH/yi2a6L45sAf/KOCpKPL0/KV7gSAyVGr/mMh+IM/+IN/Q/GvPQCAP/hHDIlr7/5PvxMqDMAf/MEf/BuLf20BAPzBvw6QrNvq2NX/74bgD/7gD/4Nx7/6AAD+4F8nSFyaAAwPWl29aMEf/MEf/BuOf3UBAPzBv06QtLRJ3avdCQBn3wvBH/zBH/ybgn/lAQD8wb+OkKxa59bT//qPh+AP/uAP/k3BX7aSAAD+4F9nSFZvcEf/iVGroU+O/8Ef/MEf/BuEf/kTAPAH/wZAsnq9OwFg4MQnfqjAH/zBH/wbiH95AQD8wb9BkLgUAC5+GII/+IM/+DcN/6UDAPiDf4Mg8bNS9xp3AsDHZy34gz/4g3/T8F88AIA/+DcQku7VnjxHbgEwPGQ1MWrBH/zBH/ybhv/CAQD8wb/BkHQ59PG/y+fAH/zBH/ybi3/pAAD+4N8ESLocGv8PXQjBH/zBH/yb3jMD/uAfB0hcugHQ0IAFf/AHf/Bves8M+IN/HCBx5RRAGEjXPrbgD/7gD/5N75kBf/CPAySuBIDhQasgAH/wB3/wb37PDPiDf7Mh8TNSW6cT/uv6FfAHf/AH/+b3TJIM+IN/syHpWO7O+f8bV0LwB3/wB/+m429V7eOAwR/8I4Skfbkz/uvGoE1Fz8Af/ME/2fgr0gAA/uBf5XY7utyZAIxes6noGfiDP/gnG//oAgD4g38N23XpFMDYDfAHf/AH/+bjH00AAH/wr3G7HY6cAijkpalxm4qegT/4g3+y8a89AIA/+Eew3dYONyYA48PgD/7gD/7xwL+2AAD+4B/Rdlva3JgATI7Z1PQM/MEf/JONf/UBAPzBP8LtuhIApsbAH/zBH/zjgX91AQD8wT/iNci2unEKYHLcpqZn4A/+4J9s/CsPAOAP/nVYg1ZXJgDj4A/+4A/+8cBftpIAAP7gX6c1yDoSAPLTNjU9A3/wB/9k41/+BAD8wb+Oa5BpceMUQD4H/uAP/uAfD/zLCwDgD/517pkxcqIK0+AP/uAP/vHAf+kAAP7g34CeGd+NAJDPW/AHf/AH/1jgv3gAAH/wb0TPPMlz5E7ANgB/8Ad/8I8H/gsHAPAH/wb1zJXxvySFIfiDP/iDfzzwLx0AwB/8G9gzV8b/8wIA+IM/+IN/E/GfHwDAH/wb3DOXJgA2BH/wB3/wjwf+cwMA+IN/E3pmQ3cCgOeBP/iDP/jHA//ZAAD+4N+knoUOBYAoph3gD/7gD/5RbdOAP/g3s2dh4NAEwCSkZ+AP/uCfevxnJwDgD/5N6pm1xS8mAOAP/uAP/o3Dv6YAAP7gH1XPXJkCZFrAH/zBH/zjgX/VAQD8wT/KnrlyIWA1jz0Gf/AHf/Cv1xqYyPYf/MG/yp4V8o4EgJaY9gz8wR/8ncO/4gAA/uBfj57lpty4CKCSCQD4gz/4g3+918DUvEPgD/419iw35cYEoLUD/MEf/ME/HviXHQDAH/zr2TNXJgDtnV58egb+4A/+TuNfVgAAf/Cvd89y025MANqWgT/4gz/4xwP/JQMA+IN/I3qWd+QUQPsyD/zBH/zBPxb4LxoAwB/8G9WzqQk3TgF0dnvgD/7gD/6xwH/BAAD+4N/Ink2OujEByLbOvxAQ/MEf/MG/WT9jBvzBv9k9mxh15F7Akpat8MAf/MEf/JuO/7wJAPiDfzN6NjHsTgBYvtIDf/AHf/CPxc+YAX/wb2rPrDThyCkASepe64E/+IM/+Df9Z+xmAAB/8G8W/pI0PuLOBGDFWgP+4A/+4N90/CXJgD/4NxN/SQry0vSkGwFg5VoP/MEf/MG/6e5aLfQxQPAH/wbhP1MjV92YAqxY68kY8Ad/8Af/5uKvkgEA/MG/wfhL0siQGwHAz0grej3wB3/wB/+m4j8/AIA/+DcBf0kavurOdQA9faYp/QJ/8Ad/8FfJAAD+4N8k/F2aAEhSzyavaccX+IM/+IP/3AAA/uDfRPwladihALBuq2nK8QX+4A/+4D83AIA/+DcZ/5sTAEcywIq13qIPBgJ/8Ad/8G9Ez0yzFgz8wf/Wyufcug5g3TavYccX+IM/+IN/YwIA+IN/lft67ZI7AWDDTtOQ4wv8wR/8wb8xAQD8wb+GfR266E4A2Hy7qfvxBf7gD/7g35gAAP7gX+O+Xr3gTgDo7Pa0eoNXt7UFf/AHf/BvTAAAf/CPYF+vXgrlUm3eY+qytuAP/uAP/o0JAOAP/hHt6/SENHrNnSnA9gM++IM/+IN/03pm6r1g4A/+ldTHZ90JACt7Pa1a74E/+IM/+DelZ6aeCwb+4F9xADjn1mmAHYd88Ad/8Af/pvTM1GvBwB/8q6lLDk0AJGnXYTP7dEDwB3/wB/8G9szUY8HAH/yrrbFrVmM33AkBHV3e/IsBwR/8wR/8G9AzE/UKgD/417qvHzs2Bdhzvw/+4A/+4N/wnpkoVwD8wT+KfR04GTgVAPp2Gq1Y64E/+IM/+De0ZyaqFQB/8I8GEquBk6GsS9cCetKBh/z6tAH8wR/8wX+BjZgovjnwB/+o8Jek6Unpcr9bnwbYedhXZ7cXbRvAH/zBH/wX2YgBf/CPE/4z1X/MrQBgfOngwz74gz/4g3/DembAH/zjhr8kfXTCrQAgSbff52v5Sg/8wR/8wb8hPTPgD/5xw1+SblyxujHo1qcBjC/d9bgP/uAP/uDfkJ4Z8Af/uOE/U2eOuPVpAEnacdjXmj4P/MEf/MG/7mtgwB/844i/JJ1+x73TAJ4nferJjOSBP/iDP/jXdw0M+IN/HPGXik8GvNLvXghYu9notnv8mPUM/MEf/NPWMwP+4B9H/GfqQwenAJJ07+9m1NHlxaRn4A/+4J/GnhnwB/+44i9JZ94LFbp3KYBa26WHfj8D/uAP/uBftzUwdfn+wR/8I8BfkqbGrc6+7+YUYNPtRrff64M/+IM/+Ndlfw34g39c8Z+pY686OAL4bT3w5YxWrvPAH/zBH/wj318D/uAfZ/wl6fL5UNcuuXVPgJnKZKXP/WlW2RbwB3/wB/9o99eAP/jHGX+mANKKHk+f+Wq2fn8B+IM/+DuHf8kAAP7gHzf8ZaXT7wSannQ2A2jbfqN7Pp8B/wTi377M02P/KFu3fQV/8I8kAIA/+McRf0kq5KSjL7s7BZCkQ4/62n23X6eegX898N9+0Oir/yar7QdNXfYV/MG/lsqAP/jHHf+Z+uDlgvY96M+eD3ewHvyDjHKTVuc+CCPsGfhHjX/7Mk+f/kpGOw7/9lYrYfT7Cv7gH8kEAPzBP+74S9L0hHT8VbenAMZIj/7DrPp2mYh6Bv5Rf1u33+fra3/RchN/3vmDfxx7JkkG/ME/CfjP1HsvBgoKTmcA+Rnp8a9ntXG3qbFn4B/lt7Wix9OX/2lWD38to9YOdyAB/2Tib1XGjYDAH/zjgr8kTY5aHX/d7SmAVPx44Oe/kdXWfabKnoF/VC/Ntkr3fSmjr/7rFm3YYeq6r+AP/lFuzjRzJ8Af/KvZ33eeDZSfdj4DyPjFewTsud+vsGfgH8lLPWn33b7++C9adPizvoxf330Ff/CPdHP2losAwR/8k4C/VLw98JHnC7q7Hh+LS1h5Rvr0VzJasdbTq08VSl9sBv6R479um9EDX/bVu8U0ZF/BH/yjxl8qNwCAP/jHBP+Zev+lQHvu99XZ7YmS9n3a14q1np77bkFT4xb864T/qnWe7vtSRlvuMA3bV/AH/3rgL5VzCgD8wT9m+EtSkJfe/AXXAtxaG3cbfeVfZrVumwH/iPHvWu3pkT/O6A//VQv4g38q8F96AgD+4B9D/Gfq9NuB9jzgq2cjU4CZ6uz29MQ/yerIc4HefLqgMAD/Wl7a3ePprsd87brTl2fq2DjwB/8G4794AAB/8I8x/pIUWunFv83ryX/WUt9/nBNWnpEOfdbXljuMnv9eXoMDNjY9Swr+PRs9HXw4ox2HTP2PLfAH/ybgv3AAAH/wjzn+M5u4esnq/ZcC7X/QFzW3Vq7z9OS/aNHx1wL95ufB3GsDwH/+Sz1py16jgw/70X6cD/zBP4b4lw4A4A/+CcF/pt58pqBt+4yWreRUwLxpgCftud/XjoO+3vplQUdfDlTIg/+tL21f5um2e432PuCra3UDjyHwB/8m4j8/AIA/+CcMf0kqTEsv/bigz3+9jo/MTXi1tEv3P5HRgYd8vfNsoGOvlXlHxZTiLyNt2m10+72+tu030X6GH/zBPwH4zw0A4A/+CcR/5jc/Oh7q5BtBtE/LS2F1dHn61JMZ3fmYr6Mvh/rglUCTo9YZ/Hs2e9p1p6+dh43alzdpYgT+4B8D/GcDAPiDf4Lxn6lXflLQum2msWPchFZbp6c7H/N16FFfZ98PdfI3gQZOhbM3EkoJ/sZI63cYbd1X/Gr6aSLwB/+Y4F8MAOAP/inAX5LyOen57xX0xJ9nZfhUQFllfGnHQaMdB40mRqw+fCfU2fdCXT4XytoI+9bAn7HuHk99u4w27jbq22XU0h6TxQZ/8I8R/rMTAPAH/4TjP1NX+kO982xBd36O2wRXWh1dnvY/5Gv/Q76mxqz6j4e6eNrq4ulQYzds9X2r43FrfGlNn6ferUZrtxit3+apc0UyJkDgD/7NxD/6AAD+4B+Dnr39bKD1243Wb2cMUG21LfO0+25fu+8u/vfIVavBj6wGB0INfWR17bLV1JhtGP7GSMtWeupe42nV+uLX6vVGK3o9+QnMeuAP/s3GP9oAAP7gH5Oe2VB69jsFPfnPszwrIKLqWu2pa7WnHYdmQ9X0hHRjMNTIkNX4sDQ+bDU+bDU1KeUmrXITUm7aKgx080sq3qjI+MWvTMZTS3vxkbot7Z5a26WO5Z46ujx1dEmdXZ661nhattJLzWkd8Af/OOAfXQAAf/CPWc8mx6ye+Zu8nvjzlkS+Q0xCtXZIvVuMerewFuAP/knDXyrnYUDgD/4J7dnggNXLPy6IosAf/ME/6gAA/uAf856d+E2go6/w1ECKBAH+4B9dAAB/8E9Iz175u4LOHw0BhAJ/8Af/mgMA+IN/gnpmQ+m57+Z1pZ8QQIE/+IN/9QEA/ME/gT0r5KRf/HVBw0P1WmSKAn/wTw7+lQcA8Af/JPbst9ucGrf6+X/Pa2KEEECBP/i7jb9sJQEA/ME/wfjP1Og1q5/+l/zCD8ChKPAHfwfwL38CAP7gnwL8Z2p40Oqn38prcowQQIE/+LuJf3kBAPzBP0X4z9QNQgAF/uDvMP5LBwDwB/8U4n8zBFwpng7gmgAK/MHfNfwXDwDgD/4pxn/mpdcvW/34m3k+HUCBP/g7hf/CAQD8wd8B/Gdq7LrVT76Z1+AAIYACf/B3A//SAQD8wd8h/GdqatzqqW/lNHCSmwVR4A/+6cd/fgAAf/B3EP+ZKuSk//fXeR1/jWcHUOAP/unGf24AAH/wdxj/mQoD6dd/W9BLPyrcfH49RYE/+KcN/9kAAP7gD/5zXnz0lUA/+695TY1zXQAF/uCfPvyLAQD8wR/8S7740plQP/pPeQ1xcSAF/uCfMvxvTgDAH/zBv/SLx65b/fg/5/TeC0H9ekqBP/iDfxP21YA/+IP/4i8OA+nVpwr6+f/gzoFUHHIE+IN/7ftqVc3jgMEf/B3C/9b66ESoH/6HPB8VpMAf/BOPvyINAOAP/inGf6Ymx6z+/r/l9cIPCspNAhIF/uCfTPyjCwDgD/4O4H9rnXg90Pf/fU7n3mcaQIE/+CcP/2gCAPiDv2P4z9TEiNXT/yuvZ/4mr4lRrg2gwB/8k4O/JGXAH/zBv7Y6+16ogZM5HX40o30P+vIzoqg5NV7lEyfBH/zrhX9tAQD8wR/8b1Z+Snr9ZwUdezXQfU9ktG2/EUWNXLV655dBVbeXBn/wryf+1QcA8Ad/8C+53ZFrxdMC67cb3fdERms3eSjoYN24YvXW04FOvREoDKs5pMAf/OuLvyR5//PfTttGLxj4g38a8S/1V2zeY3TX4xn1bCQIuFBXL1q99YuCPnwnlLXVHlLgD/71x7/yCQD4gz/4l42/JPUfC9V/LKctdxjd9VhGa/oIAmmsC6dCvft8oHMfhDUdb+AP/o3CX7aSAAD+4A/+FeF/a53/INT5D3LadJvR/od8bdzNNQJJr+lJ6eTrgd5/KdCNyzaCQwr8wb9x+Jc/AQB/8Af/qvG/tT46EeqjE6FW9nra/6CvXXfxqYGk1eCA1QcvBjr1ZqBCLqpDCvzBv7H4S+VcAwD+4A/+keBfqto6Pd12j9Ft9/pa0cPpgbhWflr68EigD14MdKXf8mAf8E88/ksHAPAHf/CvG/6f3FjvVqPb7jHacchXthV0m11BvngNx6m3A51/P1QhH33zwR/8m4X/4gEA/MEf/BuG/62VyUrbDvjaftBo427DKYIGVhhIAydCnX471Jl3A+Wn63d8gT/4NxN/aaFrAMAf/MG/KfhLUiEvnXqzeI452ypt2Wu0bb+vTXuMMlmQjroKOeni6VBn3wv14ZFA0xP1P77AH/ybjX/pAAD+4A/+TcP/k5Wflk6/XXxHmslKG3YVpwIbbzNcM1DD2l+9aNV/PNTAiVCXzoQKCo07vsAf/OOA//wAAP7gD/6xwb/UZKD/aKj+o8Vby3Wu8LTxtmIgWLfNU2cXgWChmhyzGjhh9dHx4qcwJkZsU44v8Af/uOA/NwCAP/iDf2zxL7WZsRtWx18LdPzV4n3ml6/01LvVU+8Wo96tRqs3eDK+g2/wQ+n6ZauPz4X6+KzVx2dDDQ/apvYL/ME/bvjPBgDwB3/wTxT+pf5j9LrV6HWr028XJwR+Rlq5ztPq9Uar1ntavd7TqvWe2penZ1KQny5if+1iqKEBq8EBq6sXQ+Vz8ekX+IN/HPEvBgDwB3/wTzz+pSooSEMDVkMDc59E19bpqbvHU9cqqWuNp65VnrrWeFq+ylPHck9ezG5SWMgVH7I0etVq5LdfN65YXb9sNXbNxh4S8Af/OOI/OwEAf/AH/1Thv1hNjVtNjVtdPjf/zzxJ7cs8tXdJHcuL04KO5cXQ0NImZds8tbZJ2Tappc1TtrU4afCMZHxPxkjGl4wp/p61xZF8GBZ/DQpW+ZxUmJbyOSmfs8pNSFMTVtMTxX2bHJPGh60mhq3GR2xFV+WDP/iDf4MCAPiDP/gnC/+l/hIraWLUamJUuiob356BP/iDf81rYCL9PsAf/ME/sfgnpmfgD/7gH8kamMi+D/AHf/AHf/AHf/BPBP5VBQDwB3/wB3/wB3/wTzb+FQcA8Ad/8Ad/8Ad/8E8+/hUFAPAHf/AHf/AHf/BPB/5lBwDwB3/wB3/wB3/wTw/+ZQUA8Ad/8Ad/8Ad/8E8X/ksGAPAHf/AHf/AHf/BPH/6LBgDwB3/wB3/wB3/wTyf+sgsEAPAHf/AHf/AHf/BPL/4lJwDgD/7gD/7gD/7gn2785wUA8Ad/8Ad/8Ad/8E8//nMCAPiDP/iDP/iDP/i7gf/NAAD+4A/+4A/+4A/+7uAvSQb8wR/8wR/8wR/83cLfqtSnAMAf/MEf/MEf/ME/1fhrXgAAf/AHf/AHf/AH/9TjPzcAgD/4gz/4gz/4g78T+M8GAPAHf/AHf/AHf/B3Bv9iAAB/8Ad/8Ad/8Ad/p/CfnQCAP/iDP/iDP/iDvzP41ycAgD/4gz/4gz/4g3+s8Y8+AIA/+IM/+IM/+IN/7PGPNgCAP/iDP/iDP/iDfyLwjy4AgD/4gz/4gz/4g39i8I8mAIA/+IM/+IM/+IN/ovCvPQCAP/iDP/iDP/iDf+Lwry0AgD/4gz/4gz/4g38i8a8+AIA/+IM/+IM/+IN/YvGvLgCAP/iDP/iDP/iDf6LxrzwAgD/4gz/4gz/4g3/i8ZetJACAP/iDP/iDP/iDfyrwL38CAP7gD/7gD/7gD/6pwb+8AAD+4A/+4A/+4A/+qcJ/6QAA/uAP/uAP/uAP/qnDf/EAAP7gD/7gD/7gD/6pxH/hAAD+4A/+4A/+4A/+qcW/dAAAf/AHf/AHf/AH/1TjPz8AgD/4gz/4gz/4g3/q8Z8bAMAf/MEf/MEf/MHfCfxnAwD4gz/4gz/4gz/4O4N/MQCAP/iDP/iDP/iDv1P4z04AwB/8wR/8wR/8wd8Z/JcMAOAP/uAP/uAP/uCfPvwXDQDgD/7gD/7gD/7gn078FwwA4A/+4A/+4A/+4J9e/EsGAPAHf/AHf/AHf/BPN/7zAgD4gz/4gz/4gz/4px//OQEA/MEf/MEf/MEf/N3A/2YAAH/wB3/wB3/wB3938JckA/7gD/7gD/7gD/5u4W9Vxo2AwB/8wR/8wR/8wT9d+MtWEgDAH/zBH/zBH/zBPxX4q+wJAPiDP/iDP/iDP/inBv/yAgD4gz/4gz/4gz/4pwr/pQMA+IM/+IM/+IM/+KcO/8UDAPiDP/iDP/iDP/inEv+FAwD4gz/4gz/4gz/4pxb/0gEA/MEf/MEf/MEf/FON//wAAP7gD/7gD/7gD/6px39uAAB/8Ad/8Ad/8Ad/J/CfDQDgD/7gD/7gD/7g7wz+xQAA/uAP/uAP/uAP/k7hPzsBAH/wB3/wB3/wB39n8K9PAAB/8Ad/8Ad/8Af/WOMffQAAf/AHf/AHf/AH/9jjH20AAH/wB3/wB3/wB/9E4B9dAAB/8Ad/8Ad/8Af/xOAfTQAAf/AHf/AHf/AH/0ThX3sAAH/wB3/wB3/wB//E4V9bAAB/8Ad/8Ad/8Af/ROJffQAAf/AHf/AHf/AH/8TiX10AAH/wB3/wB3/wB/9E4195AAB/8Ad/8Ad/8Af/xOMvW0kAAH/wB3/wB3/wB/9U4F/+BAD8wR/8wR/8wR/8U4N/eQEA/MEf/MEf/MEf/FOF/9IBAPzBH/zBH/zBH/xTh//iAQD8wR/8wR/8wR/8U4n/wgEA/MEf/MEf/MEf/FOLf+kAAP7gD/7gD/7gD/6pxn9+AAB/8Ad/8Ad/8Af/1OM/NwCAP/iDP/iDP/iDvxP4zwYA8Ad/8Ad/8Ad/8HcG/2IAAH/wB3/wB3/wB3+n8J+dAIA/+IM/+IM/+IO/M/jfDADgD/7gD/7gD/7g7w7+kmTAH/zBH/zBH/zB3y38rWyFjwMGf/AHf/AHf/AH/8TjLymiAAD+4A/+4A/+4A/+icE/mgAA/uAP/uAP/uAP/onCv/YAAP7gD/7gD/7gD/6Jw7+2AAD+4A/+4A/+4A/+icS/+gAA/uAP/uAP/uAP/onFv7oAAP7gD/7gD/7gD/6Jxr/yAAD+4A/+4A/+4A/+icdftpIAAP7gD/7gD/7gD/6pwL/8CQD4gz/4gz/4gz/4pwb/8gIA+IM/+IM/+IM/+KcK/6UDAPiDP/iDP/iDP/inDv/FAwD4gz/4gz/4gz/4pxL/hQMA+IM/+IM/+IM/+KcW/9IBAPzBH/zBH/zBH/xTjf/8AAD+4A/+4A/+4A/+qcd/bgAAf/AHf/AHf/AHfyfwnw0A4A/+4A/+4A/+4O8M/sUAAP7gD/7gD/7gD/5O4T87AQB/8Ad/8Ad/8Ad/Z/CvOQCAP/iDP/iDP/iDf/LwrykAgD/4gz/4gz/4g38y8a86AIA/+IM/+IM/+IN/cvGvKgCAP/iDP/iDP/iDf7LxrzgAgD/4gz/4gz/4g3/y8a8oAIA/+IM/+IM/+IN/OvAvOwCAP/iDP/iDP/iDf3rwLysAgD/4gz/4gz/4g3+68F8yAIA/+IM/+IM/+IN/+vBfNACAP/iDP/iDP/iDfzrxl10gAIA/+IM/+IM/+IN/evEvOQEAf/AHf/AHf/AH/3TjPy8AgD/4gz/4gz/4g3/68Z8TAMAf/MEf/MEf/MHfDfxvBgDwB3/wB3/wB3/wdwd/STLgD/7gD/7gD/7g7xb+VqU+BQD+4A/+4A/+4A/+qcZf8wIA+IM/+IM/+IM/+Kce/7kBAPzBH/zBH/zBH/ydwH82AIA/+IM/+IM/+IO/M/gXAwD4gz/4gz/4gz/4O4X/7AQA/MEf/MEf/MEf/J3BP/oAAP7gD/7gD/7gD/6J6JlpxoKBP/iDP/iDP/iDf3N7Zhq9YOAP/uAP/uAP/uDf/J6ZRi4Y+IM/+IM/+IM/+MejZ6ZRCwb+4A/+4A/+4A/+8emZacSCgT/4gz/4gz/4g3+8embqvWDgD/7gD/7gD/7gH7+emXouGPiDP/iDP/iDP/jHs2emXgsG/uAP/uAP/uAP/jHtma0kAIA/+IM/+IM/+IN/KvAvfwIA/uAP/uAP/uAP/qnBv7wAAP7gD/7gD/7gD/6pwn/pAAD+4A/+4A/+4A/+qcN/8QAA/uAP/uAP/uAP/qnEf+EAAP7gD/7gD/7gD/6pxb90AAB/8Ad/8Ad/8Af/VOM/PwCAP/iDP/iDP/iDf+rxnxsAwB/8wR/8wR/8wd8J/GcDAPiDP/iDP/iDP/g7g38xAIA/+IM/+IM/+IO/U/jPTgDAH/zBH/zBH/zB3xn8yw4A4A/+4A/+4A/+4J8e/MsKAOAP/uAP/uAP/uCfLvyXDADgD/7gD/7gD/7gnz78Fw0A4A/+4A/+4A/+4J9O/BcMAOAP/uAP/uAP/uCfXvxLBgDwB3/wB3/wB3/wTzf+8wIA+IM/+IM/+IM/+Kcf/zkBAPzBH/zBH/zBH/zdwP9mAAB/8Ad/8Ad/8Ad/d/CXJAP+4A/+4A/+4A/+buFvbQW3AgZ/8Ad/8Ad/8Af/dOAvlRsAwB/8wR/8wR/8wT81+JcXAMAf/MEf/MEf/ME/VfgvHQDAH/zBH/zBH/zBP3X4Lx4AwB/8wR/8wR/8wT+V+C8cAMAf/MEf/MEf/ME/tfiXDgDgD/7gD/7gD/7gn2r85wcA8Ad/8Ad/8Ad/8E89/nMDAPiDP/iDP/iDP/g7gf9sAAB/8Ad/8Ad/8Ad/Z/AvBgDwB3/wB3/wB3/wdwr/2QkA+IM/+IM/+IM/+DuDf30CAPiDP/iDP/iDP/jHGv/oAwD4gz/4gz/4gz/4xx7/aAMA+IM/+IM/+IM/+CcC/+gCAPiDP/iDP/iDP/gnBv9oAgD4gz/4gz/4gz/4Jwr/2gMA+IM/+IM/+IM/+CcO/9oCAPiDP/iDP/iDP/gnEv/qAwD4gz/4gz/4gz/4Jxb/6gIA+IM/+IM/+IM/+Cca/8oDAPiDP/iDP/iDP/gnHn/ZSgIA+IM/+IM/+IM/+KcC//InAOAP/uAP/uAP/uCfGvzLCwDgD/7gD/7gD/7gnyr8lw4A4A/+4A/+4A/+4J86/BcPAOAP/uAP/uAP/uCfSvwXDgDgD/7gD/7gD/7gn1r8SwcA8Ad/8Ad/8Ad/8E81/vMDAPiDP/iDP/iDP/inHv+5AQD8wR/8wR/8wR/8ncB/NgCAP/iDP/iDP/iDvzP4FwMA+IM/+IM/+IM/+DuF/+wEAPzBH/zBH/zBH/ydwf9mAAB/8Ad/8Ad/8Ad/d/CXJAP+4A/+4A/+4A/+buFvVenjgMEf/MEf/MEf/ME/8fgrsgAA/uAP/uAP/uAP/onBP5oAAP7gD/7gD/7gD/6Jwr/2AAD+4A/+4A/+4A/+icO/tgAA/uAP/uAP/uAP/onEv/oAAP7gD/7gD/7gD/6Jxb+6AAD+4A/+4A/+4A/+ica/8gAA/uAP/uAP/uAP/onHX7aSAAD+4A/+4A/+4A/+qcC//AkA+IM/+IM/+IM/+KcG//ICAPiDP/iDP/iDP/inCv+lAwD4gz/4gz/4gz/4pw7/xQMA+IM/+IM/+IM/+KcS/4UDAPiDP/iDP/iDP/inFv/SAQD8wR/8wR/8wR/8U43//AAA/uAP/uAP/uAP/qnHf24AAH/wB3/wB3/wB38n8J8NAOAP/uAP/uAP/uDvDP6S9P8BqSTjapatHakAAAAASUVORK5CYII='

SW_JS = """
const CACHE = "chatterbox-v1";
const SHELL = ["/", "/manifest.webmanifest", "/icon.svg", "/icon-180.png", "/icon-512.png"];
self.addEventListener("install", e => {
  e.waitUntil(caches.open(CACHE).then(c => c.addAll(SHELL)).then(() => self.skipWaiting()));
});
self.addEventListener("activate", e => e.waitUntil(
  caches.keys().then(ks => Promise.all(ks.filter(k => k !== CACHE).map(k => caches.delete(k))))
    .then(() => self.clients.claim())
));
self.addEventListener("fetch", e => {
  const u = new URL(e.request.url);
  if (u.pathname.startsWith("/api/")) return;   // never cache API/SSE
  e.respondWith(
    fetch(e.request).then(r => {
      const copy = r.clone();
      if (r.ok && e.request.method === "GET" && SHELL.includes(u.pathname))
        caches.open(CACHE).then(c => c.put(e.request, copy));
      return r;
    }).catch(() => caches.match(e.request).then(r => r || caches.match("/")))
  );
});
"""

print(f"frontend assets built ({len(INDEX_HTML):,} bytes html)")


### 7️⃣ Backend API · instant
Jobs save output locally before browser delivery and mirror to Google Drive when configured. Also serves the model-control endpoints (`/api/model`, `/api/model/load`, `/api/model/unload`), EPUB chapter text, and a job-list endpoint the EPUB queue polls.


In [ ]:
# ── STEP 7 · Backend API (jobs, SSE progress, voice uploads) ─────────────

import asyncio, base64, json, re as _re, uuid, threading, queue as pyqueue, time
from urllib.parse import quote
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path
from fastapi import FastAPI, HTTPException, Response, Request, UploadFile, File
from fastapi.responses import HTMLResponse, StreamingResponse
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"],
                   allow_headers=["*"], expose_headers=["X-Meta"])

# ---------------------------------------------------------------------------
# Voice registry — built-in voice + user clips persisted under /kaggle/working
# ---------------------------------------------------------------------------
VOICES_DIR = Path("/kaggle/working/voices")
VOICES_DIR.mkdir(parents=True, exist_ok=True)

def _voice_path(vid: str) -> Path: return VOICES_DIR / f"{vid}.wav"
def _voice_meta(vid: str) -> Path: return VOICES_DIR / f"{vid}.json"

def list_voices() -> list[dict]:
    out = [{"id": "default", "name": "Default voice", "kind": "builtin", "seconds": None}]
    for meta in sorted(VOICES_DIR.glob("*.json"), key=lambda p: p.stat().st_mtime):
        try:
            j = json.loads(meta.read_text())
            vid = meta.stem
            if _voice_path(vid).exists():
                out.append({"id": vid, "name": j.get("name", vid[:8]),
                            "kind": "cloned", "seconds": j.get("seconds")})
        except Exception:
            pass
    return out

@app.post("/api/voices")
async def upload_voice(file: UploadFile):
    raw = await file.read()
    if len(raw) < 1000:        raise HTTPException(400, "file looks empty")
    if len(raw) > 30 << 20:    raise HTTPException(413, "clip too large (30 MB max)")
    vid = uuid.uuid4().hex[:12]
    tmp = VOICES_DIR / f"_up_{vid}"
    try:
        tmp.write_bytes(raw)
        # Transcode anything the phone throws at us (webm/m4a/mp3/…) to 24k mono
        # WAV, capped at 30 s — that's well past the 10 s Chatterbox actually uses.
        from pydub import AudioSegment
        seg = AudioSegment.from_file(str(tmp))
        seg = seg.set_channels(1).set_frame_rate(SAMPLE_RATE)
        if len(seg) > 30_000: seg = seg[:30_000]
        if len(seg) < 1000:   raise HTTPException(400, "clip too short — need at least ~1 s of speech")
        seg.export(str(_voice_path(vid)), format="wav")
        name = _re.sub(r"\s+", " ", Path(file.filename or "my voice").stem).strip()[:60] or "my voice"
        _voice_meta(vid).write_text(json.dumps({"name": name, "seconds": round(len(seg)/1000, 2)}))
    except HTTPException:
        raise
    except Exception as e:
        for p in (_voice_path(vid), _voice_meta(vid)):
            try: p.unlink()
            except Exception: pass
        raise HTTPException(400, f"could not decode audio clip: {e!s}")
    finally:
        try: tmp.unlink()
        except Exception: pass
    # pre-warm the (expensive) conditionals on one GPU eagerly in background —
    # by the time you hit Generate it's usually embedded already on both.
    def _prewarm():
        for m in MODELS.values():
            try:
                with m._cb_lock: ensure_conditionals(m, str(_voice_path(vid)), 0.5)
            except Exception: pass
    threading.Thread(target=_prewarm, daemon=True).start()
    meta = json.loads(_voice_meta(vid).read_text())
    return {"voice": {"id": vid, "name": meta["name"], "kind": "cloned", "seconds": meta["seconds"]}}

@app.delete("/api/voices/{vid}")
def delete_voice(vid: str):
    if not _re.fullmatch(r"[0-9a-f]{12}", vid): raise HTTPException(400, "bad voice id")
    existed = _voice_path(vid).exists()
    for p in (_voice_path(vid), _voice_meta(vid)):
        try: p.unlink()
        except Exception: pass
    for m in MODELS.values():                    # drop cached conds pointing at it
        cache = getattr(m, "_cb_conds_cache", None)
        if cache: cache.pop(str(_voice_path(vid)), None)
        if str(_voice_path(vid)) == getattr(m, "_cb_voice_path", None):
            try: m.conds = m._cb_default_conds; m._cb_voice_path = None
            except Exception: pass
    if not existed: raise HTTPException(404, "voice not found")
    return {"ok": True}

# ---------------------------------------------------------------------------
# Jobs
# ---------------------------------------------------------------------------
# Every job keeps an append-only event log instead of a one-shot Queue. A phone
# that drops its connection mid-render reconnects with `Last-Event-ID` and the
# server replays everything it missed — including a `done` that fired while the
# radio was off. Cheap: a few hundred small dicts per job at most.
@dataclass
class Job:
    id: str
    text: str; language_id: str; voice_ref: str | None; params: dict; fmt: str
    output_name: str = "chatterbox"
    log: list = field(default_factory=list)          # [(seq, event, data), ...]
    _cv: threading.Condition = field(default_factory=threading.Condition)
    finished: bool = False                           # no further events will arrive
    audio: bytes | None = None
    mime: str = "audio/wav"
    meta: dict = field(default_factory=dict)
    cancelled: bool = False
    error: str | None = None
    started: float = 0.0
    queued: float = field(default_factory=time.time)
    done: int = 0
    total: int = 0

    # -- event log ----------------------------------------------------------
    def emit(self, event: str, data: dict | None):
        with self._cv:
            # Terminal events are idempotent: the cancel endpoint and the
            # worker that notices it both fire "cancelled", and a reconnect
            # must see exactly one terminal event, not a duplicate.
            if event in ("done", "error", "cancelled") and self.finished:
                return
            # Progress events are strictly superseded by the next one, so keep
            # only the newest: a client that reconnects late gets the current
            # position immediately instead of a replay of every chunk tick.
            if event == "progress" and self.log and self.log[-1][1] == "progress":
                self.log[-1] = (self.log[-1][0], event, data)
            else:
                self.log.append((len(self.log) + 1, event, data))
            if event in ("done", "error", "cancelled"):
                self.finished = True
            self._cv.notify_all()

    def since(self, cursor: int) -> list:
        with self._cv:
            return [e for e in self.log if e[0] > cursor]

    def wait_for(self, cursor: int, timeout: float) -> list:
        """Block until there is something newer than `cursor` (or time out)."""
        with self._cv:
            if not any(e[0] > cursor for e in self.log) and not self.finished:
                self._cv.wait(timeout)
            return [e for e in self.log if e[0] > cursor]

    @property
    def state(self) -> str:
        return ("error" if self.error else "cancelled" if self.cancelled
                else "done" if self.audio is not None
                else "running" if self.started else "queued")

    def snapshot(self) -> dict:
        return {"job_id": self.id, "state": self.state, "name": self.output_name,
                "fmt": self.fmt, "done": self.done, "total": self.total,
                "error": self.error, "meta": self.meta,
                "audio_ready": self.audio is not None,
                "chars": len(self.text), "queued": self.queued}

JOBS: dict[str, Job] = {}
EPUB_BOOKS: dict[str, dict] = {}
JOBS_LOCK = threading.Lock()

def _run_job(job: Job):
    try:
        job.started = time.time()
        job.emit("progress", {"done": 0, "total": 0, "phase": "starting"})
        def prog(done, total):
            if job.cancelled: raise RuntimeError("cancelled")
            job.done, job.total = done, total
            job.emit("progress", {"done": done, "total": total, "phase": "synthesizing"})
        sr, wav, nchunks = synth(job.text, job.language_id, job.voice_ref, job.params,
                                 on_progress=prog, is_cancelled=lambda: job.cancelled)
        if job.cancelled:
            job.emit("cancelled", {"message": "cancelled"}); return
        job.emit("progress", {"done": nchunks, "total": nchunks, "phase": "encoding"})
        t_enc0 = time.time()
        audio_bytes, mime = encode_async(sr, wav, job.fmt).result()
        # Persist before notifying the browser. A tunnel/page disconnect cannot
        # waste a completed chapter, and Drive upload runs from the Kaggle VM.
        ext = job.fmt if job.fmt != "opus" else "opus"
        target = OUTPUT_DIR / f"{safe_filename(job.output_name)}.{ext}"
        target.write_bytes(audio_bytes)
        try: drive_id = mirror_to_drive(target)
        except Exception as drive_error:
            drive_id = None; print(f"Drive mirror failed for {target.name}: {drive_error!s}")
        job.audio, job.mime = audio_bytes, mime
        job.meta = {"audio_s": len(wav)/sr, "chunks": nchunks,
                    "wall_s": time.time()-job.started,
                    "encode_s": time.time()-t_enc0,
                    "gpus": len({MODEL_DEVICE[d] for d in WORKERS if MODEL_DEVICE[d].startswith("cuda")}) or 1, "workers": len(WORKERS), "file": target.name, "drive_id": drive_id}
        job.emit("done", job.meta)
    except Exception as e:
        if job.cancelled: job.emit("cancelled", {"message": "cancelled"})
        else:
            job.error = str(e); job.emit("error", {"message": job.error})
    # emit() already marks the job finished on every terminal event, so the
    # SSE stream closes without a redundant second flag write.

_job_pool = ThreadPoolExecutor(max_workers=4)   # several concurrent jobs share the GPU locks

@app.get("/", response_class=HTMLResponse)
def index():
    return HTMLResponse(INDEX_HTML)

@app.get("/manifest.webmanifest")
def manifest():
    return Response(MANIFEST_JSON, media_type="application/manifest+json")

@app.get("/icon.svg")
def icon():
    return Response(ICON_SVG, media_type="image/svg+xml")

@app.get("/icon-180.png")
def icon_180():
    return Response(base64.b64decode(ICON_PNG_180), media_type="image/png",
                    headers={"Cache-Control": "public, max-age=86400"})

@app.get("/icon-512.png")
def icon_512():
    return Response(base64.b64decode(ICON_PNG_512), media_type="image/png",
                    headers={"Cache-Control": "public, max-age=86400"})

@app.get("/sw.js")
def sw():
    return Response(SW_JS, media_type="application/javascript")

@app.get("/api/status")
def status():
    gpu_names = [g["name"] for g in GPUS] if GPUS else []
    cuda_workers = [d for d in WORKERS if MODEL_DEVICE.get(d, "").startswith("cuda")]
    return {
        "model": f"chatterbox-multilingual-{ACTIVE_T3}" if ACTIVE_T3 else "no model loaded",
        "model_ready": model_ready(),
        "model_state": model_status(),
        "watermark": HAS_WATERMARK,
        "n_gpus": len({MODEL_DEVICE[d] for d in cuda_workers}),
        "n_workers": len(WORKERS),
        "n_cpu": N_CPU,
        "gpus": GPUS,
        "fp16": all(FP16.get(d, False) for d in cuda_workers) if cuda_workers else False,
        "languages": LANGUAGE_LIST,
        "voices": list_voices(),
        "sample_rate": globals().get("MODEL_SR", 24000),
        "accelerator": " + ".join(gpu_names) if gpu_names else "CPU",
    }

# ---------------------------------------------------------------------------
# Model control — everything the "Model" card on the web page needs, so weights
# can be downloaded/loaded/unloaded from your phone without touching Kaggle.
# ---------------------------------------------------------------------------
@app.get("/api/model")
def api_model():
    return model_status()

@app.post("/api/model/load")
async def api_model_load(request: Request):
    try: body = await request.json()
    except Exception: body = {}
    st = model_status()
    if st["busy"]: raise HTTPException(409, f"already {st['status']}: {st['message']}")
    t3 = str(body.get("t3") or "auto").lower()
    if t3 not in (["auto"] + T3_MODELS): raise HTTPException(400, f"unknown checkpoint: {t3}")
    try: reps = int(body.get("replicas_per_gpu") or REPLICAS_PER_GPU)
    except Exception: reps = REPLICAS_PER_GPU
    reps = max(1, min(MAX_REPLICAS_PER_GPU, reps))
    validate = bool(body.get("validate", True))
    if body.get("reload") and model_ready():
        unload_models()
    return start_load_async(t3=t3, replicas_per_gpu=reps, validate=validate)

@app.post("/api/model/unload")
def api_model_unload():
    if model_status()["busy"]: raise HTTPException(409, "a load is running — wait for it to finish")
    with JOBS_LOCK:
        active = [j for j in JOBS.values() if j.audio is None and not j.error and not j.cancelled]
    if active: raise HTTPException(409, f"{len(active)} job(s) still rendering — cancel them first")
    return unload_models()

def _clamp(v, lo, hi, default):
    try: v = float(v)
    except Exception: return default
    return max(lo, min(hi, v))

@app.post("/api/jobs")
async def submit(req: Request):
    body = await req.json()
    if not model_ready():
        raise HTTPException(409, "no model loaded — open the Model card and press “Load model”")
    text = (body.get("text") or "").strip()
    if not text: raise HTTPException(400, "text is required")
    if len(text) > 500_000: raise HTTPException(413, "text too large (500k char cap)")
    lang = (body.get("language_id") or DEFAULT_LANG).lower()
    if lang not in LANGUAGES: raise HTTPException(400, f"unknown language: {lang}")
    voice = (body.get("voice") or "default").strip()
    if voice not in ("default", ""):
        if not _re.fullmatch(r"[0-9a-f]{12}", voice) or not _voice_path(voice).exists():
            raise HTTPException(400, "unknown cloned voice — re-upload the clip")
        voice_ref = str(_voice_path(voice))
    else:
        voice_ref = None
    fmt = (body.get("format") or "wav").lower()
    if fmt not in ("wav", "mp3", "flac", "opus"): fmt = "wav"
    params = {
        "exaggeration":       _clamp(body.get("exaggeration", 0.5),      0.0, 2.0,  0.5),
        "cfg_weight":         _clamp(body.get("cfg_weight", 0.5),        0.0, 1.0,  0.5),
        "temperature":        _clamp(body.get("temperature", 0.8),       0.05, 2.0, 0.8),
        "repetition_penalty": _clamp(body.get("repetition_penalty",1.2), 1.0, 5.0,  1.2),
        "min_p":              _clamp(body.get("min_p", 0.05),            0.0, 0.5,  0.05),
        "top_p":              _clamp(body.get("top_p", 1.0),             0.01, 1.0, 1.0),
        "seed":               int(_clamp(body.get("seed", 0),            0, 99999999, 0)),
        "gap_ms":             int(_clamp(body.get("gap_ms", 120),        0, 1500, 120)),
        "chunk_chars":        int(_clamp(body.get("chunk_chars", 260), 140, 400, 260)),
    }
    output_name = safe_filename(body.get("output_name") or "chatterbox")
    job = Job(id=uuid.uuid4().hex, text=text, language_id=lang,
              voice_ref=voice_ref, params=params, fmt=fmt, output_name=output_name)
    with JOBS_LOCK: JOBS[job.id] = job
    _job_pool.submit(_run_job, job)
    return {"job_id": job.id}

@app.post("/api/epub")
async def upload_epub(file: UploadFile = File(...)):
    if not (file.filename or "").lower().endswith(".epub"): raise HTTPException(400, "upload an .epub file")
    raw = await file.read()
    if len(raw) > 100 << 20: raise HTTPException(413, "EPUB is too large (100 MB max)")
    eid = uuid.uuid4().hex; path = EPUB_DIR / f"{eid}.epub"; path.write_bytes(raw)
    try: chapters = epub_chapters(path)
    except Exception as e: path.unlink(missing_ok=True); raise HTTPException(400, f"could not read EPUB: {e!s}")
    EPUB_BOOKS[eid] = {"path": path, "chapters": chapters}
    return {"epub_id": eid, "chapters": [{"id": c["id"], "title": c["title"], "chars": len(c["text"])} for c in chapters]}

@app.post("/api/epub/{epub_id}/jobs")
async def epub_jobs(epub_id: str, request: Request):
    # Browser can submit selected IDs; it receives normal job IDs, one per title.
    book = EPUB_BOOKS.get(epub_id)
    if not book: raise HTTPException(404, "EPUB expired; upload it again")
    body = await request.json(); wanted=set(body.get("chapter_ids") or [])
    selected=[c for c in book["chapters"] if c["id"] in wanted]
    if not selected: raise HTTPException(400, "select at least one chapter")
    if not model_ready():
        raise HTTPException(409, "no model loaded — open the Model card and press “Load model”")
    # Fail fast on anything that would reject EVERY chapter before a single job
    # is submitted. Without this, a mid-list exception aborted the response
    # while chapters already handed to the pool kept rendering server-side,
    # with no job ID ever reaching the client.
    oversized=[c["title"] for c in selected if len(c["text"]) > 500_000]
    if oversized:
        shown=", ".join(oversized[:3]) + ("…" if len(oversized) > 3 else "")
        raise HTTPException(413, f"chapter(s) exceed the 500k char cap: {shown}")
    jobs, failed = [], []
    for c in selected:
        # Reuse validated normal-job route so EPUBs get identical TTS settings.
        body2=dict(body); body2.update({"text": c["text"], "output_name": c["title"]})
        class _R:
            async def json(self): return body2
        try:
            result=await submit(_R())
        except HTTPException as e:
            failed.append({"title": c["title"], "error": str(e.detail)})
            if not jobs: raise            # nothing started yet — clean 4xx
            continue                      # keep submitting the rest, report below
        jobs.append({"title":c["title"], **result})
    return {"jobs": jobs, "failed": failed,
            "saved_to": str(OUTPUT_DIR), "drive_enabled": bool(DRIVE_SERVICE)}

@app.get("/api/epub/{epub_id}/chapters/{chapter_id}")
def epub_chapter_text(epub_id: str, chapter_id: str):
    book = EPUB_BOOKS.get(epub_id)
    if not book: raise HTTPException(404, "EPUB expired; upload it again")
    for c in book["chapters"]:
        if c["id"] == chapter_id:
            return {"id": c["id"], "title": c["title"], "text": c["text"]}
    raise HTTPException(404, "chapter not found")

@app.get("/api/jobs")
def list_jobs():
    with JOBS_LOCK: js = list(JOBS.values())
    js.sort(key=lambda j: j.queued)
    return {"jobs": [j.snapshot() for j in js]}

@app.get("/api/jobs/{job_id}")
def job_state(job_id: str):
    job = JOBS.get(job_id)
    if not job: raise HTTPException(404, "job not found")
    return job.snapshot()

@app.get("/api/jobs/{job_id}/events")
async def events(job_id: str, request: Request):
    job = JOBS.get(job_id)
    if not job: raise HTTPException(404, "job not found")
    # Resume point: the browser's EventSource sends back the last id it saw, so
    # a reconnect after a tunnel/radio blip replays only what was missed.
    try: cursor = int(request.headers.get("last-event-id")
                      or request.query_params.get("since") or 0)
    except Exception: cursor = 0

    async def gen():
        nonlocal cursor
        loop = asyncio.get_event_loop()
        yield "retry: 2000\n\n"
        while True:
            if await request.is_disconnected(): break
            pending = await loop.run_in_executor(None, job.wait_for, cursor, 1.0)
            for seq, evt, data in pending:
                cursor = seq
                yield f"id: {seq}\nevent: {evt}\ndata: {json.dumps(data)}\n\n"
            if job.finished and not job.since(cursor):
                # Terminal state already delivered — close so the client stops
                # reconnecting instead of hanging on keepalives forever.
                yield f"id: {cursor}\nevent: eof\ndata: {json.dumps(job.snapshot())}\n\n"
                break
            if not pending:
                yield ": keepalive\n\n"
    headers = {"Cache-Control": "no-cache", "X-Accel-Buffering": "no", "Connection": "keep-alive"}
    return StreamingResponse(gen(), media_type="text/event-stream", headers=headers)

@app.get("/api/jobs/{job_id}/audio")
def get_audio(job_id: str):
    job = JOBS.get(job_id)
    if not job: raise HTTPException(404, "job not found")
    if job.error: raise HTTPException(500, job.error)
    if job.audio is None: raise HTTPException(425, "not ready")
    download_name = f"{safe_filename(job.output_name)}.{job.fmt}"
    # filename*= avoids a latin-1 header error for Hindi/CJK/Arabic chapter titles.
    hdr = {"X-Meta": json.dumps(job.meta), "Cache-Control": "no-store",
           "Content-Disposition": "attachment; filename*=UTF-8''" + quote(download_name)}
    return Response(job.audio, media_type=job.mime, headers=hdr)

@app.delete("/api/jobs/{job_id}")
def cancel(job_id: str):
    job = JOBS.get(job_id)
    if not job: raise HTTPException(404, "job not found")
    if job.finished:                    # already terminal: don't re-fire or flip state
        return {"ok": True, "state": job.state}
    job.cancelled = True
    job.emit("cancelled", {"message": "cancelled"})
    return {"ok": True, "state": "cancelled"}

def _reap():
    while True:
        time.sleep(60)
        with JOBS_LOCK:
            # Never reap queued/running EPUB jobs: a long book can have many
            # queued chapters. Only discard terminal jobs beyond the history cap.
            terminal = [j for j in JOBS.values() if j.audio is not None or j.error or j.cancelled]
            if len(terminal) > 32:
                for old in sorted(terminal, key=lambda j: j.started)[:len(terminal)-32]:
                    JOBS.pop(old.id, None)
if not globals().get("_reaper_started"):             # cell re-run: don't stack reapers
    globals()["_reaper_started"] = True
    threading.Thread(target=_reap, daemon=True).start()

print("FastAPI app defined")


### 8️⃣ Start server + tunnel · ~10 s → **your URL appears here**

Uvicorn on `0.0.0.0:7860` + a Cloudflare quick tunnel. Watch this cell's output and **tap the big
`trycloudflare.com` link** — that's your app. Keep this Kaggle tab open while you use it.


In [ ]:
# ── STEP 8 · Start server + tunnel → tap the big URL in the output ──────

import threading, uvicorn, subprocess, re, time, sys

PORT = 7860
_server = None
def _serve():
    global _server
    cfg = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning", access_log=False)
    _server = uvicorn.Server(cfg)
    _server.run()

# Re-run friendly: only one uvicorn + one tunnel, ever.
if not globals().get("_server_started"):
    threading.Thread(target=_serve, daemon=True).start()
    globals()["_server_started"] = True
    time.sleep(2)
print(f"uvicorn on http://0.0.0.0:{PORT}")
print("model:", "ready" if globals().get("WORKERS") else "not loaded — use the Model card in the web app")

URL_RE = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
_prev = globals().get("_cf_proc")

if _prev is not None and _prev.poll() is None and globals().get("_public_url"):
    public_url = globals()["_public_url"]            # tunnel still alive — reuse it
    print(f"tunnel already up: {public_url}")
else:
    # Start cloudflared quick tunnel; scrape stdout for the public URL
    cf_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    globals()["_cf_proc"] = cf_proc
    public_url = None
    t_start = time.time()
    while time.time() - t_start < 60:
        line = cf_proc.stdout.readline()
        if not line: time.sleep(0.1); continue
        sys.stdout.write(line)
        m = URL_RE.search(line)
        if m: public_url = m.group(0); break
    globals()["_public_url"] = public_url

if public_url:
    print("\n" + "="*60)
    print(f"  📱  Open this on your phone:  {public_url}")
    print(f"      (also http://localhost:{PORT} from inside the kernel)")
    print("="*60)
    try:
        from IPython.display import display, HTML
        display(HTML(
            f'<div style="padding:14px;border-radius:16px;background:#0d0b14;border:1px solid #2b2540;'
            f'font-family:-apple-system,system-ui,sans-serif;text-align:center">'
            f'<div style="font-size:12px;color:#a49eb8;margin-bottom:10px">✅ Your app is live — tap to open</div>'
            f'<a href="{public_url}" target="_blank" style="display:block;padding:16px 18px;border-radius:14px;'
            f'background:linear-gradient(135deg,#a78bfa,#8b5cf6);color:#fff;text-decoration:none;'
            f'font-size:18px;font-weight:800">📲 Open Chatterbox TTS</a>'
            f'<div style="margin-top:10px;font-size:11px;color:#a49eb8;word-break:break-all">{public_url}</div>'
            f'</div>'))
    except Exception:
        pass
else:
    print("cloudflared did not report a URL — check its output above.")


### 9️⃣ (Optional) Benchmark — one worker vs all loaded workers · ~2 min

Same seed, same chunks: single replica vs both GPUs, plus a warm rerun. Expect ~**N× wall-clock win** on the long
text; safe to skip on a fresh session.


In [ ]:
# ── STEP 9 · (Optional) benchmark: one worker vs all workers ──────────────────────

# Benchmark: prove the dual-GPU fan-out pays off. Fixed seed → identical chunks
# on both paths, so the delta is purely scheduling (and fp16, already applied).
import time

if not globals().get("WORKERS"):
    raise SystemExit("No model is loaded — press “Load model” in the app (or run the model manager cell) first.")

PARAMS = dict(exaggeration=0.5, cfg_weight=0.5, temperature=0.8,
              repetition_penalty=1.2, min_p=0.05, top_p=1.0,
              seed=42, gap_ms=120, chunk_chars=260)

BENCH_SHORT = ("Chatterbox makes short work of a quick hello. " * 2).strip()
BENCH_LONG  = ("The two graphics cards share every sentence between them, "
               "and the first one to finish simply takes the next chunk. " * 8).strip()

for name, txt in [("short", BENCH_SHORT), ("long", BENCH_LONG)]:
    t0 = time.time()
    _, w1, n = synth(txt, "en", None, dict(PARAMS), devices=WORKERS[:1])
    t_one = time.time() - t0
    t0 = time.time()
    _, wN, n = synth(txt, "en", None, dict(PARAMS))                    # all devices
    t_all = time.time() - t0
    print(f"{name:6s} {n:3d} chunks  1 gpu={t_one:6.2f}s   {len(WORKERS)} workers={t_all:6.2f}s"
          f"   ->  {t_one/t_all:4.2f}x   ({len(wN)/SAMPLE_RATE:.1f}s audio)")

t0 = time.time()
_, w2, _ = synth(BENCH_LONG, "en", None, dict(PARAMS))
print(f"long (warm rerun, conds cached): {time.time()-t0:6.2f}s")
